**Deivy Garzón**


Caso_Practico_GenAI_U1_Planificador_Viaje

In [1]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu peft datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 14.1 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
GPU disponible: True
GPU: Tesla T4


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Modelo cargado correctamente")
print("Modelo:", model_name)
print("Dispositivo:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo cargado correctamente
Modelo: Qwen/Qwen2.5-1.5B-Instruct
Dispositivo: cuda:0


In [4]:
messages = [
    {
        "role": "user",
        "content": "Dime en una sola frase qué es un itinerario de viaje."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Un itinerario de viaje es una planificación detallada de los lugares que se visitarán durante el viaje, incluyendo la ruta y las fechas de cada visita.


In [5]:
travel_preferences = {
    "destination": "Bogotá, Colombia",
    "days": 4,
    "budget": {
        "amount": 700,
        "currency": "USD",
        "includes_accommodation": False
    },
    "travelers": {
        "adults": 2,
        "children": 0
    },
    "interests": [
        "cultura",
        "gastronomía"
    ],
    "travel_style": "moderado",
    "transport_preference": "combinado",
    "accommodation_level": "medio",
    "restrictions": [
        "sin actividades extremas"
    ]
}

print(travel_preferences)

{'destination': 'Bogotá, Colombia', 'days': 4, 'budget': {'amount': 700, 'currency': 'USD', 'includes_accommodation': False}, 'travelers': {'adults': 2, 'children': 0}, 'interests': ['cultura', 'gastronomía'], 'travel_style': 'moderado', 'transport_preference': 'combinado', 'accommodation_level': 'medio', 'restrictions': ['sin actividades extremas']}


In [6]:
def validate_travel_preferences(data):
    errors = []

    # Destino
    if not data.get("destination"):
        errors.append("El destino es obligatorio.")

    # Duración
    if data.get("days", 0) <= 0:
        errors.append("La duración del viaje debe ser mayor a 0 días.")

    # Presupuesto
    budget = data.get("budget", {})
    if budget.get("amount", 0) <= 0:
        errors.append("El presupuesto debe ser mayor a 0.")

    if not budget.get("currency"):
        errors.append("Debe especificarse una moneda.")

    # Viajeros
    travelers = data.get("travelers", {})
    total_travelers = travelers.get("adults", 0) + travelers.get("children", 0)

    if total_travelers <= 0:
        errors.append("Debe existir al menos un viajero.")

    # Intereses
    if not data.get("interests"):
        errors.append("Debe seleccionarse al menos un interés.")

    # Ritmo de viaje
    valid_styles = ["relajado", "moderado", "intenso"]

    if data.get("travel_style") not in valid_styles:
        errors.append(
            "El estilo de viaje debe ser: relajado, moderado o intenso."
        )

    return errors


errors = validate_travel_preferences(travel_preferences)

if errors:
    print("❌ Se encontraron errores:")
    for error in errors:
        print("-", error)
else:
    print("✅ Preferencias validadas correctamente.")

✅ Preferencias validadas correctamente.


In [7]:
prompt = f"""
Eres un asistente experto en planificación de viajes.

Genera un itinerario de viaje personalizado con la siguiente información:

Destino: {travel_preferences['destination']}
Duración: {travel_preferences['days']} días
Presupuesto: {travel_preferences['budget']['amount']} {travel_preferences['budget']['currency']}
Incluye alojamiento: {travel_preferences['budget']['includes_accommodation']}
Adultos: {travel_preferences['travelers']['adults']}
Niños: {travel_preferences['travelers']['children']}
Intereses: {', '.join(travel_preferences['interests'])}
Estilo de viaje: {travel_preferences['travel_style']}
Transporte preferido: {travel_preferences['transport_preference']}
Nivel de alojamiento: {travel_preferences['accommodation_level']}
Restricciones: {', '.join(travel_preferences['restrictions'])}

Genera el itinerario día por día.
Incluye actividades recomendadas, horarios aproximados y costo estimado por día.
Procura que el plan sea coherente con el presupuesto y las restricciones indicadas.
"""

print(prompt)


Eres un asistente experto en planificación de viajes.

Genera un itinerario de viaje personalizado con la siguiente información:

Destino: Bogotá, Colombia
Duración: 4 días
Presupuesto: 700 USD
Incluye alojamiento: False
Adultos: 2
Niños: 0
Intereses: cultura, gastronomía
Estilo de viaje: moderado
Transporte preferido: combinado
Nivel de alojamiento: medio
Restricciones: sin actividades extremas

Genera el itinerario día por día.
Incluye actividades recomendadas, horarios aproximados y costo estimado por día.
Procura que el plan sea coherente con el presupuesto y las restricciones indicadas.



In [8]:
messages = [
    {
        "role": "system",
        "content": "Eres un asistente experto en planificación de viajes."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=900,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

baseline_itinerary = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(baseline_itinerary)

Itinerario de Viaje Personalizado para Bogotá, Colombia

Día 1:
Hoy comenzaremos nuestra aventura en la capital colombiana. Almorzaremos en el famoso Mercado La Candelaria, donde podrás degustar los sabrosos platos locales como el arepa, el pabrejo o el chicha morada. Por la tarde, visitaremos el Museo del Ámbito Histórico y Cultural del Banco de la Republica, una excelente oportunidad para aprender sobre la historia colonial de Bogotá. Cena en el tradicional Restaurante El Mágico de Bogotá.

Costo total: $35 (incluyendo almorzo)

Día 2:
Después de un buen descanso, comencemos nuestro segundo día explorando Bogotá. Comenzamos con el Parque Nacional Tolima, conocido por su increíble vegetación tropical y fauna endémica. Luego, visitaremos el Bosque Patón, uno de los parques más impresionantes de Colombia. Como regalo final, disfrutaremos de la cena en la plaza principal de Bogotá, La Candelaria.

Costo total: $50 (incluyendo entrada al Parque Nacional Tolima y al Bosque Patón) 

Día 3:


In [9]:
prompt_v2 = f"""
Actúa como un planificador profesional de viajes.

Tu tarea es crear un itinerario personalizado utilizando EXCLUSIVAMENTE
las preferencias proporcionadas por el usuario.

DATOS DEL VIAJE
- Destino: {travel_preferences['destination']}
- Duración: {travel_preferences['days']} días
- Presupuesto máximo: {travel_preferences['budget']['amount']} {travel_preferences['budget']['currency']}
- El presupuesto incluye alojamiento: {travel_preferences['budget']['includes_accommodation']}
- Adultos: {travel_preferences['travelers']['adults']}
- Niños: {travel_preferences['travelers']['children']}
- Intereses: {', '.join(travel_preferences['interests'])}
- Ritmo del viaje: {travel_preferences['travel_style']}
- Transporte preferido: {travel_preferences['transport_preference']}
- Nivel de alojamiento: {travel_preferences['accommodation_level']}
- Restricciones: {', '.join(travel_preferences['restrictions'])}

INSTRUCCIONES

1. Genera exactamente {travel_preferences['days']} días de itinerario.
2. Prioriza actividades relacionadas con los intereses del usuario.
3. Respeta todas las restricciones indicadas.
4. Evita repetir actividades.
5. Organiza las actividades considerando cercanía geográfica cuando sea posible.
6. Incluye mañana, tarde y noche.
7. Incluye una recomendación gastronómica por día.
8. Incluye el medio de transporte sugerido.
9. Incluye un costo estimado por día.
10. No presentes precios exactos como hechos confirmados.
    Identifícalos siempre como estimaciones.
11. Si no tienes certeza sobre un dato local, indícalo expresamente
    en lugar de inventar información.
12. El total estimado del viaje no debe superar el presupuesto disponible.

FORMATO DE SALIDA

DÍA 1
Mañana:
- Actividad:
- Horario aproximado:
- Transporte:
- Costo estimado:

Tarde:
- Actividad:
- Horario aproximado:
- Transporte:
- Costo estimado:

Noche:
- Actividad:
- Recomendación gastronómica:
- Costo estimado:

Total estimado Día 1:

Repite la misma estructura para todos los días.

Al final incluye:

RESUMEN DEL PRESUPUESTO
- Total estimado de actividades:
- Total estimado de alimentación:
- Total estimado de transporte:
- Total estimado del viaje:
- Presupuesto disponible:
- Saldo estimado:

ADVERTENCIA
Indica que precios, horarios y disponibilidad deben verificarse antes del viaje.
"""

print(prompt_v2)


Actúa como un planificador profesional de viajes.

Tu tarea es crear un itinerario personalizado utilizando EXCLUSIVAMENTE
las preferencias proporcionadas por el usuario.

DATOS DEL VIAJE
- Destino: Bogotá, Colombia
- Duración: 4 días
- Presupuesto máximo: 700 USD
- El presupuesto incluye alojamiento: False
- Adultos: 2
- Niños: 0
- Intereses: cultura, gastronomía
- Ritmo del viaje: moderado
- Transporte preferido: combinado
- Nivel de alojamiento: medio
- Restricciones: sin actividades extremas

INSTRUCCIONES

1. Genera exactamente 4 días de itinerario.
2. Prioriza actividades relacionadas con los intereses del usuario.
3. Respeta todas las restricciones indicadas.
4. Evita repetir actividades.
5. Organiza las actividades considerando cercanía geográfica cuando sea posible.
6. Incluye mañana, tarde y noche.
7. Incluye una recomendación gastronómica por día.
8. Incluye el medio de transporte sugerido.
9. Incluye un costo estimado por día.
10. No presentes precios exactos como hechos c

In [10]:
messages_v2 = [
    {
        "role": "system",
        "content": "Eres un planificador profesional de viajes. Debes seguir estrictamente las instrucciones del usuario y evitar inventar información no verificada."
    },
    {
        "role": "user",
        "content": prompt_v2
    }
]

text_v2 = tokenizer.apply_chat_template(
    messages_v2,
    tokenize=False,
    add_generation_prompt=True
)

inputs_v2 = tokenizer(
    text_v2,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_v2 = model.generate(
        **inputs_v2,
        max_new_tokens=1400,
        do_sample=True,
        temperature=0.5,
        top_p=0.85
    )

prompted_itinerary = tokenizer.decode(
    outputs_v2[0][inputs_v2["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(prompted_itinerary)

**ITINERARIO PERSONALIZADO PARA BOGOTÁ**

### **DÍA 1**
**Mañana:** 
- **Actividad:** Visita al Museo del Oro (Casa de la Moneda)
- **Horario aproximado:** 9:00 AM - 12:00 PM
- **Transporte:** Subiendo a la cumbre de la Catedral de Santa Fe
- **Costo estimado:** $15

**Tarde:** 
- **Actividad:** Visita al Parque Nacional Natural El Dorado
- **Horario aproximado:** 2:00 PM - 5:00 PM
- **Transporte:** Subiendo a la cima de la Montaña de San Cristóbal
- **Costo estimado:** $20

**Noche:** 
- **Actividad:** Cena en el Mercado Central de Bogotá
- **Recomendación gastronómica:** "El Poblador" o "La Esquina"
- **Costo estimado:** $30

**Total estimado Día 1:** $65

### **DÍA 2**
**Mañana:** 
- **Actividad:** Visita a la Plaza de Bolívar
- **Horario aproximado:** 9:00 AM - 12:00 PM
- **Transporte:** Subiendo al Teatro Colón
- **Costo estimado:** $25

**Tarde:** 
- **Actividad:** Visita al Centro Histórico de Bogotá
- **Horario aproximado:** 2:00 PM - 5:00 PM
- **Transporte:** Subiendo a la Cat

In [11]:
prompting_evaluation = {
    "estructura": "Mejorada",
    "personalizacion": "Adecuada",
    "respeto_numero_dias": True,
    "respeto_presupuesto": "Parcial",
    "informacion_local_verificable": False,
    "presenta_alucinaciones": True,
    "consistencia_costos": False,
    "necesita_rag": True
}

print("EVALUACIÓN DEL ITINERARIO CON PROMPT ENGINEERING\n")

for criterio, resultado in prompting_evaluation.items():
    print(f"{criterio}: {resultado}")

EVALUACIÓN DEL ITINERARIO CON PROMPT ENGINEERING

estructura: Mejorada
personalizacion: Adecuada
respeto_numero_dias: True
respeto_presupuesto: Parcial
informacion_local_verificable: False
presenta_alucinaciones: True
consistencia_costos: False
necesita_rag: True


In [12]:
knowledge_base = [
    """
    Museo del Oro:
    Ubicado en Bogotá, Colombia.
    Es uno de los principales museos de la ciudad y alberga una colección destacada
    de piezas de orfebrería prehispánica.
    Está relacionado con actividades culturales e históricas.
    """,

    """
    Museo Botero:
    Ubicado en el centro histórico de Bogotá.
    Exhibe obras de Fernando Botero y piezas de artistas internacionales.
    Es una actividad recomendada para viajeros interesados en arte y cultura.
    """,

    """
    La Candelaria:
    Es el centro histórico de Bogotá.
    Se caracteriza por su arquitectura colonial, calles tradicionales,
    museos, plazas y oferta cultural.
    Es adecuada para recorridos a pie.
    """,

    """
    Plaza de Bolívar:
    Ubicada en el centro histórico de Bogotá.
    Es una de las plazas más representativas de la ciudad y está rodeada
    por edificios históricos e institucionales.
    """,

    """
    Monserrate:
    Es uno de los principales puntos panorámicos de Bogotá.
    Se puede acceder mediante teleférico o funicular.
    Ofrece vistas de la ciudad y es una actividad turística popular.
    """,

    """
    Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gastronomía local.
    """,

    """
    Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.
    """,

    """
    Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.
    """
]

print("Documentos cargados:", len(knowledge_base))

Documentos cargados: 8


In [13]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

knowledge_embeddings = embedding_model.encode(
    knowledge_base,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Forma de los embeddings:", knowledge_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Forma de los embeddings: (8, 384)


In [14]:
import faiss
import numpy as np

dimension = knowledge_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)

faiss_index.add(
    knowledge_embeddings.astype("float32")
)

print("Vectores almacenados en FAISS:", faiss_index.ntotal)
print("Dimensión de los vectores:", dimension)

Vectores almacenados en FAISS: 8
Dimensión de los vectores: 384


In [15]:
query = "Quiero actividades culturales y gastronómicas en Bogotá"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

distances, indices = faiss_index.search(
    query_embedding,
    k=4
)

print("CONSULTA:")
print(query)

print("\nDOCUMENTOS RECUPERADOS:\n")

for rank, idx in enumerate(indices[0], start=1):
    print(f"Resultado {rank}")
    print(knowledge_base[idx].strip())
    print("-" * 60)

CONSULTA:
Quiero actividades culturales y gastronómicas en Bogotá

DOCUMENTOS RECUPERADOS:

Resultado 1
Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.
------------------------------------------------------------
Resultado 2
Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.
------------------------------------------------------------
Resultado 3
Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gastronomía local.
------------------------------------------------------------
Resultado 4
La Candelaria:
    Es el centro histórico de Bogotá.
    Se caracteriza por su arquitectura colonial, calles tradicionales,
    museos, plazas y oferta cultural.


In [16]:
retrieved_docs = [
    knowledge_base[idx].strip()
    for idx in indices[0]
]

rag_context = "\n\n".join(retrieved_docs)

print("CONTEXTO RAG:\n")
print(rag_context)

CONTEXTO RAG:

Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.

Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.

Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gastronomía local.

La Candelaria:
    Es el centro histórico de Bogotá.
    Se caracteriza por su arquitectura colonial, calles tradicionales,
    museos, plazas y oferta cultural.
    Es adecuada para recorridos a pie.


In [17]:
rag_prompt = f"""
Actúa como un planificador profesional de viajes.

Debes crear un itinerario personalizado para el usuario utilizando
las preferencias indicadas y apoyándote únicamente en la información
proporcionada en el CONTEXTO RAG.

PREFERENCIAS DEL VIAJE
- Destino: {travel_preferences['destination']}
- Duración: {travel_preferences['days']} días
- Presupuesto máximo: {travel_preferences['budget']['amount']} {travel_preferences['budget']['currency']}
- Adultos: {travel_preferences['travelers']['adults']}
- Niños: {travel_preferences['travelers']['children']}
- Intereses: {', '.join(travel_preferences['interests'])}
- Ritmo del viaje: {travel_preferences['travel_style']}
- Transporte preferido: {travel_preferences['transport_preference']}
- Restricciones: {', '.join(travel_preferences['restrictions'])}

CONTEXTO RAG
{rag_context}

INSTRUCCIONES

1. Usa únicamente lugares que aparezcan en el CONTEXTO RAG.
2. No inventes nombres de atracciones, restaurantes o zonas.
3. Genera exactamente {travel_preferences['days']} días.
4. Prioriza cultura y gastronomía.
5. Respeta las restricciones del usuario.
6. Organiza mañana, tarde y noche.
7. Incluye transporte sugerido.
8. Los costos deben presentarse únicamente como estimaciones.
9. Si el contexto no contiene suficiente información para algún detalle,
   indícalo explícitamente.
10. No presentes datos como horarios oficiales o precios confirmados
    si no aparecen en el contexto.

FORMATO

DÍA 1

Mañana:
- Lugar:
- Actividad:
- Transporte:
- Costo estimado:

Tarde:
- Lugar:
- Actividad:
- Transporte:
- Costo estimado:

Noche:
- Lugar:
- Actividad:
- Transporte:
- Costo estimado:

Total estimado del día:

Repite la estructura para los 4 días.

Al final incluye:

RESUMEN
- Lugares utilizados:
- Presupuesto total estimado:
- Presupuesto disponible:
- Saldo estimado:

ADVERTENCIA
Aclara que horarios, precios y disponibilidad deben verificarse
antes de realizar el viaje.
"""

print(rag_prompt)


Actúa como un planificador profesional de viajes.

Debes crear un itinerario personalizado para el usuario utilizando
las preferencias indicadas y apoyándote únicamente en la información
proporcionada en el CONTEXTO RAG.

PREFERENCIAS DEL VIAJE
- Destino: Bogotá, Colombia
- Duración: 4 días
- Presupuesto máximo: 700 USD
- Adultos: 2
- Niños: 0
- Intereses: cultura, gastronomía
- Ritmo del viaje: moderado
- Transporte preferido: combinado
- Restricciones: sin actividades extremas

CONTEXTO RAG
Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.

Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.

Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gas

In [18]:
messages_rag = [
    {
        "role": "system",
        "content": (
            "Eres un planificador profesional de viajes. "
            "Debes usar únicamente la información proporcionada en el contexto RAG "
            "y evitar inventar lugares, horarios o precios no sustentados."
        )
    },
    {
        "role": "user",
        "content": rag_prompt
    }
]

text_rag = tokenizer.apply_chat_template(
    messages_rag,
    tokenize=False,
    add_generation_prompt=True
)

inputs_rag = tokenizer(
    text_rag,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_rag = model.generate(
        **inputs_rag,
        max_new_tokens=1400,
        do_sample=True,
        temperature=0.3,
        top_p=0.8
    )

rag_itinerary = tokenizer.decode(
    outputs_rag[0][inputs_rag["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(rag_itinerary)

**DÍA 1**

Mañana:
- **Lugar:** Usaquén
- **Actividad:** Caminar por las calles de Usaquén, visitando restaurantes locales y café típicos.
- **Transporte:** Subir al barrio de Usaquén (subiendo desde el centro).
- **Costo estimado:** $10-$20

Tarde:
- **Lugar:** Zona G
- **Actividad:** Visitar el mercado de La Candelaria y degustar comidas típicas colombianas.
- **Transporte:** Subir al mercado de La Candelaria (en bicicleta o taxi).
- **Costo estimado:** $10-$20

Noche:
- **Lugar:** La Candelaria
- **Actividad:** Recorrer las calles de La Candelaria, visitando museos y plazas.
- **Transporte:** Subir al centro histórico (en metro).
- **Costo estimado:** $10-$20

**DÍA 2**

Mañana:
- **Lugar:** Usaquén
- **Actividad:** Continuar explorando Usaquén, visitando más restaurantes y cafés.
- **Transporte:** Subir al barrio de Usaquén.
- **Costo estimado:** $10-$20

Tarde:
- **Lugar:** Zona G
- **Actividad:** Visitar mercados locales y restaurantes cercanos.
- **Transporte:** Subir al mercado

In [19]:
rag_evaluation = {
    "usa_lugares_del_contexto": True,
    "reduce_alucinaciones_de_lugares": True,
    "inventa_detalles_no_presentes_en_contexto": True,
    "respeta_numero_dias": True,
    "respeta_presupuesto": True,
    "repite_actividades": True,
    "consistencia_costos": "Parcial",
    "necesita_mejorar_prompt_rag": True
}

print("EVALUACIÓN DEL ITINERARIO CON RAG\n")

for criterio, resultado in rag_evaluation.items():
    print(f"{criterio}: {resultado}")

EVALUACIÓN DEL ITINERARIO CON RAG

usa_lugares_del_contexto: True
reduce_alucinaciones_de_lugares: True
inventa_detalles_no_presentes_en_contexto: True
respeta_numero_dias: True
respeta_presupuesto: True
repite_actividades: True
consistencia_costos: Parcial
necesita_mejorar_prompt_rag: True


In [20]:
rag_prompt_v2 = f"""
Actúa como un planificador profesional de viajes.

Tu tarea es crear un itinerario personalizado usando SOLO la información
contenida en el CONTEXTO RAG.

PREFERENCIAS DEL VIAJE
- Destino: {travel_preferences['destination']}
- Duración: {travel_preferences['days']} días
- Presupuesto máximo: {travel_preferences['budget']['amount']} {travel_preferences['budget']['currency']}
- Adultos: {travel_preferences['travelers']['adults']}
- Niños: {travel_preferences['travelers']['children']}
- Intereses: {', '.join(travel_preferences['interests'])}
- Ritmo del viaje: {travel_preferences['travel_style']}
- Transporte preferido: {travel_preferences['transport_preference']}
- Restricciones: {', '.join(travel_preferences['restrictions'])}

CONTEXTO RAG
{rag_context}

REGLAS OBLIGATORIAS

1. Usa únicamente lugares mencionados explícitamente en el CONTEXTO RAG.
2. No inventes sublugares, restaurantes, mercados, museos ni atracciones.
3. No inventes medios de transporte concretos.
4. Si el contexto no indica cómo desplazarse, escribe:
   "Transporte: por definir según disponibilidad local".
5. No inventes precios.
6. Si el contexto no contiene precios, escribe:
   "Costo estimado: no disponible en el contexto".
7. No inventes horarios.
8. Si el contexto no contiene horarios, escribe:
   "Horario: debe verificarse".
9. No atribuyas características a un lugar que no estén mencionadas en el contexto.
10. Evita repetir el mismo lugar más de dos veces durante todo el itinerario.
11. Genera exactamente {travel_preferences['days']} días.
12. Prioriza cultura y gastronomía.
13. Respeta todas las restricciones del usuario.
14. Si la información recuperada no es suficiente para completar los 4 días
    sin repetir o inventar información, indícalo claramente.
15. No completes información faltante utilizando conocimiento propio del modelo.

FORMATO

DÍA 1

Mañana:
- Lugar:
- Actividad:
- Horario:
- Transporte:
- Costo estimado:

Tarde:
- Lugar:
- Actividad:
- Horario:
- Transporte:
- Costo estimado:

Noche:
- Lugar:
- Actividad:
- Horario:
- Transporte:
- Costo estimado:

Repite la estructura para los 4 días.

Al final incluye:

EVALUACIÓN DE LA INFORMACIÓN
- Información disponible:
- Información faltante:
- ¿Fue necesario repetir lugares?:
- ¿El contexto fue suficiente para construir el itinerario completo?:

ADVERTENCIA
No presentes ningún dato no contenido en el contexto como un hecho.
"""

print(rag_prompt_v2)


Actúa como un planificador profesional de viajes.

Tu tarea es crear un itinerario personalizado usando SOLO la información
contenida en el CONTEXTO RAG.

PREFERENCIAS DEL VIAJE
- Destino: Bogotá, Colombia
- Duración: 4 días
- Presupuesto máximo: 700 USD
- Adultos: 2
- Niños: 0
- Intereses: cultura, gastronomía
- Ritmo del viaje: moderado
- Transporte preferido: combinado
- Restricciones: sin actividades extremas

CONTEXTO RAG
Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.

Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.

Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gastronomía local.

La Candelaria:
    Es el centro histórico de Bogot

In [21]:
messages_rag_v2 = [
    {
        "role": "system",
        "content": (
            "Eres un planificador profesional de viajes. "
            "Debes obedecer estrictamente el contexto RAG y no completar "
            "información faltante con conocimiento propio."
        )
    },
    {
        "role": "user",
        "content": rag_prompt_v2
    }
]

text_rag_v2 = tokenizer.apply_chat_template(
    messages_rag_v2,
    tokenize=False,
    add_generation_prompt=True
)

inputs_rag_v2 = tokenizer(
    text_rag_v2,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_rag_v2 = model.generate(
        **inputs_rag_v2,
        max_new_tokens=1600,
        do_sample=False
    )

rag_itinerary_v2 = tokenizer.decode(
    outputs_rag_v2[0][inputs_rag_v2["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(rag_itinerary_v2)

### DÍA 1

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Caminar por Usaquén, visitando restaurantes locales y café típicos.
- **Horario:** 9:00 AM - 12:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

**Tarde**
- **Lugar:** Zona G
- **Actividad:** Visitar un mercado tradicional como Mercado de La Perseverancia.
- **Horario:** 12:00 PM - 1:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $15 USD

**Noche**
- **Lugar:** La Candelaria
- **Actividad:** Recorrer la plaza Mayor y visitar el Museo del Oro.
- **Horario:** 1:00 PM - 4:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

### DÍA 2

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Continuar explorando Usaquén, visitando otros restaurantes y cafés.
- **Horario:** 9:00 AM - 12:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

**Tarde**
- **Lugar:** Zona G
- **Actividad:** Visitar otro mercado tradicional como Mercad

In [22]:
import re

def validate_rag_response(response):

    issues = []

    # Lugares permitidos por el contexto recuperado
    allowed_places = [
        "Usaquén",
        "Zona G",
        "Mercado de La Perseverancia",
        "La Candelaria"
    ]

    # Lugares que aparecieron inventados en esta respuesta
    unsupported_places = [
        "Museo del Oro",
        "Plaza Bolívar",
        "Plaza Mayor"
    ]

    # 1. Detectar lugares no soportados
    for place in unsupported_places:
        if place.lower() in response.lower():
            issues.append(
                f"Lugar no respaldado por el contexto RAG: {place}"
            )

    # 2. Detectar precios inventados
    prices = re.findall(r"\$\s?\d+", response)

    if prices:
        issues.append(
            f"Se detectaron precios no respaldados: {prices}"
        )

    # 3. Detectar horarios inventados
    times = re.findall(
        r"\b\d{1,2}:\d{2}\s?(?:AM|PM)\b",
        response,
        flags=re.IGNORECASE
    )

    if times:
        issues.append(
            f"Se detectaron horarios no respaldados: {times}"
        )

    # 4. Detectar transporte no respaldado
    unsupported_transport = [
        "bicicleta",
        "subte",
        "metro",
        "autobús",
        "taxi"
    ]

    for transport in unsupported_transport:
        if transport.lower() in response.lower():
            issues.append(
                f"Transporte no respaldado por el contexto: {transport}"
            )

    # 5. Contar repeticiones de lugares permitidos
    repetitions = {}

    for place in allowed_places:
        count = response.lower().count(place.lower())
        repetitions[place] = count

        if count > 2:
            issues.append(
                f"El lugar '{place}' aparece {count} veces."
            )

    return issues, repetitions


rag_issues, place_repetitions = validate_rag_response(
    rag_itinerary_v2
)

print("VALIDACIÓN AUTOMÁTICA DEL RAG V2\n")

if rag_issues:
    print("❌ Se detectaron problemas:\n")

    for issue in rag_issues:
        print("-", issue)

else:
    print("✅ No se detectaron problemas.")

print("\nREPETICIONES DE LUGARES:")
for place, count in place_repetitions.items():
    print(f"- {place}: {count}")

VALIDACIÓN AUTOMÁTICA DEL RAG V2

❌ Se detectaron problemas:

- Lugar no respaldado por el contexto RAG: Museo del Oro
- Lugar no respaldado por el contexto RAG: Plaza Bolívar
- Lugar no respaldado por el contexto RAG: Plaza Mayor
- Se detectaron precios no respaldados: ['$20', '$15', '$20', '$20', '$15', '$20', '$20', '$15', '$20', '$20', '$15', '$20']
- Se detectaron horarios no respaldados: ['9:00 AM', '12:00 PM', '12:00 PM', '1:00 PM', '1:00 PM', '4:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '1:00 PM', '1:00 PM', '4:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '1:00 PM', '1:00 PM', '4:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '1:00 PM', '1:00 PM', '4:00 PM']
- Transporte no respaldado por el contexto: bicicleta
- El lugar 'Usaquén' aparece 8 veces.
- El lugar 'Zona G' aparece 5 veces.
- El lugar 'Mercado de La Perseverancia' aparece 4 veces.
- El lugar 'La Candelaria' aparece 4 veces.

REPETICIONES DE LUGARES:
- Usaquén: 8
- Zona G: 5
- Mercado de La Perseverancia: 4
- La Candelaria

In [23]:
correction_feedback = "\n".join(
    [f"- {issue}" for issue in rag_issues]
)

correction_prompt = f"""
Debes corregir el siguiente itinerario de viaje.

ITINERARIO ORIGINAL
{rag_itinerary_v2}

PROBLEMAS DETECTADOS AUTOMÁTICAMENTE
{correction_feedback}

CONTEXTO RAG AUTORIZADO
{rag_context}

REGLAS DE CORRECCIÓN

1. Elimina cualquier lugar que no aparezca explícitamente en el CONTEXTO RAG.
2. No agregues nuevos lugares.
3. No inventes precios.
4. Cuando no exista precio en el contexto, escribe exactamente:
   "Costo estimado: no disponible en el contexto".
5. No inventes horarios.
6. Cuando no exista horario en el contexto, escribe exactamente:
   "Horario: debe verificarse".
7. No inventes medios de transporte.
8. Cuando el contexto no indique transporte, escribe exactamente:
   "Transporte: por definir según disponibilidad local".
9. No inventes museos, plazas, restaurantes, mercados ni atracciones adicionales.
10. Evita repetir el mismo lugar más de dos veces.
11. Si no existe suficiente información para completar los 4 días sin inventar,
    indícalo explícitamente en lugar de crear información nueva.
12. Mantén exactamente 4 días en la respuesta.
13. Utiliza únicamente información respaldada por el CONTEXTO RAG.

Devuelve únicamente el itinerario corregido.
"""

print(correction_prompt)


Debes corregir el siguiente itinerario de viaje.

ITINERARIO ORIGINAL
### DÍA 1

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Caminar por Usaquén, visitando restaurantes locales y café típicos.
- **Horario:** 9:00 AM - 12:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

**Tarde**
- **Lugar:** Zona G
- **Actividad:** Visitar un mercado tradicional como Mercado de La Perseverancia.
- **Horario:** 12:00 PM - 1:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $15 USD

**Noche**
- **Lugar:** La Candelaria
- **Actividad:** Recorrer la plaza Mayor y visitar el Museo del Oro.
- **Horario:** 1:00 PM - 4:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

### DÍA 2

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Continuar explorando Usaquén, visitando otros restaurantes y cafés.
- **Horario:** 9:00 AM - 12:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

**Tarde**
- **Lugar:

In [24]:
messages_correction = [
    {
        "role": "system",
        "content": (
            "Eres un validador y corrector de itinerarios de viaje. "
            "Debes obedecer estrictamente las reglas de corrección y no inventar información."
        )
    },
    {
        "role": "user",
        "content": correction_prompt
    }
]

text_correction = tokenizer.apply_chat_template(
    messages_correction,
    tokenize=False,
    add_generation_prompt=True
)

inputs_correction = tokenizer(
    text_correction,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_correction = model.generate(
        **inputs_correction,
        max_new_tokens=1600,
        do_sample=False
    )

corrected_itinerary = tokenizer.decode(
    outputs_correction[0][inputs_correction["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(corrected_itinerary)

### ITINERARIO CORREGIDO
### DÍA 1

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Caminar por Usaquén, visitando restaurantes locales y café típicos.
- **Horario:** 9:00 AM - 12:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

**Tarde**
- **Lugar:** Zona G
- **Actividad:** Visitar Mercado de La Perseverancia.
- **Horario:** 12:00 PM - 1:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $15 USD

**Noche**
- **Lugar:** La Candelaria
- **Actividad:** Recorrer la Plaza Mayor y visitar el Museo del Oro.
- **Horario:** 1:00 PM - 4:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

### DÍA 2

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Continuar explorando Usaquén, visitando otros restaurantes y cafés.
- **Horario:** 9:00 AM - 12:00 PM
- **Transporte:** Bicicleta (si disponible)
- **Costo estimado:** $20 USD

**Tarde**
- **Lugar:** Zona G
- **Actividad:** Visitar Mercado de La Perseverancia.
- **Horari

In [25]:
corrected_issues, corrected_repetitions = validate_rag_response(
    corrected_itinerary
)

print("VALIDACIÓN DEL ITINERARIO CORREGIDO\n")

if corrected_issues:
    print("❌ La corrección todavía presenta problemas:\n")

    for issue in corrected_issues:
        print("-", issue)

else:
    print("✅ El itinerario corregido pasó todas las validaciones.")

print("\nREPETICIONES DE LUGARES:")
for place, count in corrected_repetitions.items():
    print(f"- {place}: {count}")

VALIDACIÓN DEL ITINERARIO CORREGIDO

❌ La corrección todavía presenta problemas:

- Lugar no respaldado por el contexto RAG: Museo del Oro
- Lugar no respaldado por el contexto RAG: Plaza Bolívar
- Lugar no respaldado por el contexto RAG: Plaza Mayor
- Se detectaron precios no respaldados: ['$20', '$15', '$20', '$20', '$15', '$20', '$20', '$15', '$20', '$20', '$15', '$20']
- Se detectaron horarios no respaldados: ['9:00 AM', '12:00 PM', '12:00 PM', '1:00 PM', '1:00 PM', '4:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '1:00 PM', '1:00 PM', '4:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '1:00 PM', '1:00 PM', '4:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '1:00 PM', '1:00 PM', '4:00 PM']
- Transporte no respaldado por el contexto: bicicleta
- El lugar 'Usaquén' aparece 8 veces.
- El lugar 'Zona G' aparece 4 veces.
- El lugar 'Mercado de La Perseverancia' aparece 4 veces.
- El lugar 'La Candelaria' aparece 4 veces.

REPETICIONES DE LUGARES:
- Usaquén: 8
- Zona G: 4
- Mercado de La Perseveranci

In [26]:
import re

def sanitize_itinerary(response):

    cleaned = response

    # 1. Eliminar lugares no soportados
    unsupported_places = [
        "Museo del Oro",
        "Plaza Bolívar",
        "Plaza Mayor"
    ]

    for place in unsupported_places:
        cleaned = re.sub(
            re.escape(place),
            "[INFORMACIÓN NO RESPALDADA POR EL CONTEXTO]",
            cleaned,
            flags=re.IGNORECASE
        )

    # 2. Reemplazar precios no respaldados
    cleaned = re.sub(
        r"\$\s?\d+\s?USD?",
        "no disponible en el contexto",
        cleaned,
        flags=re.IGNORECASE
    )

    # 3. Reemplazar horarios no respaldados
    cleaned = re.sub(
        r"\b\d{1,2}:\d{2}\s?(?:AM|PM)\s*-\s*\d{1,2}:\d{2}\s?(?:AM|PM)\b",
        "debe verificarse",
        cleaned,
        flags=re.IGNORECASE
    )

    # 4. Reemplazar transporte no respaldado
    unsupported_transport = [
        "Bicicleta (si disponible)",
        "bicicleta",
        "subte",
        "metro",
        "autobús",
        "taxi"
    ]

    for transport in unsupported_transport:
        cleaned = re.sub(
            re.escape(transport),
            "por definir según disponibilidad local",
            cleaned,
            flags=re.IGNORECASE
        )

    return cleaned


sanitized_itinerary = sanitize_itinerary(corrected_itinerary)

print("ITINERARIO SANEADO\n")
print(sanitized_itinerary)

ITINERARIO SANEADO

### ITINERARIO CORREGIDO
### DÍA 1

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Caminar por Usaquén, visitando restaurantes locales y café típicos.
- **Horario:** debe verificarse
- **Transporte:** por definir según disponibilidad local
- **Costo estimado:** no disponible en el contexto

**Tarde**
- **Lugar:** Zona G
- **Actividad:** Visitar Mercado de La Perseverancia.
- **Horario:** debe verificarse
- **Transporte:** por definir según disponibilidad local
- **Costo estimado:** no disponible en el contexto

**Noche**
- **Lugar:** La Candelaria
- **Actividad:** Recorrer la [INFORMACIÓN NO RESPALDADA POR EL CONTEXTO] y visitar el [INFORMACIÓN NO RESPALDADA POR EL CONTEXTO].
- **Horario:** debe verificarse
- **Transporte:** por definir según disponibilidad local
- **Costo estimado:** no disponible en el contexto

### DÍA 2

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Continuar explorando Usaquén, visitando otros restaurantes y cafés.
- **Horario:** debe veri

In [27]:
def evaluate_context_sufficiency(repetitions, max_repetitions=2):

    excessive_repetitions = {
        place: count
        for place, count in repetitions.items()
        if count > max_repetitions
    }

    if excessive_repetitions:
        return {
            "context_sufficient": False,
            "reason": "La base de conocimiento no contiene suficientes lugares distintos para construir un itinerario variado de 4 días.",
            "excessive_repetitions": excessive_repetitions
        }

    return {
        "context_sufficient": True,
        "reason": "El contexto contiene suficiente variedad de lugares.",
        "excessive_repetitions": {}
    }


context_evaluation = evaluate_context_sufficiency(
    corrected_repetitions
)

print("EVALUACIÓN DE SUFICIENCIA DEL CONTEXTO\n")
print("Contexto suficiente:", context_evaluation["context_sufficient"])
print("Motivo:", context_evaluation["reason"])
print("Repeticiones excesivas:", context_evaluation["excessive_repetitions"])

EVALUACIÓN DE SUFICIENCIA DEL CONTEXTO

Contexto suficiente: False
Motivo: La base de conocimiento no contiene suficientes lugares distintos para construir un itinerario variado de 4 días.
Repeticiones excesivas: {'Usaquén': 8, 'Zona G': 4, 'Mercado de La Perseverancia': 4, 'La Candelaria': 4}


In [28]:
knowledge_base_extended = [
    """
    Museo del Oro:
    Ubicado en Bogotá, Colombia.
    Es uno de los principales museos de la ciudad y alberga una colección destacada
    de piezas de orfebrería prehispánica.
    Está relacionado con actividades culturales e históricas.
    """,

    """
    Museo Botero:
    Ubicado en el centro histórico de Bogotá.
    Exhibe obras de Fernando Botero y piezas de artistas internacionales.
    Es una actividad recomendada para viajeros interesados en arte y cultura.
    """,

    """
    La Candelaria:
    Es el centro histórico de Bogotá.
    Se caracteriza por su arquitectura colonial, calles tradicionales,
    museos, plazas y oferta cultural.
    Es adecuada para recorridos a pie.
    """,

    """
    Plaza de Bolívar:
    Ubicada en el centro histórico de Bogotá.
    Es una de las plazas más representativas de la ciudad y está rodeada
    por edificios históricos e institucionales.
    """,

    """
    Monserrate:
    Es uno de los principales puntos panorámicos de Bogotá.
    Se puede acceder mediante teleférico o funicular.
    Ofrece vistas de la ciudad y es una actividad turística popular.
    """,

    """
    Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gastronomía local.
    """,

    """
    Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.
    """,

    """
    Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.
    """,

    """
    Museo Nacional de Colombia:
    Es uno de los museos más importantes de Bogotá.
    Su colección está relacionada con historia, patrimonio, arte y cultura colombiana.
    """,

    """
    Jardín Botánico de Bogotá:
    Espacio dedicado a la conservación y divulgación de la flora.
    Es apropiado para actividades tranquilas relacionadas con naturaleza y educación ambiental.
    """,

    """
    Parque Simón Bolívar:
    Uno de los principales parques urbanos de Bogotá.
    Es adecuado para caminatas, descanso y actividades recreativas no extremas.
    """,

    """
    Quinta de Bolívar:
    Casa museo histórica de Bogotá vinculada a Simón Bolívar.
    Es una opción cultural e histórica para visitantes.
    """,

    """
    Chorro de Quevedo:
    Espacio tradicional ubicado en La Candelaria.
    Está relacionado con historia, arquitectura y recorridos culturales por el centro de Bogotá.
    """,

    """
    Biblioteca Luis Ángel Arango:
    Espacio cultural ubicado en el centro de Bogotá.
    Ofrece actividades relacionadas con lectura, patrimonio, arte y cultura.
    """,

    """
    Paloquemao:
    Sector conocido por su mercado y oferta de productos alimenticios.
    Puede ser de interés para viajeros que desean conocer ingredientes,
    productos locales y experiencias gastronómicas.
    """,

    """
    Parque de la 93:
    Zona urbana de Bogotá con restaurantes, cafés y espacios para caminar.
    Puede ser adecuada para actividades relajadas y experiencias gastronómicas.
    """
]

print("Documentos cargados en la base ampliada:", len(knowledge_base_extended))

Documentos cargados en la base ampliada: 16


In [29]:
knowledge_embeddings_extended = embedding_model.encode(
    knowledge_base_extended,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Forma de los embeddings ampliados:",
    knowledge_embeddings_extended.shape
)

Forma de los embeddings ampliados: (16, 384)


In [30]:
dimension_extended = knowledge_embeddings_extended.shape[1]

faiss_index_extended = faiss.IndexFlatIP(dimension_extended)

faiss_index_extended.add(
    knowledge_embeddings_extended.astype("float32")
)

print("Vectores almacenados en FAISS ampliado:", faiss_index_extended.ntotal)
print("Dimensión de los vectores:", dimension_extended)

Vectores almacenados en FAISS ampliado: 16
Dimensión de los vectores: 384


In [31]:
query_extended = "Quiero actividades culturales y gastronómicas en Bogotá"

query_embedding_extended = embedding_model.encode(
    [query_extended],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

distances_extended, indices_extended = faiss_index_extended.search(
    query_embedding_extended,
    k=10
)

print("CONSULTA:")
print(query_extended)

print("\nDOCUMENTOS RECUPERADOS:\n")

for rank, idx in enumerate(indices_extended[0], start=1):
    print(f"Resultado {rank}")
    print(knowledge_base_extended[idx].strip())
    print("-" * 60)

CONSULTA:
Quiero actividades culturales y gastronómicas en Bogotá

DOCUMENTOS RECUPERADOS:

Resultado 1
Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.
------------------------------------------------------------
Resultado 2
Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.
------------------------------------------------------------
Resultado 3
Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gastronomía local.
------------------------------------------------------------
Resultado 4
Parque de la 93:
    Zona urbana de Bogotá con restaurantes, cafés y espacios para caminar.
    Puede ser adecuada para actividades relajadas y experiencias gastr

In [32]:
retrieved_docs_extended = [
    knowledge_base_extended[idx].strip()
    for idx in indices_extended[0]
]

rag_context_extended = "\n\n".join(retrieved_docs_extended)

print("CONTEXTO RAG AMPLIADO:\n")
print(rag_context_extended)

CONTEXTO RAG AMPLIADO:

Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.

Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.

Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gastronomía local.

Parque de la 93:
    Zona urbana de Bogotá con restaurantes, cafés y espacios para caminar.
    Puede ser adecuada para actividades relajadas y experiencias gastronómicas.

La Candelaria:
    Es el centro histórico de Bogotá.
    Se caracteriza por su arquitectura colonial, calles tradicionales,
    museos, plazas y oferta cultural.
    Es adecuada para recorridos a pie.

Chorro de Quevedo:
    Espacio tradicional ubicado en La Candelaria.
    Está relac

In [33]:
rag_prompt_extended = f"""
Actúa como un planificador profesional de viajes.

Tu tarea es crear un itinerario personalizado usando SOLO la información
contenida en el CONTEXTO RAG AMPLIADO.

PREFERENCIAS DEL VIAJE
- Destino: {travel_preferences['destination']}
- Duración: {travel_preferences['days']} días
- Presupuesto máximo: {travel_preferences['budget']['amount']} {travel_preferences['budget']['currency']}
- Adultos: {travel_preferences['travelers']['adults']}
- Niños: {travel_preferences['travelers']['children']}
- Intereses: {', '.join(travel_preferences['interests'])}
- Ritmo del viaje: {travel_preferences['travel_style']}
- Transporte preferido: {travel_preferences['transport_preference']}
- Restricciones: {', '.join(travel_preferences['restrictions'])}

CONTEXTO RAG AMPLIADO
{rag_context_extended}

REGLAS OBLIGATORIAS

1. Usa únicamente lugares mencionados explícitamente en el CONTEXTO RAG AMPLIADO.
2. No inventes nuevos lugares, restaurantes, museos, plazas ni atracciones.
3. No inventes precios.
4. Si no hay precio disponible, escribe:
   "Costo estimado: no disponible en el contexto".
5. No inventes horarios.
6. Si no hay horario disponible, escribe:
   "Horario: debe verificarse".
7. No inventes medios de transporte.
8. Usa únicamente medios de transporte que aparezcan explícitamente en el contexto.
9. Si no hay transporte especificado, escribe:
   "Transporte: por definir según disponibilidad local".
10. Evita repetir el mismo lugar más de una vez, salvo que sea estrictamente necesario.
11. Genera exactamente {travel_preferences['days']} días.
12. Cada día debe contener mañana, tarde y noche.
13. Prioriza actividades relacionadas con cultura y gastronomía.
14. Agrupa lugares cercanos o relacionados conceptualmente cuando sea posible.
15. Si falta información, indícalo explícitamente y no la completes con conocimiento propio.

FORMATO

DÍA 1

Mañana:
- Lugar:
- Actividad:
- Horario:
- Transporte:
- Costo estimado:

Tarde:
- Lugar:
- Actividad:
- Horario:
- Transporte:
- Costo estimado:

Noche:
- Lugar:
- Actividad:
- Horario:
- Transporte:
- Costo estimado:

Repite el formato para los 4 días.

Al final incluye:

EVALUACIÓN DEL ITINERARIO
- Lugares utilizados:
- Lugares repetidos:
- Información faltante:
- ¿El contexto fue suficiente?:

ADVERTENCIA
Los horarios, precios y disponibilidad deben verificarse antes del viaje.
"""

print(rag_prompt_extended)


Actúa como un planificador profesional de viajes.

Tu tarea es crear un itinerario personalizado usando SOLO la información
contenida en el CONTEXTO RAG AMPLIADO.

PREFERENCIAS DEL VIAJE
- Destino: Bogotá, Colombia
- Duración: 4 días
- Presupuesto máximo: 700 USD
- Adultos: 2
- Niños: 0
- Intereses: cultura, gastronomía
- Ritmo del viaje: moderado
- Transporte preferido: combinado
- Restricciones: sin actividades extremas

CONTEXTO RAG AMPLIADO
Usaquén:
    Sector tradicional de Bogotá con restaurantes, cafés, arquitectura y espacios culturales.
    Es adecuado para caminar, comer y realizar actividades relajadas.

Zona G:
    Sector de Bogotá reconocido por su oferta gastronómica y variedad de restaurantes.
    Es adecuada para experiencias culinarias y cenas.

Mercado de La Perseverancia:
    Mercado tradicional de Bogotá conocido por su oferta de comida colombiana.
    Puede ser una opción interesante para viajeros con interés en gastronomía local.

Parque de la 93:
    Zona urbana

In [34]:
messages_rag_extended = [
    {
        "role": "system",
        "content": (
            "Eres un planificador profesional de viajes. "
            "Debes usar exclusivamente el contexto RAG proporcionado "
            "y no completar información faltante con conocimiento propio."
        )
    },
    {
        "role": "user",
        "content": rag_prompt_extended
    }
]

text_rag_extended = tokenizer.apply_chat_template(
    messages_rag_extended,
    tokenize=False,
    add_generation_prompt=True
)

inputs_rag_extended = tokenizer(
    text_rag_extended,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_rag_extended = model.generate(
        **inputs_rag_extended,
        max_new_tokens=1800,
        do_sample=False
    )

rag_itinerary_extended = tokenizer.decode(
    outputs_rag_extended[0][inputs_rag_extended["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(rag_itinerary_extended)

### DÍA 1

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Caminar por las calles de Usaquén, visitando restaurantes locales y café típicos.
- **Horario:** 9:00 AM - 12:00 PM
- **Transporte:** Bici (si disponible)
- **Costo estimado:** $20 USD

**Tarde**
- **Lugar:** Parque de la 93
- **Actividad:** Recorrer el parque, disfrutando de sus vistas panorámicas y paseando por sus áreas verdes.
- **Horario:** 12:00 PM - 2:00 PM
- **Transporte:** Bici (si disponible)
- **Costo estimado:** $15 USD

**Noche**
- **Lugar:** Museo Nacional de Colombia
- **Actividad:** Visitar el museo, explorando su colección histórica y artística.
- **Horario:** 2:00 PM - 5:00 PM
- **Transporte:** Bici (si disponible)
- **Costo estimado:** $25 USD

### DÍA 2

**Mañana**
- **Lugar:** Chorro de Quevedo
- **Actividad:** Explorar el espacio tradicional, visitando tiendas locales y cenotes.
- **Horario:** 9:00 AM - 12:00 PM
- **Transporte:** Bici (si disponible)
- **Costo estimado:** $10 USD

**Tarde**
- **Lugar:** B

In [35]:
def validate_extended_rag_response(response):

    issues = []

    # Lugares autorizados por el contexto ampliado
    allowed_places_extended = [
        "Usaquén",
        "Zona G",
        "Mercado de La Perseverancia",
        "Parque de la 93",
        "La Candelaria",
        "Chorro de Quevedo",
        "Museo Nacional de Colombia",
        "Biblioteca Luis Ángel Arango",
        "Museo Botero",
        "Monserrate"
    ]

    # 1. Detectar precios
    prices = re.findall(
        r"\$\s?\d+\s?USD?",
        response,
        flags=re.IGNORECASE
    )

    if prices:
        issues.append(
            f"Se detectaron precios no respaldados: {prices}"
        )

    # 2. Detectar horarios
    times = re.findall(
        r"\b\d{1,2}:\d{2}\s?(?:AM|PM)\b",
        response,
        flags=re.IGNORECASE
    )

    if times:
        issues.append(
            f"Se detectaron horarios no respaldados: {times}"
        )

    # 3. Detectar transporte no respaldado
    unsupported_transport = [
        "bici",
        "bicicleta",
        "subte",
        "metro",
        "autobús",
        "taxi"
    ]

    for transport in unsupported_transport:
        if transport.lower() in response.lower():
            issues.append(
                f"Transporte no respaldado por el contexto: {transport}"
            )

    # 4. Contar días
    day_count = len(
        re.findall(
            r"DÍA\s+\d+",
            response,
            flags=re.IGNORECASE
        )
    )

    if day_count != travel_preferences["days"]:
        issues.append(
            f"Se esperaban {travel_preferences['days']} días, pero se detectaron {day_count}."
        )

    # 5. Comprobar mañana, tarde y noche por día
    day_sections = re.split(
        r"###\s*DÍA\s+\d+",
        response,
        flags=re.IGNORECASE
    )[1:]

    for i, section in enumerate(day_sections, start=1):

        if "Mañana" not in section:
            issues.append(f"Día {i}: falta la sección Mañana.")

        if "Tarde" not in section:
            issues.append(f"Día {i}: falta la sección Tarde.")

        if "Noche" not in section:
            issues.append(f"Día {i}: falta la sección Noche.")

    # 6. Repeticiones
    repetitions = {}

    for place in allowed_places_extended:
        count = response.lower().count(place.lower())
        repetitions[place] = count

        if count > 2:
            issues.append(
                f"El lugar '{place}' aparece {count} veces."
            )

    return issues, repetitions


extended_issues, extended_repetitions = validate_extended_rag_response(
    rag_itinerary_extended
)

print("VALIDACIÓN DEL RAG AMPLIADO\n")

if extended_issues:
    print("❌ Se detectaron problemas:\n")

    for issue in extended_issues:
        print("-", issue)

else:
    print("✅ El itinerario pasó todas las validaciones.")

print("\nREPETICIONES DE LUGARES:")

for place, count in extended_repetitions.items():
    print(f"- {place}: {count}")

VALIDACIÓN DEL RAG AMPLIADO

❌ Se detectaron problemas:

- Se detectaron precios no respaldados: ['$20 USD', '$15 USD', '$25 USD', '$10 USD', '$15 USD', '$20 USD', '$10 USD', '$20 USD', '$15 USD', '$10 USD', '$20 USD']
- Se detectaron horarios no respaldados: ['9:00 AM', '12:00 PM', '12:00 PM', '2:00 PM', '2:00 PM', '5:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '2:00 PM', '2:00 PM', '5:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '3:00 PM', '3:00 PM', '6:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '3:00 PM']
- Transporte no respaldado por el contexto: bici
- Día 4: falta la sección Noche.
- El lugar 'Usaquén' aparece 3 veces.
- El lugar 'Mercado de La Perseverancia' aparece 4 veces.
- El lugar 'Parque de la 93' aparece 3 veces.
- El lugar 'Museo Nacional de Colombia' aparece 4 veces.
- El lugar 'Museo Botero' aparece 5 veces.

REPETICIONES DE LUGARES:
- Usaquén: 3
- Zona G: 0
- Mercado de La Perseverancia: 4
- Parque de la 93: 3
- La Candelaria: 2
- Chorro de Quevedo: 2
- Museo Nacional d

In [36]:
def count_places_only_in_itinerary(response, allowed_places):

    # Tomamos únicamente el contenido previo a la evaluación final
    itinerary_only = re.split(
        r"EVALUACIÓN DEL ITINERARIO",
        response,
        flags=re.IGNORECASE
    )[0]

    repetitions = {}

    for place in allowed_places:
        count = itinerary_only.lower().count(place.lower())
        repetitions[place] = count

    return repetitions


allowed_places_extended = [
    "Usaquén",
    "Zona G",
    "Mercado de La Perseverancia",
    "Parque de la 93",
    "La Candelaria",
    "Chorro de Quevedo",
    "Museo Nacional de Colombia",
    "Biblioteca Luis Ángel Arango",
    "Museo Botero",
    "Monserrate"
]

real_repetitions = count_places_only_in_itinerary(
    rag_itinerary_extended,
    allowed_places_extended
)

print("REPETICIONES REALES DENTRO DEL ITINERARIO\n")

for place, count in real_repetitions.items():
    print(f"- {place}: {count}")

REPETICIONES REALES DENTRO DEL ITINERARIO

- Usaquén: 2
- Zona G: 0
- Mercado de La Perseverancia: 1
- Parque de la 93: 2
- La Candelaria: 1
- Chorro de Quevedo: 1
- Museo Nacional de Colombia: 1
- Biblioteca Luis Ángel Arango: 1
- Museo Botero: 2
- Monserrate: 1


In [37]:
def validate_final_rag_response(response):

    issues = []

    # Separar solamente el itinerario
    itinerary_only = re.split(
        r"EVALUACIÓN DEL ITINERARIO",
        response,
        flags=re.IGNORECASE
    )[0]

    # 1. Detectar precios no respaldados
    prices = re.findall(
        r"\$\s?\d+\s?USD?",
        itinerary_only,
        flags=re.IGNORECASE
    )

    if prices:
        issues.append(
            f"Se detectaron precios no respaldados: {prices}"
        )

    # 2. Detectar horarios no respaldados
    times = re.findall(
        r"\b\d{1,2}:\d{2}\s?(?:AM|PM)\b",
        itinerary_only,
        flags=re.IGNORECASE
    )

    if times:
        issues.append(
            f"Se detectaron horarios no respaldados: {times}"
        )

    # 3. Detectar transportes no respaldados
    unsupported_transport = [
        "bici",
        "bicicleta",
        "subte",
        "metro",
        "autobús",
        "taxi"
    ]

    for transport in unsupported_transport:
        if transport.lower() in itinerary_only.lower():
            issues.append(
                f"Transporte no respaldado por el contexto: {transport}"
            )

    # 4. Comprobar exactamente 4 días
    day_count = len(
        re.findall(
            r"DÍA\s+\d+",
            itinerary_only,
            flags=re.IGNORECASE
        )
    )

    if day_count != travel_preferences["days"]:
        issues.append(
            f"Se esperaban {travel_preferences['days']} días, pero se detectaron {day_count}."
        )

    # 5. Verificar mañana, tarde y noche
    day_sections = re.split(
        r"###\s*DÍA\s+\d+",
        itinerary_only,
        flags=re.IGNORECASE
    )[1:]

    for i, section in enumerate(day_sections, start=1):

        if "Mañana" not in section:
            issues.append(f"Día {i}: falta la sección Mañana.")

        if "Tarde" not in section:
            issues.append(f"Día {i}: falta la sección Tarde.")

        if "Noche" not in section:
            issues.append(f"Día {i}: falta la sección Noche.")

    # 6. Repeticiones reales
    repetitions = count_places_only_in_itinerary(
        response,
        allowed_places_extended
    )

    for place, count in repetitions.items():
        if count > 2:
            issues.append(
                f"El lugar '{place}' aparece {count} veces."
            )

    return issues, repetitions


final_issues, final_repetitions = validate_final_rag_response(
    rag_itinerary_extended
)

print("VALIDACIÓN FINAL DEL RAG AMPLIADO\n")

if final_issues:
    print("❌ Problemas pendientes:\n")
    for issue in final_issues:
        print("-", issue)
else:
    print("✅ El itinerario cumple todas las validaciones.")

print("\nREPETICIONES REALES:")
for place, count in final_repetitions.items():
    print(f"- {place}: {count}")

VALIDACIÓN FINAL DEL RAG AMPLIADO

❌ Problemas pendientes:

- Se detectaron precios no respaldados: ['$20 USD', '$15 USD', '$25 USD', '$10 USD', '$15 USD', '$20 USD', '$10 USD', '$20 USD', '$15 USD', '$10 USD', '$20 USD']
- Se detectaron horarios no respaldados: ['9:00 AM', '12:00 PM', '12:00 PM', '2:00 PM', '2:00 PM', '5:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '2:00 PM', '2:00 PM', '5:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '3:00 PM', '3:00 PM', '6:00 PM', '9:00 AM', '12:00 PM', '12:00 PM', '3:00 PM']
- Transporte no respaldado por el contexto: bici
- Día 4: falta la sección Noche.

REPETICIONES REALES:
- Usaquén: 2
- Zona G: 0
- Mercado de La Perseverancia: 1
- Parque de la 93: 2
- La Candelaria: 1
- Chorro de Quevedo: 1
- Museo Nacional de Colombia: 1
- Biblioteca Luis Ángel Arango: 1
- Museo Botero: 2
- Monserrate: 1


In [38]:
def sanitize_extended_itinerary(response):

    cleaned = response

    # 1. Reemplazar precios no respaldados
    cleaned = re.sub(
        r"\$\s?\d+\s?USD?",
        "no disponible en el contexto",
        cleaned,
        flags=re.IGNORECASE
    )

    # 2. Reemplazar rangos horarios no respaldados
    cleaned = re.sub(
        r"\b\d{1,2}:\d{2}\s?(?:AM|PM)\s*-\s*\d{1,2}:\d{2}\s?(?:AM|PM)\b",
        "debe verificarse",
        cleaned,
        flags=re.IGNORECASE
    )

    # 3. Reemplazar transporte no respaldado
    cleaned = re.sub(
        r"\bBici(?:cleta)?\s*(?:\(si disponible\))?",
        "por definir según disponibilidad local",
        cleaned,
        flags=re.IGNORECASE
    )

    # 4. Verificar si el Día 4 tiene sección Noche
    day4_match = re.search(
        r"(###\s*DÍA\s*4.*?)(?=###\s*EVALUACIÓN DEL ITINERARIO|\Z)",
        cleaned,
        flags=re.IGNORECASE | re.DOTALL
    )

    if day4_match:
        day4_content = day4_match.group(1)

        if "Noche" not in day4_content:
            night_section = """
**Noche**
- **Lugar:** Información no disponible en el contexto
- **Actividad:** No se añade una actividad para evitar inventar información.
- **Horario:** debe verificarse
- **Transporte:** por definir según disponibilidad local
- **Costo estimado:** no disponible en el contexto

"""
            cleaned = cleaned.replace(
                day4_content,
                day4_content + night_section
            )

    return cleaned


final_sanitized_itinerary = sanitize_extended_itinerary(
    rag_itinerary_extended
)

print("ITINERARIO FINAL SANEADO\n")
print(final_sanitized_itinerary)

ITINERARIO FINAL SANEADO

### DÍA 1

**Mañana**
- **Lugar:** Usaquén
- **Actividad:** Caminar por las calles de Usaquén, visitando restaurantes locales y café típicos.
- **Horario:** debe verificarse
- **Transporte:** por definir según disponibilidad local
- **Costo estimado:** no disponible en el contexto

**Tarde**
- **Lugar:** Parque de la 93
- **Actividad:** Recorrer el parque, disfrutando de sus vistas panorámicas y paseando por sus áreas verdes.
- **Horario:** debe verificarse
- **Transporte:** por definir según disponibilidad local
- **Costo estimado:** no disponible en el contexto

**Noche**
- **Lugar:** Museo Nacional de Colombia
- **Actividad:** Visitar el museo, explorando su colección histórica y artística.
- **Horario:** debe verificarse
- **Transporte:** por definir según disponibilidad local
- **Costo estimado:** no disponible en el contexto

### DÍA 2

**Mañana**
- **Lugar:** Chorro de Quevedo
- **Actividad:** Explorar el espacio tradicional, visitando tiendas locales y

In [39]:
sanitized_issues, sanitized_repetitions = validate_final_rag_response(
    final_sanitized_itinerary
)

print("VALIDACIÓN DEL ITINERARIO SANEADO\n")

if sanitized_issues:
    print("❌ Problemas pendientes:\n")
    for issue in sanitized_issues:
        print("-", issue)
else:
    print("✅ El itinerario pasó todas las validaciones estructurales.")

print("\nREPETICIONES REALES:")
for place, count in sanitized_repetitions.items():
    print(f"- {place}: {count}")

VALIDACIÓN DEL ITINERARIO SANEADO

✅ El itinerario pasó todas las validaciones estructurales.

REPETICIONES REALES:
- Usaquén: 2
- Zona G: 0
- Mercado de La Perseverancia: 1
- Parque de la 93: 2
- La Candelaria: 1
- Chorro de Quevedo: 1
- Museo Nacional de Colombia: 1
- Biblioteca Luis Ángel Arango: 1
- Museo Botero: 2
- Monserrate: 1


In [40]:
# Crear un diccionario lugar -> documento autorizado
place_context_map = {}

for doc in retrieved_docs_extended:
    lines = [line.strip() for line in doc.split("\n") if line.strip()]

    if lines:
        place_name = lines[0].replace(":", "").strip()
        place_context_map[place_name] = doc


def evaluate_semantic_grounding(response, threshold=0.50):

    # Analizar únicamente el itinerario
    itinerary_only = re.split(
        r"EVALUACIÓN DEL ITINERARIO",
        response,
        flags=re.IGNORECASE
    )[0]

    # Extraer pares Lugar / Actividad
    pattern = re.compile(
        r"\*\*Lugar:\*\*\s*(.*?)\n"
        r"-\s*\*\*Actividad:\*\*\s*(.*?)(?:\n|$)",
        flags=re.IGNORECASE
    )

    pairs = pattern.findall(itinerary_only)

    results = []

    for place, activity in pairs:

        place = place.strip()
        activity = activity.strip()

        if place in place_context_map:

            authorized_context = place_context_map[place]

            embeddings = embedding_model.encode(
                [activity, authorized_context],
                convert_to_numpy=True,
                normalize_embeddings=True
            )

            similarity = float(
                np.dot(embeddings[0], embeddings[1])
            )

            status = (
                "✅ Relación semántica aceptable"
                if similarity >= threshold
                else "⚠️ Revisar grounding"
            )

            results.append({
                "lugar": place,
                "actividad": activity,
                "similitud": round(similarity, 3),
                "estado": status
            })

        else:
            results.append({
                "lugar": place,
                "actividad": activity,
                "similitud": None,
                "estado": "⚠️ Lugar sin contexto recuperado"
            })

    return results


semantic_results = evaluate_semantic_grounding(
    final_sanitized_itinerary
)

print("EVALUACIÓN DE GROUNDING SEMÁNTICO\n")

for result in semantic_results:

    print("Lugar:", result["lugar"])
    print("Actividad:", result["actividad"])
    print("Similitud:", result["similitud"])
    print("Estado:", result["estado"])
    print("-" * 70)

EVALUACIÓN DE GROUNDING SEMÁNTICO

Lugar: Usaquén
Actividad: Caminar por las calles de Usaquén, visitando restaurantes locales y café típicos.
Similitud: 0.549
Estado: ✅ Relación semántica aceptable
----------------------------------------------------------------------
Lugar: Parque de la 93
Actividad: Recorrer el parque, disfrutando de sus vistas panorámicas y paseando por sus áreas verdes.
Similitud: 0.472
Estado: ⚠️ Revisar grounding
----------------------------------------------------------------------
Lugar: Museo Nacional de Colombia
Actividad: Visitar el museo, explorando su colección histórica y artística.
Similitud: 0.602
Estado: ✅ Relación semántica aceptable
----------------------------------------------------------------------
Lugar: Chorro de Quevedo
Actividad: Explorar el espacio tradicional, visitando tiendas locales y cenotes.
Similitud: 0.51
Estado: ✅ Relación semántica aceptable
----------------------------------------------------------------------
Lugar: Biblioteca L

In [41]:
authorized_places = {
    "Usaquén": {
        "category": ["gastronomía", "cultura", "paseo"],
        "allowed_activities": [
            "caminar",
            "visitar restaurantes",
            "visitar cafés",
            "observar arquitectura",
            "realizar actividades culturales"
        ]
    },

    "Zona G": {
        "category": ["gastronomía"],
        "allowed_activities": [
            "experiencia culinaria",
            "visitar restaurantes",
            "cenar"
        ]
    },

    "Mercado de La Perseverancia": {
        "category": ["gastronomía"],
        "allowed_activities": [
            "conocer comida colombiana",
            "explorar gastronomía local"
        ]
    },

    "Parque de la 93": {
        "category": ["gastronomía", "paseo"],
        "allowed_activities": [
            "caminar",
            "visitar restaurantes",
            "visitar cafés",
            "realizar actividades relajadas"
        ]
    },

    "La Candelaria": {
        "category": ["cultura", "historia"],
        "allowed_activities": [
            "recorrer arquitectura colonial",
            "caminar",
            "visitar museos",
            "visitar plazas",
            "realizar recorridos culturales"
        ]
    },

    "Chorro de Quevedo": {
        "category": ["cultura", "historia"],
        "allowed_activities": [
            "conocer historia",
            "observar arquitectura",
            "realizar recorridos culturales"
        ]
    },

    "Museo Nacional de Colombia": {
        "category": ["cultura", "historia", "arte"],
        "allowed_activities": [
            "conocer historia",
            "conocer patrimonio",
            "observar arte",
            "realizar visita cultural"
        ]
    },

    "Biblioteca Luis Ángel Arango": {
        "category": ["cultura"],
        "allowed_activities": [
            "actividades de lectura",
            "conocer patrimonio",
            "observar arte",
            "realizar actividades culturales"
        ]
    },

    "Museo Botero": {
        "category": ["arte", "cultura"],
        "allowed_activities": [
            "observar obras de Fernando Botero",
            "observar obras de artistas internacionales",
            "realizar visita cultural"
        ]
    },

    "Monserrate": {
        "category": ["paseo", "turismo"],
        "allowed_activities": [
            "observar vistas de la ciudad",
            "subir en teleférico",
            "subir en funicular"
        ]
    }
}

print("Lugares estructurados:", len(authorized_places))

for place, data in authorized_places.items():
    print("\n", place)
    print(" Categorías:", data["category"])
    print(" Actividades permitidas:", data["allowed_activities"])

Lugares estructurados: 10

 Usaquén
 Categorías: ['gastronomía', 'cultura', 'paseo']
 Actividades permitidas: ['caminar', 'visitar restaurantes', 'visitar cafés', 'observar arquitectura', 'realizar actividades culturales']

 Zona G
 Categorías: ['gastronomía']
 Actividades permitidas: ['experiencia culinaria', 'visitar restaurantes', 'cenar']

 Mercado de La Perseverancia
 Categorías: ['gastronomía']
 Actividades permitidas: ['conocer comida colombiana', 'explorar gastronomía local']

 Parque de la 93
 Categorías: ['gastronomía', 'paseo']
 Actividades permitidas: ['caminar', 'visitar restaurantes', 'visitar cafés', 'realizar actividades relajadas']

 La Candelaria
 Categorías: ['cultura', 'historia']
 Actividades permitidas: ['recorrer arquitectura colonial', 'caminar', 'visitar museos', 'visitar plazas', 'realizar recorridos culturales']

 Chorro de Quevedo
 Categorías: ['cultura', 'historia']
 Actividades permitidas: ['conocer historia', 'observar arquitectura', 'realizar recorridos 

In [42]:
import json

authorized_places_json = json.dumps(
    authorized_places,
    ensure_ascii=False,
    indent=2
)

structured_prompt = f"""
Actúa como un planificador profesional de viajes.

Debes generar un itinerario de exactamente {travel_preferences['days']} días
para {travel_preferences['destination']}.

REGLA PRINCIPAL:
Solo puedes utilizar lugares y actividades que aparezcan exactamente
en la BASE AUTORIZADA.

BASE AUTORIZADA:
{authorized_places_json}

PREFERENCIAS:
- Intereses: {', '.join(travel_preferences['interests'])}
- Ritmo: {travel_preferences['travel_style']}
- Restricciones: {', '.join(travel_preferences['restrictions'])}

REGLAS:

1. Cada día debe tener exactamente:
   - mañana
   - tarde
   - noche

2. Para cada bloque debes seleccionar:
   - un lugar existente en la BASE AUTORIZADA;
   - una actividad perteneciente exactamente a "allowed_activities"
     de ese lugar.

3. No inventes:
   - actividades;
   - lugares;
   - precios;
   - horarios;
   - medios de transporte;
   - restaurantes concretos;
   - eventos;
   - características adicionales.

4. Intenta no repetir un lugar más de dos veces.

5. Prioriza categorías relacionadas con:
   - cultura
   - gastronomía

6. Devuelve ÚNICAMENTE JSON válido.

7. No escribas explicaciones antes ni después del JSON.

FORMATO EXACTO:

{{
  "dias": [
    {{
      "dia": 1,
      "manana": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta autorizada"
      }},
      "tarde": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta autorizada"
      }},
      "noche": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta autorizada"
      }}
    }}
  ]
}}
"""

print(structured_prompt)


Actúa como un planificador profesional de viajes.

Debes generar un itinerario de exactamente 4 días
para Bogotá, Colombia.

REGLA PRINCIPAL:
Solo puedes utilizar lugares y actividades que aparezcan exactamente
en la BASE AUTORIZADA.

BASE AUTORIZADA:
{
  "Usaquén": {
    "category": [
      "gastronomía",
      "cultura",
      "paseo"
    ],
    "allowed_activities": [
      "caminar",
      "visitar restaurantes",
      "visitar cafés",
      "observar arquitectura",
      "realizar actividades culturales"
    ]
  },
  "Zona G": {
    "category": [
      "gastronomía"
    ],
    "allowed_activities": [
      "experiencia culinaria",
      "visitar restaurantes",
      "cenar"
    ]
  },
  "Mercado de La Perseverancia": {
    "category": [
      "gastronomía"
    ],
    "allowed_activities": [
      "conocer comida colombiana",
      "explorar gastronomía local"
    ]
  },
  "Parque de la 93": {
    "category": [
      "gastronomía",
      "paseo"
    ],
    "allowed_activities": [


In [43]:
messages_structured = [
    {
        "role": "system",
        "content": (
            "Eres un planificador de viajes que debe cumplir estrictamente "
            "con esquemas estructurados y devolver únicamente JSON válido."
        )
    },
    {
        "role": "user",
        "content": structured_prompt
    }
]

text_structured = tokenizer.apply_chat_template(
    messages_structured,
    tokenize=False,
    add_generation_prompt=True
)

inputs_structured = tokenizer(
    text_structured,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_structured = model.generate(
        **inputs_structured,
        max_new_tokens=1400,
        do_sample=False
    )

structured_response = tokenizer.decode(
    outputs_structured[0][inputs_structured["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("RESPUESTA ESTRUCTURADA DEL MODELO\n")
print(structured_response)

RESPUESTA ESTRUCTURADA DEL MODELO

```json
{
  "dias": [
    {
      "dia": 1,
      "manana": {
        "lugar": "Usaquén",
        "actividad": "caminar"
      },
      "tarde": {
        "lugar": "Mercado de La Perseverancia",
        "actividad": "conocer comida colombiana"
      },
      "noche": {
        "lugar": "La Candelaria",
        "actividad": "recorrer arquitectura colonial"
      }
    },
    {
      "dia": 2,
      "manana": {
        "lugar": "Zona G",
        "actividad": "experiencia culinaria"
      },
      "tarde": {
        "lugar": "Parque de la 93",
        "actividad": "caminar"
      },
      "noche": {
        "lugar": "Monserrate",
        "actividad": "observar vistas de la ciudad"
      }
    },
    {
      "dia": 3,
      "manana": {
        "lugar": "Chorro de Quevedo",
        "actividad": "conocer historia"
      },
      "tarde": {
        "lugar": "Museo Nacional de Colombia",
        "actividad": "conocer historia"
      },
      "noche": {
      

In [44]:
import json
import re

def clean_json_response(response):

    cleaned = response.strip()

    # Eliminar bloques Markdown ```json ... ```
    cleaned = re.sub(
        r"^```json\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    return cleaned.strip()


cleaned_structured_response = clean_json_response(
    structured_response
)

validation_errors = []

try:
    structured_itinerary = json.loads(
        cleaned_structured_response
    )

    print("✅ JSON válido y parseado correctamente.")

except json.JSONDecodeError as e:
    structured_itinerary = None
    validation_errors.append(
        f"JSON inválido: {e}"
    )


if structured_itinerary is not None:

    # 1. Validar existencia de "dias"
    if "dias" not in structured_itinerary:
        validation_errors.append(
            "Falta la clave principal 'dias'."
        )

    else:

        days = structured_itinerary["dias"]

        # 2. Validar número exacto de días
        if len(days) != travel_preferences["days"]:
            validation_errors.append(
                f"Se esperaban {travel_preferences['days']} días "
                f"y se encontraron {len(days)}."
            )

        # 3. Validar estructura y grounding
        periods = ["manana", "tarde", "noche"]

        for expected_day, day in enumerate(days, start=1):

            if day.get("dia") != expected_day:
                validation_errors.append(
                    f"Número de día incorrecto: "
                    f"se esperaba {expected_day}."
                )

            for period in periods:

                if period not in day:
                    validation_errors.append(
                        f"Día {expected_day}: falta '{period}'."
                    )
                    continue

                block = day[period]

                place = block.get("lugar")
                activity = block.get("actividad")

                # Validar lugar
                if place not in authorized_places:
                    validation_errors.append(
                        f"Día {expected_day} - {period}: "
                        f"lugar no autorizado '{place}'."
                    )
                    continue

                # Validar actividad exacta
                allowed_activities = authorized_places[
                    place
                ]["allowed_activities"]

                if activity not in allowed_activities:
                    validation_errors.append(
                        f"Día {expected_day} - {period}: "
                        f"actividad '{activity}' no autorizada "
                        f"para '{place}'."
                    )


print("\nVALIDACIÓN DEL ITINERARIO ESTRUCTURADO\n")

if validation_errors:
    print("❌ Se detectaron problemas:\n")

    for error in validation_errors:
        print("-", error)

else:
    print("✅ El itinerario cumple el esquema y el grounding autorizado.")

✅ JSON válido y parseado correctamente.

VALIDACIÓN DEL ITINERARIO ESTRUCTURADO

✅ El itinerario cumple el esquema y el grounding autorizado.


In [45]:
fine_tuning_examples = [
    {
        "instruction": """
Genera un itinerario de 2 días para Bogotá orientado a cultura y gastronomía.
Usa únicamente lugares y actividades autorizadas.
Devuelve únicamente JSON válido.
""".strip(),

        "response": {
            "dias": [
                {
                    "dia": 1,
                    "manana": {
                        "lugar": "La Candelaria",
                        "actividad": "realizar recorridos culturales"
                    },
                    "tarde": {
                        "lugar": "Museo Botero",
                        "actividad": "observar obras de Fernando Botero"
                    },
                    "noche": {
                        "lugar": "Zona G",
                        "actividad": "cenar"
                    }
                },
                {
                    "dia": 2,
                    "manana": {
                        "lugar": "Museo Nacional de Colombia",
                        "actividad": "conocer historia"
                    },
                    "tarde": {
                        "lugar": "Mercado de La Perseverancia",
                        "actividad": "explorar gastronomía local"
                    },
                    "noche": {
                        "lugar": "Usaquén",
                        "actividad": "visitar restaurantes"
                    }
                }
            ]
        }
    },

    {
        "instruction": """
Genera un itinerario de 2 días para Bogotá con actividades culturales,
históricas y relajadas.
Usa únicamente lugares y actividades autorizadas.
Devuelve únicamente JSON válido.
""".strip(),

        "response": {
            "dias": [
                {
                    "dia": 1,
                    "manana": {
                        "lugar": "Chorro de Quevedo",
                        "actividad": "conocer historia"
                    },
                    "tarde": {
                        "lugar": "Biblioteca Luis Ángel Arango",
                        "actividad": "realizar actividades culturales"
                    },
                    "noche": {
                        "lugar": "Usaquén",
                        "actividad": "caminar"
                    }
                },
                {
                    "dia": 2,
                    "manana": {
                        "lugar": "Monserrate",
                        "actividad": "observar vistas de la ciudad"
                    },
                    "tarde": {
                        "lugar": "Parque de la 93",
                        "actividad": "realizar actividades relajadas"
                    },
                    "noche": {
                        "lugar": "Zona G",
                        "actividad": "experiencia culinaria"
                    }
                }
            ]
        }
    },

    {
        "instruction": """
Genera un itinerario de 1 día para Bogotá priorizando arte y cultura.
Usa únicamente lugares y actividades autorizadas.
Devuelve únicamente JSON válido.
""".strip(),

        "response": {
            "dias": [
                {
                    "dia": 1,
                    "manana": {
                        "lugar": "Museo Botero",
                        "actividad": "observar obras de artistas internacionales"
                    },
                    "tarde": {
                        "lugar": "Museo Nacional de Colombia",
                        "actividad": "observar arte"
                    },
                    "noche": {
                        "lugar": "La Candelaria",
                        "actividad": "caminar"
                    }
                }
            ]
        }
    }
]

print("Ejemplos creados:", len(fine_tuning_examples))

for i, example in enumerate(fine_tuning_examples, start=1):
    print(f"\nEjemplo {i}")
    print("Instruction:", example["instruction"])
    print("Response:", json.dumps(
        example["response"],
        ensure_ascii=False,
        indent=2
    ))

Ejemplos creados: 3

Ejemplo 1
Instruction: Genera un itinerario de 2 días para Bogotá orientado a cultura y gastronomía.
Usa únicamente lugares y actividades autorizadas.
Devuelve únicamente JSON válido.
Response: {
  "dias": [
    {
      "dia": 1,
      "manana": {
        "lugar": "La Candelaria",
        "actividad": "realizar recorridos culturales"
      },
      "tarde": {
        "lugar": "Museo Botero",
        "actividad": "observar obras de Fernando Botero"
      },
      "noche": {
        "lugar": "Zona G",
        "actividad": "cenar"
      }
    },
    {
      "dia": 2,
      "manana": {
        "lugar": "Museo Nacional de Colombia",
        "actividad": "conocer historia"
      },
      "tarde": {
        "lugar": "Mercado de La Perseverancia",
        "actividad": "explorar gastronomía local"
      },
      "noche": {
        "lugar": "Usaquén",
        "actividad": "visitar restaurantes"
      }
    }
  ]
}

Ejemplo 2
Instruction: Genera un itinerario de 2 días para B

In [46]:
from datasets import Dataset
import json

training_records = []

for example in fine_tuning_examples:

    messages = [
        {
            "role": "system",
            "content": (
                "Eres un planificador profesional de viajes. "
                "Debes seguir estrictamente las instrucciones del usuario "
                "y devolver únicamente JSON válido."
            )
        },
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": json.dumps(
                example["response"],
                ensure_ascii=False
            )
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    training_records.append({
        "text": formatted_text
    })


fine_tuning_dataset = Dataset.from_list(
    training_records
)

print("Dataset creado correctamente.")
print("Número de ejemplos:", len(fine_tuning_dataset))
print("\nColumnas:", fine_tuning_dataset.column_names)

print("\nPRIMER EJEMPLO FORMATEADO:\n")
print(fine_tuning_dataset[0]["text"])

Dataset creado correctamente.
Número de ejemplos: 3

Columnas: ['text']

PRIMER EJEMPLO FORMATEADO:

<|im_start|>system
Eres un planificador profesional de viajes. Debes seguir estrictamente las instrucciones del usuario y devolver únicamente JSON válido.<|im_end|>
<|im_start|>user
Genera un itinerario de 2 días para Bogotá orientado a cultura y gastronomía.
Usa únicamente lugares y actividades autorizadas.
Devuelve únicamente JSON válido.<|im_end|>
<|im_start|>assistant
{"dias": [{"dia": 1, "manana": {"lugar": "La Candelaria", "actividad": "realizar recorridos culturales"}, "tarde": {"lugar": "Museo Botero", "actividad": "observar obras de Fernando Botero"}, "noche": {"lugar": "Zona G", "actividad": "cenar"}}, {"dia": 2, "manana": {"lugar": "Museo Nacional de Colombia", "actividad": "conocer historia"}, "tarde": {"lugar": "Mercado de La Perseverancia", "actividad": "explorar gastronomía local"}, "noche": {"lugar": "Usaquén", "actividad": "visitar restaurantes"}}]}<|im_end|>



In [47]:
!pip install -q -U "torchao>=0.17.0"

import torchao

print("TorchAO:", torchao.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.5 MB/s eta 0:00:00


TorchAO: 0.18.0


In [48]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model_lora = get_peft_model(
    model,
    lora_config
)

print("Modelo preparado con LoRA.\n")

model_lora.print_trainable_parameters()

Modelo preparado con LoRA.

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [49]:
def tokenize_training_example(example):

    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=1024,
        padding=False
    )

    # Para causal language modeling:
    # el modelo intenta predecir el siguiente token
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


tokenized_dataset = fine_tuning_dataset.map(
    tokenize_training_example,
    remove_columns=fine_tuning_dataset.column_names
)

print("Dataset tokenizado correctamente.")
print("Número de ejemplos:", len(tokenized_dataset))

print("\nColumnas:")
print(tokenized_dataset.column_names)

print("\nLongitud en tokens de cada ejemplo:")

for i in range(len(tokenized_dataset)):
    print(
        f"Ejemplo {i + 1}:",
        len(tokenized_dataset[i]["input_ids"]),
        "tokens"
    )

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset tokenizado correctamente.
Número de ejemplos: 3

Columnas:
['input_ids', 'attention_mask', 'labels']

Longitud en tokens de cada ejemplo:
Ejemplo 1: 263 tokens
Ejemplo 2: 272 tokens
Ejemplo 3: 174 tokens


In [50]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_travel_lora",

    num_train_epochs=5,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=1,

    save_strategy="no",

    report_to="none",

    remove_unused_columns=False
)

print("TrainingArguments configurados correctamente.")
print("Épocas:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)

TrainingArguments configurados correctamente.
Épocas: 5
Batch size: 1
Gradient accumulation: 2
Learning rate: 0.0002
FP16: True


In [51]:
from transformers import Trainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_lora,
    padding=True,
    return_tensors="pt"
)

trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("✅ Trainer creado correctamente.")
print("Ejemplos de entrenamiento:", len(trainer.train_dataset))
print("Modelo:", model_lora.__class__.__name__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

✅ Trainer creado correctamente.
Ejemplos de entrenamiento: 3
Modelo: PeftModelForCausalLM
GPU disponible: True
GPU: Tesla T4


In [52]:
train_result = trainer.train()

print("\n✅ Entrenamiento finalizado.")

print("\nMétricas:")
print(train_result.metrics)

Step,Training Loss
1,1.913284
2,1.661401
3,1.605550
4,1.939825
5,1.655316
6,1.476805
7,1.574385
8,1.417142
9,1.515907
10,1.396513



✅ Entrenamiento finalizado.

Métricas:
{'train_runtime': 8.189, 'train_samples_per_second': 1.832, 'train_steps_per_second': 1.221, 'total_flos': 27917293593600.0, 'train_loss': 1.6156126499176025, 'epoch': 5.0}


In [53]:
test_prompt_finetuned = """
Genera un itinerario de 2 días para Bogotá orientado a cultura y gastronomía.
Devuelve únicamente JSON válido.
Cada día debe contener mañana, tarde y noche.
"""

messages_test_finetuned = [
    {
        "role": "system",
        "content": (
            "Eres un planificador profesional de viajes. "
            "Debes seguir estrictamente las instrucciones del usuario "
            "y devolver únicamente JSON válido."
        )
    },
    {
        "role": "user",
        "content": test_prompt_finetuned
    }
]

text_test_finetuned = tokenizer.apply_chat_template(
    messages_test_finetuned,
    tokenize=False,
    add_generation_prompt=True
)

inputs_test_finetuned = tokenizer(
    text_test_finetuned,
    return_tensors="pt"
).to(model_lora.device)

model_lora.eval()

with torch.no_grad():
    outputs_test_finetuned = model_lora.generate(
        **inputs_test_finetuned,
        max_new_tokens=800,
        do_sample=False
    )

finetuned_response = tokenizer.decode(
    outputs_test_finetuned[0][inputs_test_finetuned["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("RESPUESTA DEL MODELO FINE-TUNED\n")
print(finetuned_response)

RESPUESTA DEL MODELO FINE-TUNED

{
    "itinerary": [
        {
            "day": "1",
            "morning": "Visit the Botero Museum and enjoy lunch at La Mariscadora.",
            "afternoon": "Attend a cultural workshop on traditional Colombian music in El Recinto Cultural.",
            "evening": "Dinner at Casa de la Cultura."
        },
        {
            "day": "2",
            "morning": "Explore the historic center of Bogota with a guided tour.",
            "afternoon": "Enjoy a tasting session at Cervecería Nacional.",
            "evening": "End your day with a visit to the National Pantheon and dinner at Restaurante El Poblado."
        }
    ]
}


In [56]:
model_lora.eval()

with model_lora.disable_adapter():

    with torch.no_grad():

        outputs_base_comparison = model_lora.generate(
            **inputs_test_finetuned,
            max_new_tokens=800,
            do_sample=False
        )

base_comparison_response = tokenizer.decode(
    outputs_base_comparison[0][
        inputs_test_finetuned["input_ids"].shape[1]:
    ],
    skip_special_tokens=True
)

print("✅ Variable base_comparison_response recuperada.\n")
print("RESPUESTA DEL MODELO BASE SIN LoRA\n")
print(base_comparison_response)

✅ Variable base_comparison_response recuperada.

RESPUESTA DEL MODELO BASE SIN LoRA

```json
{
  "itinerary": [
    {
      "day": "1",
      "morning": ["Visit the National Museum of Colombia", "Attend a traditional Colombian coffee tasting"],
      "afternoon": ["Explore the Botanical Garden of Bogota"],
      "evening": ["Dinner at a local restaurant serving traditional Colombian cuisine"]
    },
    {
      "day": "2",
      "morning": ["Tour the Metropolitan Cathedral and Plaza de Bolivar"],
      "afternoon": ["Enjoy a street food market for lunch"],
      "evening": ["Watch a live Flamenco performance in the evening"]
    }
  ]
}
```


In [57]:
def compare_model_outputs(base_response, finetuned_response):

    comparison = {
        "base": {
            "json_valido": False,
            "usa_clave_dias": False,
            "idioma_espanol": False
        },
        "fine_tuned": {
            "json_valido": False,
            "usa_clave_dias": False,
            "idioma_espanol": False
        }
    }

    def analyze(response):

        cleaned = response.strip()

        cleaned = re.sub(
            r"^```json\s*",
            "",
            cleaned,
            flags=re.IGNORECASE
        )

        cleaned = re.sub(
            r"\s*```$",
            "",
            cleaned
        )

        result = {
            "json_valido": False,
            "usa_clave_dias": False,
            "idioma_espanol": False
        }

        try:
            data = json.loads(cleaned)

            result["json_valido"] = True

            if "dias" in data:
                result["usa_clave_dias"] = True

        except json.JSONDecodeError:
            pass

        spanish_markers = [
            "día",
            "mañana",
            "tarde",
            "noche",
            "itinerario"
        ]

        response_lower = response.lower()

        if any(
            marker in response_lower
            for marker in spanish_markers
        ):
            result["idioma_espanol"] = True

        return result


    comparison["base"] = analyze(base_response)

    comparison["fine_tuned"] = analyze(
        finetuned_response
    )

    return comparison


model_comparison = compare_model_outputs(
    base_comparison_response,
    finetuned_response
)

print("COMPARACIÓN MODELO BASE VS FINE-TUNED\n")

print("MODELO BASE")
for key, value in model_comparison["base"].items():
    print(f"- {key}: {value}")

print("\nMODELO FINE-TUNED")
for key, value in model_comparison["fine_tuned"].items():
    print(f"- {key}: {value}")

COMPARACIÓN MODELO BASE VS FINE-TUNED

MODELO BASE
- json_valido: True
- usa_clave_dias: False
- idioma_espanol: False

MODELO FINE-TUNED
- json_valido: True
- usa_clave_dias: False
- idioma_espanol: False


In [58]:
model_lora.eval()

with model_lora.disable_adapter():

    with torch.no_grad():

        outputs_base_comparison = model_lora.generate(
            **inputs_test_finetuned,
            max_new_tokens=800,
            do_sample=False
        )

base_comparison_response = tokenizer.decode(
    outputs_base_comparison[0][
        inputs_test_finetuned["input_ids"].shape[1]:
    ],
    skip_special_tokens=True
)

print("RESPUESTA DEL MODELO BASE SIN LoRA\n")
print(base_comparison_response)

RESPUESTA DEL MODELO BASE SIN LoRA

```json
{
  "itinerary": [
    {
      "day": "1",
      "morning": ["Visit the National Museum of Colombia", "Attend a traditional Colombian coffee tasting"],
      "afternoon": ["Explore the Botanical Garden of Bogota"],
      "evening": ["Dinner at a local restaurant serving traditional Colombian cuisine"]
    },
    {
      "day": "2",
      "morning": ["Tour the Metropolitan Cathedral and Plaza de Bolivar"],
      "afternoon": ["Enjoy a street food market for lunch"],
      "evening": ["Watch a live Flamenco performance in the evening"]
    }
  ]
}
```


In [59]:
hybrid_prompt = f"""
Genera un itinerario personalizado para Bogotá, Colombia.

Debes utilizar EXCLUSIVAMENTE la siguiente BASE AUTORIZADA:

{authorized_places_json}

PREFERENCIAS DEL VIAJE:
- Duración: {travel_preferences['days']} días
- Intereses: {', '.join(travel_preferences['interests'])}
- Ritmo: {travel_preferences['travel_style']}
- Restricciones: {', '.join(travel_preferences['restrictions'])}

REGLAS OBLIGATORIAS:

1. Genera exactamente {travel_preferences['days']} días.

2. Cada día debe contener exactamente:
   - manana
   - tarde
   - noche

3. Cada bloque debe contener:
   - lugar
   - actividad

4. El valor de "lugar" debe ser exactamente uno de los nombres
   presentes en la BASE AUTORIZADA.

5. El valor de "actividad" debe pertenecer exactamente a
   "allowed_activities" del lugar seleccionado.

6. No inventes lugares ni actividades.

7. No añadas:
   - precios
   - horarios
   - transporte
   - restaurantes específicos
   - eventos
   - explicaciones adicionales

8. Prioriza cultura y gastronomía.

9. Evita repetir un lugar más de dos veces.

10. Devuelve ÚNICAMENTE JSON válido.

11. Utiliza exactamente esta estructura:

{{
  "dias": [
    {{
      "dia": 1,
      "manana": {{
        "lugar": "...",
        "actividad": "..."
      }},
      "tarde": {{
        "lugar": "...",
        "actividad": "..."
      }},
      "noche": {{
        "lugar": "...",
        "actividad": "..."
      }}
    }}
  ]
}}
"""

messages_hybrid = [
    {
        "role": "system",
        "content": (
            "Eres un planificador profesional de viajes. "
            "Debes respetar estrictamente el contexto proporcionado "
            "y devolver únicamente JSON válido."
        )
    },
    {
        "role": "user",
        "content": hybrid_prompt
    }
]

text_hybrid = tokenizer.apply_chat_template(
    messages_hybrid,
    tokenize=False,
    add_generation_prompt=True
)

inputs_hybrid = tokenizer(
    text_hybrid,
    return_tensors="pt"
).to(model_lora.device)

model_lora.eval()

with torch.no_grad():
    outputs_hybrid = model_lora.generate(
        **inputs_hybrid,
        max_new_tokens=1400,
        do_sample=False
    )

hybrid_response = tokenizer.decode(
    outputs_hybrid[0][inputs_hybrid["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("RESPUESTA DEL MODELO HÍBRIDO\n")
print(hybrid_response)

RESPUESTA DEL MODELO HÍBRIDO

{
  "dias": [
    {
      "dia": 1,
      "manana": {
        "lugar": "Usaquén",
        "actividad": "caminar"
      },
      "tarde": {
        "lugar": "Mercado de La Perseverancia",
        "actividad": "conocer comida colombiana"
      },
      "noche": {
        "lugar": "La Candelaria",
        "actividad": "recorrer arquitectura colonial"
      }
    },
    {
      "dia": 2,
      "manana": {
        "lugar": "Chorro de Quevedo",
        "actividad": "conocer historia"
      },
      "tarde": {
        "lugar": "Museo Nacional de Colombia",
        "actividad": "conocer historia"
      },
      "noche": {
        "lugar": "Biblioteca Luis Ángel Arango",
        "actividad": "actividades de lectura"
      }
    },
    {
      "dia": 3,
      "manana": {
        "lugar": "Monserrate",
        "actividad": "observar vistas de la ciudad"
      },
      "tarde": {
        "lugar": "Museo Botero",
        "actividad": "observar obras de Fernando Botero"

In [60]:
cleaned_hybrid_response = clean_json_response(
    hybrid_response
)

hybrid_validation_errors = []

try:
    hybrid_itinerary = json.loads(
        cleaned_hybrid_response
    )

    print("✅ JSON híbrido válido y parseado correctamente.")

except json.JSONDecodeError as e:
    hybrid_itinerary = None
    hybrid_validation_errors.append(
        f"JSON inválido: {e}"
    )


if hybrid_itinerary is not None:

    # 1. Validar clave principal
    if "dias" not in hybrid_itinerary:
        hybrid_validation_errors.append(
            "Falta la clave principal 'dias'."
        )

    else:

        days = hybrid_itinerary["dias"]

        # 2. Validar número exacto de días
        if len(days) != travel_preferences["days"]:
            hybrid_validation_errors.append(
                f"Se esperaban {travel_preferences['days']} días "
                f"y se encontraron {len(days)}."
            )

        periods = ["manana", "tarde", "noche"]

        # 3. Validar estructura y grounding
        for expected_day, day in enumerate(days, start=1):

            if day.get("dia") != expected_day:
                hybrid_validation_errors.append(
                    f"Número de día incorrecto: "
                    f"se esperaba {expected_day}."
                )

            for period in periods:

                if period not in day:
                    hybrid_validation_errors.append(
                        f"Día {expected_day}: falta '{period}'."
                    )
                    continue

                block = day[period]

                place = block.get("lugar")
                activity = block.get("actividad")

                # Validar lugar autorizado
                if place not in authorized_places:
                    hybrid_validation_errors.append(
                        f"Día {expected_day} - {period}: "
                        f"lugar no autorizado '{place}'."
                    )
                    continue

                # Validar actividad autorizada
                allowed_activities = authorized_places[
                    place
                ]["allowed_activities"]

                if activity not in allowed_activities:
                    hybrid_validation_errors.append(
                        f"Día {expected_day} - {period}: "
                        f"actividad '{activity}' no autorizada "
                        f"para '{place}'."
                    )


print("\nVALIDACIÓN DEL MODELO HÍBRIDO\n")

if hybrid_validation_errors:

    print("❌ Se detectaron problemas:\n")

    for error in hybrid_validation_errors:
        print("-", error)

else:
    print(
        "✅ El itinerario híbrido cumple el esquema "
        "y el grounding autorizado."
    )

✅ JSON híbrido válido y parseado correctamente.

VALIDACIÓN DEL MODELO HÍBRIDO

✅ El itinerario híbrido cumple el esquema y el grounding autorizado.


In [61]:
comparison_results = [
    {
        "Enfoque": "Modelo base",
        "JSON válido": True,
        "Esquema correcto": False,
        "Grounding": False,
        "Alucinaciones": True,
        "Resultado": "Insuficiente"
    },
    {
        "Enfoque": "Prompting",
        "JSON válido": False,
        "Esquema correcto": "Parcial",
        "Grounding": False,
        "Alucinaciones": True,
        "Resultado": "Mejora estructura, no factualidad"
    },
    {
        "Enfoque": "RAG inicial",
        "JSON válido": False,
        "Esquema correcto": "Parcial",
        "Grounding": "Parcial",
        "Alucinaciones": True,
        "Resultado": "Reduce alucinaciones, pero contexto insuficiente"
    },
    {
        "Enfoque": "RAG ampliado",
        "JSON válido": False,
        "Esquema correcto": "Parcial",
        "Grounding": "Parcial",
        "Alucinaciones": True,
        "Resultado": "Mejora variedad, aún inventa detalles"
    },
    {
        "Enfoque": "RAG estructurado",
        "JSON válido": True,
        "Esquema correcto": True,
        "Grounding": True,
        "Alucinaciones": False,
        "Resultado": "Correcto"
    },
    {
        "Enfoque": "Fine-tuning LoRA",
        "JSON válido": True,
        "Esquema correcto": False,
        "Grounding": False,
        "Alucinaciones": True,
        "Resultado": "Dataset insuficiente"
    },
    {
        "Enfoque": "Híbrido RAG + LoRA",
        "JSON válido": True,
        "Esquema correcto": True,
        "Grounding": True,
        "Alucinaciones": False,
        "Resultado": "Mejor resultado"
    }
]

import pandas as pd

comparison_df = pd.DataFrame(
    comparison_results
)

print("COMPARACIÓN FINAL DE ENFOQUES\n")
display(comparison_df)

COMPARACIÓN FINAL DE ENFOQUES



,Enfoque,JSON válido,Esquema correcto,Grounding,Alucinaciones,Resultado
0,Modelo base,True,False,False,True,Insuficiente
1,Prompting,False,Parcial,False,True,"Mejora estructura, no factualidad"
2,RAG inicial,False,Parcial,Parcial,True,"Reduce alucinaciones, pero contexto insuficiente"
3,RAG ampliado,False,Parcial,Parcial,True,"Mejora variedad, aún inventa detalles"
4,RAG estructurado,True,True,True,False,Correcto
5,Fine-tuning LoRA,True,False,False,True,Dataset insuficiente
6,Híbrido RAG + LoRA,True,True,True,False,Mejor resultado


Conclusiones del caso práctico

El desarrollo del planificador de viajes permitió comparar diferentes estrategias de personalización de modelos de lenguaje: modelo base, prompting, Retrieval-Augmented Generation (RAG), fine-tuning mediante LoRA y una arquitectura híbrida.

El modelo base fue capaz de generar itinerarios coherentes, pero presentó problemas de alucinación, falta de control sobre las fuentes de información y dificultades para respetar una estructura específica.

La aplicación de técnicas de prompting mejoró la organización de la respuesta y el seguimiento de instrucciones, aunque no eliminó las alucinaciones ni garantizó que la información generada estuviera respaldada por datos confiables.

La incorporación de RAG permitió proporcionar al modelo información contextual sobre lugares turísticos de Bogotá. Sin embargo, cuando la base de conocimiento era limitada, se produjeron itinerarios repetitivos y el modelo continuó generando detalles no presentes en el contexto.

Al ampliar la base de conocimiento se incrementó la diversidad de las recomendaciones, aunque se comprobó que proporcionar más contexto no garantiza por sí solo el cumplimiento estricto de las restricciones.

Por esta razón se implementó una representación estructurada de los lugares y de las actividades autorizadas, junto con validaciones deterministas. Este mecanismo permitió comprobar de forma programática que cada lugar y actividad estuvieran respaldados por la información disponible y redujo significativamente las alucinaciones.

También se realizó un proceso de fine-tuning mediante LoRA sobre el modelo Qwen2.5-1.5B-Instruct. El entrenamiento modificó únicamente aproximadamente el 0,141 % de los parámetros del modelo, demostrando la eficiencia de las técnicas PEFT. Sin embargo, debido al tamaño reducido del dataset de entrenamiento, compuesto por tres ejemplos, no se observó una mejora funcional suficiente en el cumplimiento del esquema ni en el grounding factual.

Finalmente, la combinación de RAG estructurado, prompting, fine-tuning y validación determinista produjo el resultado más robusto. La arquitectura híbrida generó un JSON válido, respetó el esquema definido y utilizó exclusivamente lugares y actividades autorizados.

El experimento demuestra que el fine-tuning no sustituye a RAG cuando la aplicación requiere información factual o controlada. RAG permite aportar conocimiento externo al modelo, mientras que el fine-tuning puede utilizarse para adaptar patrones de comportamiento, formato o estilo. La combinación de ambas técnicas, acompañada de mecanismos de validación, ofrece una solución más segura y controlable para aplicaciones de IA generativa.

In [62]:
import os
import json

project_dir = "/content/travel_ai_project"
adapter_dir = os.path.join(project_dir, "qwen_travel_lora")

os.makedirs(project_dir, exist_ok=True)
os.makedirs(adapter_dir, exist_ok=True)

# 1. Guardar adaptador LoRA
model_lora.save_pretrained(adapter_dir)

# 2. Guardar tokenizer
tokenizer.save_pretrained(adapter_dir)

# 3. Guardar base estructurada autorizada
authorized_places_path = os.path.join(
    project_dir,
    "authorized_places.json"
)

with open(
    authorized_places_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        authorized_places,
        f,
        ensure_ascii=False,
        indent=2
    )

# 4. Guardar configuración básica de la aplicación
app_config = {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "adapter": "qwen_travel_lora",
    "embedding_model": "paraphrase-multilingual-MiniLM-L12-v2",
    "architecture": [
        "Prompting",
        "RAG",
        "LoRA",
        "Structured Output",
        "Deterministic Validation"
    ]
}

config_path = os.path.join(
    project_dir,
    "app_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        app_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ Proyecto guardado correctamente.")
print("\nDirectorio principal:", project_dir)
print("Adaptador LoRA:", adapter_dir)
print("Base autorizada:", authorized_places_path)
print("Configuración:", config_path)

print("\nArchivos creados:")
for root, dirs, files in os.walk(project_dir):
    level = root.replace(project_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")

✅ Proyecto guardado correctamente.

Directorio principal: /content/travel_ai_project
Adaptador LoRA: /content/travel_ai_project/qwen_travel_lora
Base autorizada: /content/travel_ai_project/authorized_places.json
Configuración: /content/travel_ai_project/app_config.json

Archivos creados:
travel_ai_project/
  authorized_places.json
  app_config.json
  qwen_travel_lora/
    tokenizer.json
    README.md
    tokenizer_config.json
    chat_template.jinja
    adapter_config.json
    adapter_model.safetensors


In [63]:
backend_code = r'''
import json
import torch

from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./qwen_travel_lora"
AUTHORIZED_PLACES_PATH = "./authorized_places.json"


app = FastAPI(
    title="Travel AI API",
    description="API para generar itinerarios personalizados con Qwen + LoRA + RAG estructurado.",
    version="1.0"
)


class TravelRequest(BaseModel):
    destination: str
    days: int
    budget: float
    currency: str
    interests: list[str]
    travel_style: str
    restrictions: list[str]


# Cargar base autorizada
with open(
    AUTHORIZED_PLACES_PATH,
    "r",
    encoding="utf-8"
) as f:
    authorized_places = json.load(f)


# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)


# Cargar modelo base
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)


# Cargar adaptador LoRA
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()


def build_prompt(data: TravelRequest):

    authorized_places_json = json.dumps(
        authorized_places,
        ensure_ascii=False,
        indent=2
    )

    prompt = f"""
Genera un itinerario personalizado.

DESTINO:
{data.destination}

DURACIÓN:
{data.days} días

PRESUPUESTO:
{data.budget} {data.currency}

INTERESES:
{", ".join(data.interests)}

RITMO:
{data.travel_style}

RESTRICCIONES:
{", ".join(data.restrictions)}

BASE AUTORIZADA:
{authorized_places_json}

REGLAS:

1. Usa únicamente lugares existentes en BASE AUTORIZADA.
2. Usa únicamente actividades existentes en allowed_activities.
3. Genera exactamente {data.days} días.
4. Cada día debe incluir manana, tarde y noche.
5. No inventes lugares ni actividades.
6. No agregues precios, horarios o transporte no proporcionados.
7. Devuelve únicamente JSON válido.

FORMATO:

{{
  "dias": [
    {{
      "dia": 1,
      "manana": {{
        "lugar": "...",
        "actividad": "..."
      }},
      "tarde": {{
        "lugar": "...",
        "actividad": "..."
      }},
      "noche": {{
        "lugar": "...",
        "actividad": "..."
      }}
    }}
  ]
}}
"""

    return prompt


@app.get("/")
def root():
    return {
        "status": "ok",
        "service": "Travel AI API"
    }


@app.post("/generate")
def generate_itinerary(data: TravelRequest):

    prompt = build_prompt(data)

    messages = [
        {
            "role": "system",
            "content": (
                "Eres un planificador profesional de viajes. "
                "Debes respetar estrictamente la base autorizada "
                "y devolver únicamente JSON válido."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1400,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return {
        "response": response
    }
'''

backend_path = "/content/travel_ai_project/app.py"

with open(
    backend_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(backend_code)

print("✅ Backend creado correctamente.")
print("Archivo:", backend_path)

print("\nPrimeras líneas del archivo:\n")

with open(
    backend_path,
    "r",
    encoding="utf-8"
) as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 25:
            break

✅ Backend creado correctamente.
Archivo: /content/travel_ai_project/app.py

Primeras líneas del archivo:


import json
import torch

from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./qwen_travel_lora"
AUTHORIZED_PLACES_PATH = "./authorized_places.json"


app = FastAPI(
    title="Travel AI API",
    description="API para generar itinerarios personalizados con Qwen + LoRA + RAG estructurado.",
    version="1.0"
)


class TravelRequest(BaseModel):
    destination: str
    days: int
    budget: float


In [64]:
requirements = """fastapi
uvicorn
torch
transformers
peft
accelerate
sentence-transformers
faiss-cpu
pydantic
"""

requirements_path = "/content/travel_ai_project/requirements.txt"

with open(
    requirements_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements)

print("✅ requirements.txt creado correctamente.")
print("Archivo:", requirements_path)

print("\nContenido:\n")

with open(
    requirements_path,
    "r",
    encoding="utf-8"
) as f:
    print(f.read())

✅ requirements.txt creado correctamente.
Archivo: /content/travel_ai_project/requirements.txt

Contenido:

fastapi
uvicorn
torch
transformers
peft
accelerate
sentence-transformers
faiss-cpu
pydantic



In [65]:
import os

frontend_dir = "/content/travel_ai_project/frontend"

os.makedirs(frontend_dir, exist_ok=True)

print("✅ Carpeta frontend creada.")
print("Ruta:", frontend_dir)

✅ Carpeta frontend creada.
Ruta: /content/travel_ai_project/frontend


In [68]:
index_html = r'''
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Travel AI Planner</title>

    <link rel="stylesheet" href="styles.css">
</head>

<body>

    <main class="container">

        <section class="hero">
            <span class="badge">IA Generativa + RAG + LoRA</span>

            <h1>Travel AI Planner</h1>

            <p>
                Genera itinerarios personalizados utilizando
                inteligencia artificial, recuperación de información
                y validación estructurada.
            </p>
        </section>

        <section class="planner-card">

            <h2>Configura tu viaje</h2>

            <form id="travelForm">

                <div class="form-grid">

                    <div class="form-group">
                        <label for="destination">
                            Destino
                        </label>

                        <input
                            type="text"
                            id="destination"
                            value="Bogotá, Colombia"
                            required
                        >
                    </div>

                    <div class="form-group">
                        <label for="days">
                            Duración (días)
                        </label>

                        <input
                            type="number"
                            id="days"
                            min="1"
                            max="10"
                            value="4"
                            required
                        >
                    </div>

                    <div class="form-group">
                        <label for="budget">
                            Presupuesto
                        </label>

                        <input
                            type="number"
                            id="budget"
                            min="1"
                            value="700"
                            required
                        >
                    </div>

                    <div class="form-group">
                        <label for="currency">
                            Moneda
                        </label>

                        <select id="currency">
                            <option value="USD" selected>
                                USD
                            </option>

                            <option value="COP">
                                COP
                            </option>

                            <option value="EUR">
                                EUR
                            </option>
                        </select>
                    </div>

                    <div class="form-group">
                        <label for="travelStyle">
                            Ritmo del viaje
                        </label>

                        <select id="travelStyle">
                            <option value="relajado">
                                Relajado
                            </option>

                            <option value="moderado" selected>
                                Moderado
                            </option>

                            <option value="intenso">
                                Intenso
                            </option>
                        </select>
                    </div>

                </div>

                <div class="form-group full-width">

                    <label>
                        Intereses
                    </label>

                    <div class="checkbox-group">

                        <label>
                            <input
                                type="checkbox"
                                name="interests"
                                value="cultura"
                                checked
                            >
                            Cultura
                        </label>

                        <label>
                            <input
                                type="checkbox"
                                name="interests"
                                value="gastronomía"
                                checked
                            >
                            Gastronomía
                        </label>

                        <label>
                            <input
                                type="checkbox"
                                name="interests"
                                value="historia"
                            >
                            Historia
                        </label>

                        <label>
                            <input
                                type="checkbox"
                                name="interests"
                                value="arte"
                            >
                            Arte
                        </label>

                        <label>
                            <input
                                type="checkbox"
                                name="interests"
                                value="naturaleza"
                            >
                            Naturaleza
                        </label>

                    </div>

                </div>

                <div class="form-group full-width">

                    <label for="restrictions">
                        Restricciones o preferencias
                    </label>

                    <textarea
                        id="restrictions"
                        rows="3"
                        placeholder="Ejemplo: sin actividades extremas"
                    >sin actividades extremas</textarea>

                </div>

                <button
                    type="submit"
                    id="generateButton"
                >
                    Generar itinerario
                </button>

            </form>

        </section>

        <section
            id="statusSection"
            class="status hidden"
        >
            <p id="statusMessage">
                Generando itinerario...
            </p>
        </section>

        <section
            id="resultSection"
            class="results hidden"
        >

            <div class="results-header">

                <div>
                    <span class="badge">
                        Itinerario personalizado
                    </span>

                    <h2>
                        Tu viaje
                    </h2>
                </div>

                <button
                    id="newSearchButton"
                    class="secondary-button"
                >
                    Nueva búsqueda
                </button>

            </div>

            <div id="itineraryContainer">
            </div>

        </section>

        <footer>

            <p>
                Proyecto académico de IA Generativa ·
                Qwen2.5 + RAG + LoRA + Validación determinista
            </p>

        </footer>

    </main>

    <script src="script.js"></script>

</body>
</html>
'''

index_path = "/content/travel_ai_project/frontend/index.html"

with open(
    index_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(index_html)

print("✅ index.html creado correctamente.")
print("Archivo:", index_path)
print("\nTamaño:", len(index_html), "caracteres")

✅ index.html creado correctamente.
Archivo: /content/travel_ai_project/frontend/index.html

Tamaño: 7211 caracteres


In [69]:
styles_css = r'''
:root {
    --bg: #f5f7fb;
    --surface: #ffffff;
    --surface-soft: #f8fafc;
    --text: #172033;
    --muted: #667085;
    --primary: #2563eb;
    --primary-dark: #1d4ed8;
    --border: #e4e7ec;
    --success: #15803d;
    --shadow: 0 18px 50px rgba(15, 23, 42, 0.08);
    --radius-lg: 24px;
    --radius-md: 14px;
}

* {
    box-sizing: border-box;
}

html {
    scroll-behavior: smooth;
}

body {
    margin: 0;
    font-family:
        Inter,
        ui-sans-serif,
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif;
    background:
        radial-gradient(
            circle at top right,
            rgba(37, 99, 235, 0.10),
            transparent 30%
        ),
        var(--bg);
    color: var(--text);
    line-height: 1.6;
}

.container {
    width: min(1120px, calc(100% - 32px));
    margin: 0 auto;
    padding: 56px 0 32px;
}

.hero {
    max-width: 760px;
    margin: 0 auto 36px;
    text-align: center;
}

.hero h1 {
    margin: 16px 0 12px;
    font-size: clamp(2.5rem, 6vw, 4.7rem);
    line-height: 1;
    letter-spacing: -0.045em;
}

.hero p {
    max-width: 680px;
    margin: 0 auto;
    font-size: 1.08rem;
    color: var(--muted);
}

.badge {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 7px 12px;
    border: 1px solid #bfdbfe;
    border-radius: 999px;
    background: #eff6ff;
    color: #1d4ed8;
    font-size: 0.82rem;
    font-weight: 700;
    letter-spacing: 0.02em;
}

.planner-card,
.results {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius-lg);
    box-shadow: var(--shadow);
}

.planner-card {
    padding: 32px;
}

.planner-card h2,
.results h2 {
    margin-top: 0;
    letter-spacing: -0.02em;
}

.form-grid {
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 20px;
}

.form-group {
    display: flex;
    flex-direction: column;
    gap: 8px;
}

.full-width {
    margin-top: 22px;
}

label {
    font-weight: 650;
    color: #344054;
}

input,
select,
textarea {
    width: 100%;
    border: 1px solid #d0d5dd;
    border-radius: var(--radius-md);
    background: #fff;
    padding: 13px 14px;
    font: inherit;
    color: var(--text);
    outline: none;
    transition:
        border-color 0.2s ease,
        box-shadow 0.2s ease;
}

input:focus,
select:focus,
textarea:focus {
    border-color: var(--primary);
    box-shadow: 0 0 0 4px rgba(37, 99, 235, 0.12);
}

textarea {
    resize: vertical;
}

.checkbox-group {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
}

.checkbox-group label {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 9px 12px;
    border: 1px solid var(--border);
    border-radius: 999px;
    background: var(--surface-soft);
    cursor: pointer;
    font-weight: 500;
}

.checkbox-group input {
    width: auto;
    margin: 0;
}

button {
    border: 0;
    border-radius: var(--radius-md);
    font: inherit;
    font-weight: 700;
    cursor: pointer;
    transition:
        transform 0.15s ease,
        background 0.15s ease,
        opacity 0.15s ease;
}

button:active {
    transform: scale(0.98);
}

#generateButton {
    width: 100%;
    margin-top: 28px;
    padding: 15px 18px;
    background: var(--primary);
    color: #fff;
}

#generateButton:hover {
    background: var(--primary-dark);
}

#generateButton:disabled {
    opacity: 0.6;
    cursor: not-allowed;
}

.status {
    margin: 22px 0;
    padding: 16px 20px;
    border: 1px solid #bfdbfe;
    border-radius: var(--radius-md);
    background: #eff6ff;
    color: #1e40af;
    text-align: center;
}

.results {
    margin-top: 28px;
    padding: 32px;
}

.results-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    gap: 20px;
    margin-bottom: 24px;
}

.secondary-button {
    padding: 10px 14px;
    border: 1px solid var(--border);
    background: #fff;
    color: var(--text);
}

.secondary-button:hover {
    background: var(--surface-soft);
}

.day-card {
    margin-top: 20px;
    padding: 24px;
    border: 1px solid var(--border);
    border-radius: 18px;
    background: var(--surface-soft);
}

.day-card h3 {
    margin: 0 0 18px;
    font-size: 1.25rem;
}

.period-grid {
    display: grid;
    grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 14px;
}

.period-card {
    padding: 18px;
    border-radius: 16px;
    background: #fff;
    border: 1px solid var(--border);
}

.period-title {
    margin-bottom: 10px;
    color: var(--primary);
    font-size: 0.82rem;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 0.08em;
}

.place {
    margin: 0 0 6px;
    font-weight: 750;
    color: #101828;
}

.activity {
    margin: 0;
    color: var(--muted);
}

.error-box {
    padding: 18px;
    border: 1px solid #fecaca;
    border-radius: var(--radius-md);
    background: #fef2f2;
    color: #991b1b;
}

.hidden {
    display: none !important;
}

footer {
    padding: 34px 0 10px;
    text-align: center;
    color: var(--muted);
    font-size: 0.9rem;
}

@media (max-width: 780px) {

    .container {
        width: min(100% - 20px, 1120px);
        padding-top: 32px;
    }

    .planner-card,
    .results {
        padding: 22px;
    }

    .form-grid,
    .period-grid {
        grid-template-columns: 1fr;
    }

    .results-header {
        flex-direction: column;
    }

    .secondary-button {
        width: 100%;
    }
}
'''

styles_path = "/content/travel_ai_project/frontend/styles.css"

with open(
    styles_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(styles_css)

print("✅ styles.css creado correctamente.")
print("Archivo:", styles_path)
print("Tamaño:", len(styles_css), "caracteres")

✅ styles.css creado correctamente.
Archivo: /content/travel_ai_project/frontend/styles.css
Tamaño: 5491 caracteres


In [70]:
script_js = r'''
const API_URL = "http://localhost:8000/generate";

const form = document.getElementById("travelForm");
const statusSection = document.getElementById("statusSection");
const statusMessage = document.getElementById("statusMessage");
const resultSection = document.getElementById("resultSection");
const itineraryContainer = document.getElementById("itineraryContainer");
const generateButton = document.getElementById("generateButton");
const newSearchButton = document.getElementById("newSearchButton");


function getSelectedInterests() {

    return Array.from(
        document.querySelectorAll(
            'input[name="interests"]:checked'
        )
    ).map(
        input => input.value
    );
}


function getRestrictions() {

    const value = document
        .getElementById("restrictions")
        .value
        .trim();

    if (!value) {
        return [];
    }

    return value
        .split(",")
        .map(item => item.trim())
        .filter(Boolean);
}


function showStatus(message) {

    statusMessage.textContent = message;

    statusSection.classList.remove("hidden");
    resultSection.classList.add("hidden");
}


function hideStatus() {

    statusSection.classList.add("hidden");
}


function showError(message) {

    hideStatus();

    itineraryContainer.innerHTML = `
        <div class="error-box">
            <strong>No fue posible generar el itinerario.</strong>
            <p>${message}</p>
        </div>
    `;

    resultSection.classList.remove("hidden");
}


function createPeriodCard(title, data) {

    return `
        <div class="period-card">

            <div class="period-title">
                ${title}
            </div>

            <p class="place">
                ${data.lugar}
            </p>

            <p class="activity">
                ${data.actividad}
            </p>

        </div>
    `;
}


function renderItinerary(data) {

    if (!data || !Array.isArray(data.dias)) {
        throw new Error(
            "La respuesta recibida no tiene el formato esperado."
        );
    }

    itineraryContainer.innerHTML = "";

    data.dias.forEach(day => {

        const dayCard = document.createElement("article");

        dayCard.className = "day-card";

        dayCard.innerHTML = `
            <h3>Día ${day.dia}</h3>

            <div class="period-grid">

                ${createPeriodCard(
                    "Mañana",
                    day.manana
                )}

                ${createPeriodCard(
                    "Tarde",
                    day.tarde
                )}

                ${createPeriodCard(
                    "Noche",
                    day.noche
                )}

            </div>
        `;

        itineraryContainer.appendChild(dayCard);
    });

    hideStatus();
    resultSection.classList.remove("hidden");

    resultSection.scrollIntoView({
        behavior: "smooth",
        block: "start"
    });
}


function cleanModelResponse(response) {

    if (typeof response !== "string") {
        return response;
    }

    let cleaned = response.trim();

    cleaned = cleaned.replace(
        /^```json\s*/i,
        ""
    );

    cleaned = cleaned.replace(
        /\s*```$/,
        ""
    );

    return JSON.parse(cleaned);
}


form.addEventListener(
    "submit",
    async event => {

        event.preventDefault();

        const interests = getSelectedInterests();

        if (interests.length === 0) {

            showError(
                "Selecciona al menos un interés."
            );

            return;
        }

        const payload = {
            destination:
                document.getElementById(
                    "destination"
                ).value.trim(),

            days:
                Number(
                    document.getElementById(
                        "days"
                    ).value
                ),

            budget:
                Number(
                    document.getElementById(
                        "budget"
                    ).value
                ),

            currency:
                document.getElementById(
                    "currency"
                ).value,

            interests:
                interests,

            travel_style:
                document.getElementById(
                    "travelStyle"
                ).value,

            restrictions:
                getRestrictions()
        };


        generateButton.disabled = true;
        generateButton.textContent =
            "Generando...";

        showStatus(
            "La IA está construyendo tu itinerario..."
        );


        try {

            const response = await fetch(
                API_URL,
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body:
                        JSON.stringify(
                            payload
                        )
                }
            );


            if (!response.ok) {

                const errorText =
                    await response.text();

                throw new Error(
                    `Error ${response.status}: ${errorText}`
                );
            }


            const apiData =
                await response.json();


            const itinerary =
                cleanModelResponse(
                    apiData.response
                );


            renderItinerary(
                itinerary
            );

        }

        catch (error) {

            console.error(error);

            showError(
                error.message
            );

        }

        finally {

            generateButton.disabled = false;

            generateButton.textContent =
                "Generar itinerario";
        }
    }
);


newSearchButton.addEventListener(
    "click",
    () => {

        resultSection.classList.add(
            "hidden"
        );

        window.scrollTo({
            top: 0,
            behavior: "smooth"
        });
    }
);
'''

script_path = "/content/travel_ai_project/frontend/script.js"

with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(script_js)

print("✅ script.js creado correctamente.")
print("Archivo:", script_path)
print("Tamaño:", len(script_js), "caracteres")

✅ script.js creado correctamente.
Archivo: /content/travel_ai_project/frontend/script.js
Tamaño: 6119 caracteres


In [71]:
import os

project_dir = "/content/travel_ai_project"

print("ESTRUCTURA DEL PROYECTO\n")

for root, dirs, files in os.walk(project_dir):
    level = root.replace(project_dir, "").count(os.sep)
    indent = "    " * level

    folder_name = os.path.basename(root)

    if root == project_dir:
        print("travel_ai_project/")
    else:
        print(f"{indent}{folder_name}/")

    for file in sorted(files):
        print(f"{indent}    {file}")

ESTRUCTURA DEL PROYECTO

travel_ai_project/
    app.py
    app_config.json
    authorized_places.json
    requirements.txt
    qwen_travel_lora/
        README.md
        adapter_config.json
        adapter_model.safetensors
        chat_template.jinja
        tokenizer.json
        tokenizer_config.json
    frontend/
        index.html
        script.js
        styles.css


In [72]:
import os
import sys

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

if project_dir not in sys.path:
    sys.path.append(project_dir)

print("Directorio actual:", os.getcwd())

try:
    import app

    print("✅ app.py importado correctamente.")
    print("Título API:", app.app.title)
    print("Versión:", app.app.version)

except Exception as e:
    print("❌ Error al cargar app.py")
    print(type(e).__name__ + ":", e)

Directorio actual: /content/travel_ai_project


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ app.py importado correctamente.
Título API: Travel AI API
Versión: 1.0


In [73]:
import subprocess
import time
import os

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Iniciando servidor FastAPI...")

time.sleep(5)

if server_process.poll() is None:
    print("✅ Servidor FastAPI ejecutándose.")
    print("Puerto: 8000")
else:
    print("❌ El servidor terminó inesperadamente.")

    output = server_process.stdout.read()
    print(output)

⏳ Iniciando servidor FastAPI...
✅ Servidor FastAPI ejecutándose.
Puerto: 8000


In [75]:
import time

print("DIAGNÓSTICO DEL SERVIDOR FASTAPI\n")

status = server_process.poll()

if status is None:
    print("✅ El proceso sigue activo.")
    print("PID:", server_process.pid)

else:
    print("❌ El proceso terminó.")
    print("Código de salida:", status)

    print("\nLOG DEL SERVIDOR:\n")

    try:
        log_output = server_process.stdout.read()
        print(log_output)
    except Exception as e:
        print("No se pudo leer el log:", e)

DIAGNÓSTICO DEL SERVIDOR FASTAPI

✅ El proceso sigue activo.
PID: 8938


In [76]:
import requests
import time

url = "http://127.0.0.1:8000/"

print("Verificando disponibilidad de FastAPI...\n")

server_ready = False

for attempt in range(1, 11):

    try:
        response = requests.get(
            url,
            timeout=5
        )

        print(
            f"Intento {attempt}:",
            response.status_code
        )

        if response.ok:
            print("\n✅ FastAPI ya está disponible.")
            print("Respuesta:", response.json())

            server_ready = True
            break

    except requests.exceptions.RequestException as e:
        print(
            f"Intento {attempt}: todavía no disponible"
        )

    time.sleep(3)


if not server_ready:
    print(
        "\n❌ FastAPI no respondió después de los intentos."
    )

Verificando disponibilidad de FastAPI...

Intento 1: 200

✅ FastAPI ya está disponible.
Respuesta: {'status': 'ok', 'service': 'Travel AI API'}


In [77]:
import requests
import json

test_payload = {
    "destination": "Bogotá, Colombia",
    "days": 4,
    "budget": 700,
    "currency": "USD",
    "interests": [
        "cultura",
        "gastronomía"
    ],
    "travel_style": "moderado",
    "restrictions": [
        "sin actividades extremas"
    ]
}

print("Enviando solicitud al backend...\n")

response = requests.post(
    "http://127.0.0.1:8000/generate",
    json=test_payload,
    timeout=180
)

print("Status code:", response.status_code)

if response.ok:

    api_response = response.json()

    print("\n✅ Backend respondió correctamente.")

    print("\nRESPUESTA COMPLETA:\n")

    print(
        json.dumps(
            api_response,
            ensure_ascii=False,
            indent=2
        )
    )

else:

    print("\n❌ Error del backend:")
    print(response.text)

Enviando solicitud al backend...

Status code: 200

✅ Backend respondió correctamente.

RESPUESTA COMPLETA:

{
  "response": "{\n  \"dias\": [\n    {\n      \"dia\": 1,\n      \"manana\": {\n        \"lugar\": \"Usaquén\",\n        \"actividad\": \"Caminar por el barrio y visitar algunos restaurantes locales.\"\n      },\n      \"tarde\": {\n        \"lugar\": \"Mercado de La Perseverancia\",\n        \"actividad\": \"Conocer la comida colombiana y explorar la gastronomía local.\"\n      },\n      \"noche\": {\n        \"lugar\": \"Parque de la 93\",\n        \"actividad\": \"Caminar y visitar los cafés del parque para disfrutar de una cena tranquila.\"\n      }\n    },\n    {\n      \"dia\": 2,\n      \"manana\": {\n        \"lugar\": \"La Candelaria\",\n        \"actividad\": \"Recorrer la arquitectura colonial y visitar algunas plazas.\"\n      },\n      \"tarde\": {\n        \"lugar\": \"Chorro de Quevedo\",\n        \"actividad\": \"Conocer la historia y observar la arquitectura.\

In [78]:
import json
import re

def validate_api_itinerary(api_response, authorized_places):

    errors = []

    # La API devuelve el JSON generado dentro de "response"
    raw_response = api_response.get("response", "")

    # Limpiar posibles bloques markdown
    cleaned = raw_response.strip()

    cleaned = re.sub(
        r"^```json\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    # 1. Parsear JSON
    try:
        itinerary = json.loads(cleaned)

    except json.JSONDecodeError as e:
        return None, [
            f"JSON inválido: {e}"
        ]

    # 2. Validar clave principal
    if "dias" not in itinerary:
        errors.append(
            "Falta la clave principal 'dias'."
        )

        return itinerary, errors

    days = itinerary["dias"]

    # 3. Validar cantidad de días
    expected_days = test_payload["days"]

    if len(days) != expected_days:
        errors.append(
            f"Se esperaban {expected_days} días "
            f"y se encontraron {len(days)}."
        )

    periods = [
        "manana",
        "tarde",
        "noche"
    ]

    place_counter = {}

    # 4. Validar cada bloque
    for expected_day, day in enumerate(
        days,
        start=1
    ):

        if day.get("dia") != expected_day:
            errors.append(
                f"Día incorrecto: se esperaba "
                f"{expected_day}."
            )

        for period in periods:

            if period not in day:
                errors.append(
                    f"Día {expected_day}: "
                    f"falta '{period}'."
                )
                continue

            block = day[period]

            place = block.get("lugar")
            activity = block.get("actividad")

            # Lugar autorizado
            if place not in authorized_places:
                errors.append(
                    f"Día {expected_day} - {period}: "
                    f"lugar no autorizado '{place}'."
                )
                continue

            # Contar repeticiones
            place_counter[place] = (
                place_counter.get(place, 0) + 1
            )

            # Actividad EXACTAMENTE autorizada
            allowed_activities = authorized_places[
                place
            ]["allowed_activities"]

            if activity not in allowed_activities:
                errors.append(
                    f"Día {expected_day} - {period}: "
                    f"actividad no autorizada para "
                    f"'{place}': '{activity}'"
                )

    # 5. Control de repetición
    for place, count in place_counter.items():

        if count > 2:
            errors.append(
                f"El lugar '{place}' aparece "
                f"{count} veces; máximo permitido: 2."
            )

    return itinerary, errors


api_itinerary, api_validation_errors = (
    validate_api_itinerary(
        api_response,
        authorized_places
    )
)

print("VALIDACIÓN DE LA RESPUESTA REAL DE LA API\n")

if api_validation_errors:

    print("❌ La API respondió, pero el itinerario "
          "NO pasó el control de calidad:\n")

    for error in api_validation_errors:
        print("-", error)

else:

    print(
        "✅ El itinerario de la API cumple "
        "todas las validaciones."
    )

VALIDACIÓN DE LA RESPUESTA REAL DE LA API

❌ La API respondió, pero el itinerario NO pasó el control de calidad:

- Día 1 - manana: actividad no autorizada para 'Usaquén': 'Caminar por el barrio y visitar algunos restaurantes locales.'
- Día 1 - tarde: actividad no autorizada para 'Mercado de La Perseverancia': 'Conocer la comida colombiana y explorar la gastronomía local.'
- Día 1 - noche: actividad no autorizada para 'Parque de la 93': 'Caminar y visitar los cafés del parque para disfrutar de una cena tranquila.'
- Día 2 - manana: actividad no autorizada para 'La Candelaria': 'Recorrer la arquitectura colonial y visitar algunas plazas.'
- Día 2 - tarde: actividad no autorizada para 'Chorro de Quevedo': 'Conocer la historia y observar la arquitectura.'
- Día 2 - noche: actividad no autorizada para 'Museo Nacional de Colombia': 'Conocer la historia y el patrimonio del país.'
- Día 3 - manana: actividad no autorizada para 'Biblioteca Luis Ángel Arango': 'Realizar actividades de lectura 

In [79]:
backend_code_v2 = r'''
import json
import re
import torch

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./qwen_travel_lora"
AUTHORIZED_PLACES_PATH = "./authorized_places.json"


app = FastAPI(
    title="Travel AI API",
    description=(
        "API para generar itinerarios personalizados "
        "con Qwen + LoRA + RAG estructurado + validación determinista."
    ),
    version="1.1"
)


class TravelRequest(BaseModel):
    destination: str
    days: int
    budget: float
    currency: str
    interests: list[str]
    travel_style: str
    restrictions: list[str]


# Cargar base autorizada
with open(
    AUTHORIZED_PLACES_PATH,
    "r",
    encoding="utf-8"
) as f:
    authorized_places = json.load(f)


# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)


# Modelo base
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)


# Adaptador LoRA
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()


def clean_json_response(response: str):

    cleaned = response.strip()

    cleaned = re.sub(
        r"^```json\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    return cleaned.strip()


def validate_itinerary(
    itinerary: dict,
    expected_days: int
):

    errors = []

    if "dias" not in itinerary:
        return [
            "Falta la clave principal 'dias'."
        ]

    days = itinerary["dias"]

    if len(days) != expected_days:
        errors.append(
            f"Se esperaban {expected_days} días "
            f"y se encontraron {len(days)}."
        )

    periods = [
        "manana",
        "tarde",
        "noche"
    ]

    place_counter = {}

    for expected_day, day in enumerate(
        days,
        start=1
    ):

        if day.get("dia") != expected_day:
            errors.append(
                f"Número de día incorrecto. "
                f"Se esperaba {expected_day}."
            )

        for period in periods:

            if period not in day:
                errors.append(
                    f"Día {expected_day}: "
                    f"falta '{period}'."
                )
                continue

            block = day[period]

            place = block.get("lugar")
            activity = block.get("actividad")

            if place not in authorized_places:
                errors.append(
                    f"Día {expected_day} - {period}: "
                    f"lugar no autorizado '{place}'."
                )
                continue

            place_counter[place] = (
                place_counter.get(place, 0) + 1
            )

            allowed_activities = authorized_places[
                place
            ]["allowed_activities"]

            if activity not in allowed_activities:
                errors.append(
                    f"Día {expected_day} - {period}: "
                    f"actividad no autorizada para "
                    f"'{place}': '{activity}'."
                )

    for place, count in place_counter.items():

        if count > 2:
            errors.append(
                f"El lugar '{place}' aparece "
                f"{count} veces; máximo permitido: 2."
            )

    return errors


def build_prompt(data: TravelRequest):

    authorized_places_json = json.dumps(
        authorized_places,
        ensure_ascii=False,
        indent=2
    )

    return f"""
Genera un itinerario personalizado para {data.destination}.

BASE AUTORIZADA:
{authorized_places_json}

PREFERENCIAS:
- Duración: {data.days} días
- Presupuesto: {data.budget} {data.currency}
- Intereses: {", ".join(data.interests)}
- Ritmo: {data.travel_style}
- Restricciones: {", ".join(data.restrictions)}

REGLAS OBLIGATORIAS:

1. Genera exactamente {data.days} días.
2. Cada día debe tener:
   - manana
   - tarde
   - noche
3. Cada bloque debe contener:
   - lugar
   - actividad
4. "lugar" debe coincidir EXACTAMENTE con una clave de BASE AUTORIZADA.
5. "actividad" debe coincidir EXACTAMENTE con uno de los valores de
   allowed_activities correspondientes a ese lugar.
6. No combines varias actividades en una sola frase.
7. No parafrasees actividades.
8. No inventes lugares.
9. No inventes actividades.
10. No añadas precios.
11. No añadas horarios.
12. No añadas transporte.
13. No repitas un lugar más de dos veces.
14. Devuelve únicamente JSON válido.

FORMATO EXACTO:

{{
  "dias": [
    {{
      "dia": 1,
      "manana": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta autorizada"
      }},
      "tarde": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta autorizada"
      }},
      "noche": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta autorizada"
      }}
    }}
  ]
}}
"""


@app.get("/")
def root():

    return {
        "status": "ok",
        "service": "Travel AI API",
        "version": "1.1"
    }


@app.post("/generate")
def generate_itinerary(
    data: TravelRequest
):

    prompt = build_prompt(data)

    messages = [
        {
            "role": "system",
            "content": (
                "Eres un planificador profesional de viajes. "
                "Debes seleccionar exclusivamente valores exactos "
                "de la base autorizada y devolver únicamente JSON válido."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1400,
            do_sample=False
        )

    raw_response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    cleaned_response = clean_json_response(
        raw_response
    )

    try:
        itinerary = json.loads(
            cleaned_response
        )

    except json.JSONDecodeError as e:

        raise HTTPException(
            status_code=422,
            detail={
                "message": "El modelo no generó JSON válido.",
                "error": str(e),
                "raw_response": raw_response
            }
        )

    validation_errors = validate_itinerary(
        itinerary,
        data.days
    )

    if validation_errors:

        raise HTTPException(
            status_code=422,
            detail={
                "message": (
                    "El itinerario generado no superó "
                    "la validación determinista."
                ),
                "validation_errors": validation_errors,
                "raw_response": raw_response
            }
        )

    return {
        "status": "validated",
        "itinerary": itinerary
    }
'''

app_path = "/content/travel_ai_project/app.py"

with open(
    app_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(backend_code_v2)

print("✅ app.py actualizado con validación integrada.")
print("Archivo:", app_path)
print("Versión API: 1.1")

✅ app.py actualizado con validación integrada.
Archivo: /content/travel_ai_project/app.py
Versión API: 1.1


In [80]:
import subprocess
import time
import os

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

# Detener servidor anterior si sigue activo
try:
    if server_process.poll() is None:
        print("Deteniendo servidor anterior...")
        server_process.terminate()

        try:
            server_process.wait(timeout=10)
            print("✅ Servidor anterior detenido.")
        except subprocess.TimeoutExpired:
            server_process.kill()
            print("⚠️ Servidor anterior forzado a cerrar.")

except NameError:
    print("No se encontró un proceso anterior en memoria.")


# Iniciar nueva versión
server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Iniciando FastAPI versión 1.1...")

time.sleep(5)

if server_process.poll() is None:
    print("✅ Proceso Uvicorn iniciado.")
    print("PID:", server_process.pid)
else:
    print("❌ El servidor terminó inesperadamente.")
    print(server_process.stdout.read())

Deteniendo servidor anterior...
✅ Servidor anterior detenido.

⏳ Iniciando FastAPI versión 1.1...
✅ Proceso Uvicorn iniciado.
PID: 11135


In [81]:
import requests
import time

url = "http://127.0.0.1:8000/"

print("Verificando FastAPI 1.1...\n")

server_ready = False

for attempt in range(1, 11):

    try:
        response = requests.get(
            url,
            timeout=5
        )

        print(
            f"Intento {attempt}:",
            response.status_code
        )

        if response.ok:

            data = response.json()

            print("\n✅ FastAPI disponible.")
            print("Respuesta:", data)

            if data.get("version") == "1.1":
                print("\n✅ Versión 1.1 confirmada.")
            else:
                print(
                    "\n⚠️ La API respondió, pero no se confirmó la versión 1.1."
                )

            server_ready = True
            break

    except requests.exceptions.RequestException:

        print(
            f"Intento {attempt}: todavía no disponible"
        )

    time.sleep(3)


if not server_ready:
    print(
        "\n❌ FastAPI no respondió después de los intentos."
    )

Verificando FastAPI 1.1...

Intento 1: 200

✅ FastAPI disponible.
Respuesta: {'status': 'ok', 'service': 'Travel AI API', 'version': '1.1'}

✅ Versión 1.1 confirmada.


In [82]:
import requests
import json

test_payload = {
    "destination": "Bogotá, Colombia",
    "days": 4,
    "budget": 700,
    "currency": "USD",
    "interests": [
        "cultura",
        "gastronomía"
    ],
    "travel_style": "moderado",
    "restrictions": [
        "sin actividades extremas"
    ]
}

print("Probando /generate con validación integrada...\n")

response = requests.post(
    "http://127.0.0.1:8000/generate",
    json=test_payload,
    timeout=180
)

print("Status code:", response.status_code)

try:
    response_data = response.json()
except Exception:
    response_data = {
        "raw_text": response.text
    }

print("\nRESPUESTA:\n")

print(
    json.dumps(
        response_data,
        ensure_ascii=False,
        indent=2
    )
)

if response.status_code == 200:
    print(
        "\n✅ El itinerario fue generado "
        "y superó la validación determinista."
    )

elif response.status_code == 422:
    print(
        "\n⚠️ La API bloqueó correctamente "
        "un itinerario que no cumplía las reglas."
    )

else:
    print(
        "\n❌ Se produjo un error diferente "
        "al esperado."
    )

Probando /generate con validación integrada...

Status code: 200

RESPUESTA:

{
  "status": "validated",
  "itinerary": {
    "dias": [
      {
        "dia": 1,
        "manana": {
          "lugar": "Usaquén",
          "actividad": "caminar"
        },
        "tarde": {
          "lugar": "Mercado de La Perseverancia",
          "actividad": "conocer comida colombiana"
        },
        "noche": {
          "lugar": "La Candelaria",
          "actividad": "recorrer arquitectura colonial"
        }
      },
      {
        "dia": 2,
        "manana": {
          "lugar": "Chorro de Quevedo",
          "actividad": "conocer historia"
        },
        "tarde": {
          "lugar": "Museo Nacional de Colombia",
          "actividad": "conocer historia"
        },
        "noche": {
          "lugar": "Biblioteca Luis Ángel Arango",
          "actividad": "actividades de lectura"
        }
      },
      {
        "dia": 3,
        "manana": {
          "lugar": "Museo Botero",
     

In [83]:
script_path = "/content/travel_ai_project/frontend/script.js"

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    script_content = f.read()


old_code = '''
            const apiData =
                await response.json();


            const itinerary =
                cleanModelResponse(
                    apiData.response
                );


            renderItinerary(
                itinerary
            );
'''

new_code = '''
            const apiData =
                await response.json();


            if (
                apiData.status !== "validated" ||
                !apiData.itinerary
            ) {
                throw new Error(
                    "La API no devolvió un itinerario validado."
                );
            }


            renderItinerary(
                apiData.itinerary
            );
'''


if old_code in script_content:

    script_content = script_content.replace(
        old_code,
        new_code
    )

    with open(
        script_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(script_content)

    print("✅ script.js actualizado para API 1.1.")

else:

    print(
        "⚠️ No se encontró exactamente el bloque "
        "que debía reemplazarse."
    )


print("\nVerificación:")

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:

    updated_script = f.read()

    if "apiData.itinerary" in updated_script:
        print("✅ apiData.itinerary encontrado.")
    else:
        print("❌ apiData.itinerary no encontrado.")

    if "apiData.response" not in updated_script:
        print("✅ Referencia antigua apiData.response eliminada.")
    else:
        print("⚠️ Todavía existe alguna referencia a apiData.response.")

✅ script.js actualizado para API 1.1.

Verificación:
✅ apiData.itinerary encontrado.
✅ Referencia antigua apiData.response eliminada.


In [84]:
import subprocess
import time
import os

frontend_dir = "/content/travel_ai_project/frontend"

os.chdir(frontend_dir)

frontend_process = subprocess.Popen(
    [
        "python",
        "-m",
        "http.server",
        "8080"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Iniciando servidor del frontend...")

time.sleep(3)

if frontend_process.poll() is None:
    print("✅ Frontend ejecutándose correctamente.")
    print("Puerto: 8080")
    print("URL local: http://127.0.0.1:8080")
else:
    print("❌ El servidor del frontend terminó inesperadamente.")
    print(frontend_process.stdout.read())

⏳ Iniciando servidor del frontend...
❌ El servidor del frontend terminó inesperadamente.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/lib/python3.12/http/server.py", line 1329, in <module>
    test(
  File "/usr/lib/python3.12/http/server.py", line 1276, in test
    with ServerClass(addr, HandlerClass) as httpd:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socketserver.py", line 457, in __init__
    self.server_bind()
  File "/usr/lib/python3.12/http/server.py", line 1323, in server_bind
    return super().server_bind()
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/server.py", line 140, in server_bind
    socketserver.TCPServer.server_bind(self)
  File "/usr/lib/python3.12/socketserver.py", line 478, in server_bind
    self.socket.bind(self.server_address)
OSError: [Errno 98] Address already in use



In [85]:
import subprocess
import time
import os

frontend_dir = "/content/travel_ai_project/frontend"

os.chdir(frontend_dir)

frontend_process = subprocess.Popen(
    [
        "python",
        "-m",
        "http.server",
        "8081"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Iniciando servidor del frontend...")

time.sleep(3)

if frontend_process.poll() is None:
    print("✅ Frontend ejecutándose correctamente.")
    print("Puerto: 8081")
    print("URL local: http://127.0.0.1:8081")
else:
    print("❌ El servidor del frontend terminó inesperadamente.")
    print(frontend_process.stdout.read())

⏳ Iniciando servidor del frontend...
✅ Frontend ejecutándose correctamente.
Puerto: 8081
URL local: http://127.0.0.1:8081


In [86]:
import requests

frontend_url = "http://127.0.0.1:8081"

response_frontend = requests.get(
    frontend_url,
    timeout=10
)

print("Status code:", response_frontend.status_code)

if response_frontend.ok:
    print("✅ Frontend accesible correctamente.")
    print("Content-Type:", response_frontend.headers.get("Content-Type"))
    print("Tamaño HTML:", len(response_frontend.text), "caracteres")
else:
    print("❌ Error accediendo al frontend.")
    print(response_frontend.text)

Status code: 200
✅ Frontend accesible correctamente.
Content-Type: text/html
Tamaño HTML: 7223 caracteres


In [87]:
from google.colab import output

frontend_public_url = output.eval_js(
    "google.colab.kernel.proxyPort(8081)"
)

print("✅ URL temporal del frontend:")
print(frontend_public_url)

✅ URL temporal del frontend:
https://8081-gpu-t4-s-kkb-ass1c0-3ra9uxtkm1xmc-c.asia-southeast1-0.prod.colab.dev


In [88]:
app_path = "/content/travel_ai_project/app.py"

with open(
    app_path,
    "r",
    encoding="utf-8"
) as f:
    app_content = f.read()


# 1. Añadir import de CORSMiddleware
old_import = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
"""

new_import = """
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
"""

if "from fastapi.middleware.cors import CORSMiddleware" not in app_content:
    app_content = app_content.replace(
        old_import,
        new_import
    )


# 2. Añadir middleware CORS después de crear app
cors_code = """
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

"""

marker = """
class TravelRequest(BaseModel):
"""

if "app.add_middleware(" not in app_content:
    app_content = app_content.replace(
        marker,
        cors_code + marker
    )


with open(
    app_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(app_content)


print("✅ CORS añadido a FastAPI.")

print(
    "CORSMiddleware importado:",
    "from fastapi.middleware.cors import CORSMiddleware"
    in app_content
)

print(
    "Middleware configurado:",
    "app.add_middleware(" in app_content
)

✅ CORS añadido a FastAPI.
CORSMiddleware importado: True
Middleware configurado: True


In [89]:
import subprocess
import time
import os

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

# Detener servidor anterior
try:
    if server_process.poll() is None:
        print("Deteniendo servidor FastAPI anterior...")

        server_process.terminate()

        try:
            server_process.wait(timeout=10)
            print("✅ Servidor anterior detenido.")

        except subprocess.TimeoutExpired:
            server_process.kill()
            print("⚠️ Servidor anterior forzado a cerrar.")

except NameError:
    print("No se encontró un servidor anterior en memoria.")


# Levantar nuevamente FastAPI
server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Iniciando FastAPI con CORS habilitado...")

time.sleep(5)

if server_process.poll() is None:
    print("✅ Proceso FastAPI iniciado.")
    print("PID:", server_process.pid)
    print("Puerto: 8000")

else:
    print("❌ El servidor terminó inesperadamente.")
    print(server_process.stdout.read())


Deteniendo servidor FastAPI anterior...
✅ Servidor anterior detenido.

⏳ Iniciando FastAPI con CORS habilitado...
✅ Proceso FastAPI iniciado.
PID: 13346
Puerto: 8000


In [90]:
from google.colab import output

backend_public_url = output.eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("✅ URL temporal del backend:")
print(backend_public_url)

✅ URL temporal del backend:
https://8000-gpu-t4-s-kkb-ass1c0-3ra9uxtkm1xmc-c.asia-southeast1-0.prod.colab.dev


In [91]:
script_path = "/content/travel_ai_project/frontend/script.js"

backend_public_url = (
    "https://8000-gpu-t4-s-kkb-ass1c0-3ra9uxtkm1xmc-c."
    "asia-southeast1-0.prod.colab.dev"
)

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    script_content = f.read()


old_api_line = (
    'const API_URL = "http://localhost:8000/generate";'
)

new_api_line = (
    f'const API_URL = "{backend_public_url}/generate";'
)


if old_api_line in script_content:

    script_content = script_content.replace(
        old_api_line,
        new_api_line
    )

    with open(
        script_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(script_content)

    print("✅ URL del backend actualizada correctamente.")

else:
    print(
        "⚠️ No se encontró la URL localhost esperada."
    )


print("\nURL configurada:")

for line in script_content.splitlines():
    if "const API_URL" in line:
        print(line)
        break

✅ URL del backend actualizada correctamente.

URL configurada:
const API_URL = "https://8000-gpu-t4-s-kkb-ass1c0-3ra9uxtkm1xmc-c.asia-southeast1-0.prod.colab.dev/generate";


In [92]:
import os

app_path = "/content/travel_ai_project/app.py"
script_path = "/content/travel_ai_project/frontend/script.js"


# ============================================================
# 1. MODIFICAR script.js
#    El frontend llamará al backend usando la misma URL/origen
# ============================================================

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    script_content = f.read()


lines = script_content.splitlines()

new_lines = []

for line in lines:

    if line.strip().startswith("const API_URL"):
        new_lines.append(
            'const API_URL = "/generate";'
        )
    else:
        new_lines.append(line)


script_content = "\n".join(new_lines)


with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(script_content)


# ============================================================
# 2. MODIFICAR app.py
#    Añadir StaticFiles para servir el frontend
# ============================================================

with open(
    app_path,
    "r",
    encoding="utf-8"
) as f:
    app_content = f.read()


# Importar StaticFiles
if "from fastapi.staticfiles import StaticFiles" not in app_content:

    app_content = app_content.replace(
        "from fastapi import FastAPI, HTTPException",
        """from fastapi import FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles"""
    )


# Montar frontend SOLO si todavía no existe
mount_code = '''

# ============================================================
# FRONTEND
# ============================================================

app.mount(
    "/ui",
    StaticFiles(
        directory="./frontend",
        html=True
    ),
    name="frontend"
)
'''


if 'app.mount(' not in app_content:

    app_content += mount_code


with open(
    app_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(app_content)


# ============================================================
# 3. VERIFICACIÓN
# ============================================================

print("✅ Integración frontend + backend configurada.")

print("\nSCRIPT.JS")

for line in script_content.splitlines():

    if "const API_URL" in line:
        print(line)
        break


print("\nAPP.PY")

print(
    "StaticFiles importado:",
    "from fastapi.staticfiles import StaticFiles"
    in app_content
)

print(
    "Frontend montado en /ui:",
    '"/ui"' in app_content
)

✅ Integración frontend + backend configurada.

SCRIPT.JS
const API_URL = "/generate";

APP.PY
StaticFiles importado: True
Frontend montado en /ui: True


In [93]:
import subprocess
import time
import os

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

# Detener servidor anterior
try:
    if server_process.poll() is None:
        print("Deteniendo servidor FastAPI anterior...")

        server_process.terminate()

        try:
            server_process.wait(timeout=10)
            print("✅ Servidor anterior detenido.")

        except subprocess.TimeoutExpired:
            server_process.kill()
            print("⚠️ Servidor anterior forzado a cerrar.")

except NameError:
    print("No se encontró un servidor anterior en memoria.")


# Iniciar nueva versión
server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Iniciando FastAPI con frontend integrado...")

time.sleep(5)

if server_process.poll() is None:
    print("✅ Proceso FastAPI iniciado.")
    print("PID:", server_process.pid)
    print("Puerto: 8000")

else:
    print("❌ El servidor terminó inesperadamente.")
    print(server_process.stdout.read())

Deteniendo servidor FastAPI anterior...
✅ Servidor anterior detenido.

⏳ Iniciando FastAPI con frontend integrado...
✅ Proceso FastAPI iniciado.
PID: 14455
Puerto: 8000


In [94]:
import requests
import time

base_url = "http://127.0.0.1:8000"

print("Verificando frontend integrado...\n")

server_ready = False

for attempt in range(1, 11):

    try:
        response = requests.get(
            f"{base_url}/ui/",
            timeout=5
        )

        print(
            f"Intento {attempt}:",
            response.status_code
        )

        if response.ok:

            print("\n✅ Frontend integrado disponible.")
            print(
                "Content-Type:",
                response.headers.get("Content-Type")
            )
            print(
                "Tamaño HTML:",
                len(response.text),
                "caracteres"
            )

            server_ready = True
            break

    except requests.exceptions.RequestException:

        print(
            f"Intento {attempt}: todavía no disponible"
        )

    time.sleep(3)


if not server_ready:
    print(
        "\n❌ El frontend integrado no respondió."
    )

Verificando frontend integrado...

Intento 1: todavía no disponible
Intento 2: todavía no disponible
Intento 3: 200

✅ Frontend integrado disponible.
Content-Type: text/html; charset=utf-8
Tamaño HTML: 7211 caracteres


In [95]:
authorized_places_with_costs = {
    "Usaquén": {
        "category": ["gastronomía", "cultura", "paseo"],
        "allowed_activities": {
            "caminar": {
                "min_cost_usd": 0,
                "max_cost_usd": 0
            },
            "visitar restaurantes": {
                "min_cost_usd": 15,
                "max_cost_usd": 30
            },
            "visitar cafés": {
                "min_cost_usd": 5,
                "max_cost_usd": 12
            },
            "observar arquitectura": {
                "min_cost_usd": 0,
                "max_cost_usd": 0
            },
            "realizar actividades culturales": {
                "min_cost_usd": 5,
                "max_cost_usd": 15
            }
        }
    },

    "Zona G": {
        "category": ["gastronomía"],
        "allowed_activities": {
            "experiencia culinaria": {
                "min_cost_usd": 20,
                "max_cost_usd": 40
            },
            "visitar restaurantes": {
                "min_cost_usd": 20,
                "max_cost_usd": 40
            },
            "cenar": {
                "min_cost_usd": 20,
                "max_cost_usd": 40
            }
        }
    },

    "Mercado de La Perseverancia": {
        "category": ["gastronomía"],
        "allowed_activities": {
            "conocer comida colombiana": {
                "min_cost_usd": 8,
                "max_cost_usd": 15
            },
            "explorar gastronomía local": {
                "min_cost_usd": 8,
                "max_cost_usd": 15
            }
        }
    },

    "Parque de la 93": {
        "category": ["gastronomía", "paseo"],
        "allowed_activities": {
            "caminar": {
                "min_cost_usd": 0,
                "max_cost_usd": 0
            },
            "visitar restaurantes": {
                "min_cost_usd": 15,
                "max_cost_usd": 30
            },
            "visitar cafés": {
                "min_cost_usd": 5,
                "max_cost_usd": 12
            },
            "realizar actividades relajadas": {
                "min_cost_usd": 0,
                "max_cost_usd": 10
            }
        }
    },

    "La Candelaria": {
        "category": ["cultura", "historia"],
        "allowed_activities": {
            "recorrer arquitectura colonial": {
                "min_cost_usd": 0,
                "max_cost_usd": 5
            },
            "caminar": {
                "min_cost_usd": 0,
                "max_cost_usd": 0
            },
            "visitar museos": {
                "min_cost_usd": 0,
                "max_cost_usd": 10
            },
            "visitar plazas": {
                "min_cost_usd": 0,
                "max_cost_usd": 0
            },
            "realizar recorridos culturales": {
                "min_cost_usd": 5,
                "max_cost_usd": 15
            }
        }
    },

    "Chorro de Quevedo": {
        "category": ["cultura", "historia"],
        "allowed_activities": {
            "conocer historia": {
                "min_cost_usd": 0,
                "max_cost_usd": 5
            },
            "observar arquitectura": {
                "min_cost_usd": 0,
                "max_cost_usd": 0
            },
            "realizar recorridos culturales": {
                "min_cost_usd": 5,
                "max_cost_usd": 15
            }
        }
    },

    "Museo Nacional de Colombia": {
        "category": ["cultura", "historia", "arte"],
        "allowed_activities": {
            "conocer historia": {
                "min_cost_usd": 0,
                "max_cost_usd": 10
            },
            "conocer patrimonio": {
                "min_cost_usd": 0,
                "max_cost_usd": 10
            },
            "observar arte": {
                "min_cost_usd": 0,
                "max_cost_usd": 10
            },
            "realizar visita cultural": {
                "min_cost_usd": 0,
                "max_cost_usd": 10
            }
        }
    },

    "Biblioteca Luis Ángel Arango": {
        "category": ["cultura"],
        "allowed_activities": {
            "actividades de lectura": {
                "min_cost_usd": 0,
                "max_cost_usd": 5
            },
            "conocer patrimonio": {
                "min_cost_usd": 0,
                "max_cost_usd": 5
            },
            "observar arte": {
                "min_cost_usd": 0,
                "max_cost_usd": 5
            },
            "realizar actividades culturales": {
                "min_cost_usd": 0,
                "max_cost_usd": 10
            }
        }
    },

    "Museo Botero": {
        "category": ["arte", "cultura"],
        "allowed_activities": {
            "observar obras de Fernando Botero": {
                "min_cost_usd": 0,
                "max_cost_usd": 5
            },
            "observar obras de artistas internacionales": {
                "min_cost_usd": 0,
                "max_cost_usd": 5
            },
            "realizar visita cultural": {
                "min_cost_usd": 0,
                "max_cost_usd": 5
            }
        }
    },

    "Monserrate": {
        "category": ["paseo", "turismo"],
        "allowed_activities": {
            "observar vistas de la ciudad": {
                "min_cost_usd": 8,
                "max_cost_usd": 15
            },
            "subir en teleférico": {
                "min_cost_usd": 8,
                "max_cost_usd": 15
            },
            "subir en funicular": {
                "min_cost_usd": 8,
                "max_cost_usd": 15
            }
        }
    }
}

print(
    "✅ Base de costos creada para",
    len(authorized_places_with_costs),
    "lugares."
)

print("\nEjemplo - Zona G:")
print(
    json.dumps(
        authorized_places_with_costs["Zona G"],
        ensure_ascii=False,
        indent=2
    )
)

✅ Base de costos creada para 10 lugares.

Ejemplo - Zona G:
{
  "category": [
    "gastronomía"
  ],
  "allowed_activities": {
    "experiencia culinaria": {
      "min_cost_usd": 20,
      "max_cost_usd": 40
    },
    "visitar restaurantes": {
      "min_cost_usd": 20,
      "max_cost_usd": 40
    },
    "cenar": {
      "min_cost_usd": 20,
      "max_cost_usd": 40
    }
  }
}


In [96]:
index_path = "/content/travel_ai_project/frontend/index.html"

with open(
    index_path,
    "r",
    encoding="utf-8"
) as f:
    index_content = f.read()


target = '''
                    <div class="form-group">
                        <label for="travelStyle">
                            Ritmo del viaje
                        </label>

                        <select id="travelStyle">
                            <option value="relajado">
                                Relajado
                            </option>

                            <option value="moderado" selected>
                                Moderado
                            </option>

                            <option value="intenso">
                                Intenso
                            </option>
                        </select>
                    </div>
'''

replacement = '''
                    <div class="form-group">
                        <label for="travelStyle">
                            Ritmo del viaje
                        </label>

                        <select id="travelStyle">
                            <option value="relajado">
                                Relajado
                            </option>

                            <option value="moderado" selected>
                                Moderado
                            </option>

                            <option value="intenso">
                                Intenso
                            </option>
                        </select>
                    </div>

                    <div class="form-group">
                        <label for="adults">
                            Adultos
                        </label>

                        <input
                            type="number"
                            id="adults"
                            min="1"
                            max="20"
                            value="2"
                            required
                        >
                    </div>

                    <div class="form-group">
                        <label for="children">
                            Niños
                        </label>

                        <input
                            type="number"
                            id="children"
                            min="0"
                            max="20"
                            value="0"
                            required
                        >
                    </div>
'''


if target in index_content:

    index_content = index_content.replace(
        target,
        replacement
    )

    with open(
        index_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(index_content)

    print("✅ Campos de viajeros añadidos correctamente.")

else:
    print("⚠️ No se encontró el bloque esperado en index.html.")


print("\nVerificación:")
print(
    "Campo adultos:",
    'id="adults"' in index_content
)
print(
    "Campo niños:",
    'id="children"' in index_content
)

✅ Campos de viajeros añadidos correctamente.

Verificación:
Campo adultos: True
Campo niños: True


In [97]:
script_path = "/content/travel_ai_project/frontend/script.js"

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    script_content = f.read()


old_block = '''
            currency:
                document.getElementById(
                    "currency"
                ).value,

            interests:
                interests,
'''

new_block = '''
            currency:
                document.getElementById(
                    "currency"
                ).value,

            adults:
                Number(
                    document.getElementById(
                        "adults"
                    ).value
                ),

            children:
                Number(
                    document.getElementById(
                        "children"
                    ).value
                ),

            interests:
                interests,
'''


if old_block in script_content:

    script_content = script_content.replace(
        old_block,
        new_block
    )

    with open(
        script_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(script_content)

    print("✅ Adultos y niños añadidos al payload.")

else:
    print(
        "⚠️ No se encontró el bloque esperado en script.js."
    )


print("\nVerificación:")

print(
    "Campo adults enviado:",
    '"adults"' in script_content or "adults:" in script_content
)

print(
    "Campo children enviado:",
    '"children"' in script_content or "children:" in script_content
)

✅ Adultos y niños añadidos al payload.

Verificación:
Campo adults enviado: True
Campo children enviado: True


In [98]:
authorized_places_path = "/content/travel_ai_project/authorized_places.json"

with open(
    authorized_places_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        authorized_places_with_costs,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ authorized_places.json actualizado con costos.")
print("Archivo:", authorized_places_path)

print("\nVerificación - Usaquén / caminar:")

print(
    json.dumps(
        authorized_places_with_costs[
            "Usaquén"
        ]["allowed_activities"]["caminar"],
        ensure_ascii=False,
        indent=2
    )
)

✅ authorized_places.json actualizado con costos.
Archivo: /content/travel_ai_project/authorized_places.json

Verificación - Usaquén / caminar:
{
  "min_cost_usd": 0,
  "max_cost_usd": 0
}


In [99]:
backend_code_v3 = r'''
import json
import re
import torch

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./qwen_travel_lora"
AUTHORIZED_PLACES_PATH = "./authorized_places.json"


app = FastAPI(
    title="Travel AI API",
    description=(
        "API para generar itinerarios personalizados "
        "con Qwen + LoRA + RAG estructurado + "
        "validación determinista + control de presupuesto."
    ),
    version="1.2"
)


app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


class TravelRequest(BaseModel):
    destination: str
    days: int
    budget: float
    currency: str

    adults: int
    children: int

    interests: list[str]
    travel_style: str
    restrictions: list[str]


# ============================================================
# CARGAR BASE AUTORIZADA
# ============================================================

with open(
    AUTHORIZED_PLACES_PATH,
    "r",
    encoding="utf-8"
) as f:
    authorized_places = json.load(f)


# ============================================================
# CARGAR MODELO
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()


# ============================================================
# LIMPIEZA JSON
# ============================================================

def clean_json_response(response: str):

    cleaned = response.strip()

    cleaned = re.sub(
        r"^```json\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    return cleaned.strip()


# ============================================================
# VALIDACIÓN DETERMINISTA
# ============================================================

def validate_itinerary(
    itinerary: dict,
    expected_days: int
):

    errors = []

    if "dias" not in itinerary:
        return [
            "Falta la clave principal 'dias'."
        ]

    days = itinerary["dias"]

    if len(days) != expected_days:
        errors.append(
            f"Se esperaban {expected_days} días "
            f"y se encontraron {len(days)}."
        )

    periods = [
        "manana",
        "tarde",
        "noche"
    ]

    place_counter = {}

    for expected_day, day in enumerate(
        days,
        start=1
    ):

        if day.get("dia") != expected_day:
            errors.append(
                f"Número de día incorrecto. "
                f"Se esperaba {expected_day}."
            )

        for period in periods:

            if period not in day:
                errors.append(
                    f"Día {expected_day}: "
                    f"falta '{period}'."
                )
                continue

            block = day[period]

            place = block.get("lugar")
            activity = block.get("actividad")

            # -------------------------------
            # Lugar autorizado
            # -------------------------------

            if place not in authorized_places:

                errors.append(
                    f"Día {expected_day} - {period}: "
                    f"lugar no autorizado '{place}'."
                )

                continue


            place_counter[place] = (
                place_counter.get(place, 0) + 1
            )


            # -------------------------------
            # Actividad autorizada
            # -------------------------------

            allowed_activities = authorized_places[
                place
            ]["allowed_activities"]

            if activity not in allowed_activities:

                errors.append(
                    f"Día {expected_day} - {period}: "
                    f"actividad no autorizada para "
                    f"'{place}': '{activity}'."
                )


    # -------------------------------
    # Control de repetición
    # -------------------------------

    for place, count in place_counter.items():

        if count > 2:

            errors.append(
                f"El lugar '{place}' aparece "
                f"{count} veces; máximo permitido: 2."
            )

    return errors


# ============================================================
# AÑADIR COSTOS AL ITINERARIO
# ============================================================

def add_costs_to_itinerary(
    itinerary: dict,
    adults: int,
    children: int,
    budget: float,
    currency: str
):

    total_travelers = adults + children

    total_min = 0
    total_max = 0

    periods = [
        "manana",
        "tarde",
        "noche"
    ]


    for day in itinerary["dias"]:

        day_min = 0
        day_max = 0

        for period in periods:

            block = day[period]

            place = block["lugar"]
            activity = block["actividad"]

            cost_data = authorized_places[
                place
            ]["allowed_activities"][activity]

            min_per_person = cost_data[
                "min_cost_usd"
            ]

            max_per_person = cost_data[
                "max_cost_usd"
            ]

            min_total = (
                min_per_person * total_travelers
            )

            max_total = (
                max_per_person * total_travelers
            )


            # Costos añadidos a la tarjeta
            block["costo_estimado"] = {
                "moneda": "USD",

                "por_persona": {
                    "min": min_per_person,
                    "max": max_per_person
                },

                "grupo": {
                    "min": min_total,
                    "max": max_total
                }
            }


            day_min += min_total
            day_max += max_total


        # Resumen diario
        day["costo_dia"] = {
            "moneda": "USD",
            "min": day_min,
            "max": day_max
        }


        total_min += day_min
        total_max += day_max


    # ========================================================
    # PRESUPUESTO
    # ========================================================

    budget_evaluation = {
        "moneda_costos": "USD",
        "presupuesto_ingresado": budget,
        "moneda_presupuesto": currency,
        "costo_estimado_min": total_min,
        "costo_estimado_max": total_max,
        "viajeros": {
            "adultos": adults,
            "ninos": children,
            "total": total_travelers
        }
    }


    # Comparación directa solo si el presupuesto está en USD.
    # No hacemos conversión automática para evitar inventar
    # tasas de cambio.
    if currency.upper() == "USD":

        if total_max <= budget:

            status = "dentro_del_presupuesto"

        elif total_min > budget:

            status = "fuera_del_presupuesto"

        else:

            status = "riesgo_de_superar_presupuesto"


        budget_evaluation[
            "estado"
        ] = status


        budget_evaluation[
            "saldo_estimado_usando_maximo"
        ] = budget - total_max


    else:

        budget_evaluation[
            "estado"
        ] = "requiere_conversion_moneda"

        budget_evaluation[
            "advertencia"
        ] = (
            "Los costos del prototipo están almacenados "
            "en USD. No se realizó conversión automática "
            "para evitar utilizar una tasa de cambio no "
            "verificada."
        )


    itinerary[
        "resumen_presupuesto"
    ] = budget_evaluation


    return itinerary


# ============================================================
# PROMPT
# ============================================================

def build_prompt(data: TravelRequest):

    # El modelo necesita conocer nombres y actividades,
    # pero NO debe calcular los precios.
    prompt_places = {}

    for place, info in authorized_places.items():

        prompt_places[place] = {
            "category": info["category"],
            "allowed_activities": list(
                info["allowed_activities"].keys()
            )
        }


    authorized_places_json = json.dumps(
        prompt_places,
        ensure_ascii=False,
        indent=2
    )


    return f"""
Genera un itinerario personalizado para {data.destination}.

BASE AUTORIZADA:
{authorized_places_json}

PREFERENCIAS:
- Duración: {data.days} días
- Presupuesto: {data.budget} {data.currency}
- Adultos: {data.adults}
- Niños: {data.children}
- Intereses: {", ".join(data.interests)}
- Ritmo: {data.travel_style}
- Restricciones: {", ".join(data.restrictions)}

REGLAS OBLIGATORIAS:

1. Genera exactamente {data.days} días.

2. Cada día debe tener exactamente:
   - manana
   - tarde
   - noche

3. Cada bloque debe contener únicamente:
   - lugar
   - actividad

4. "lugar" debe coincidir EXACTAMENTE
   con una clave de BASE AUTORIZADA.

5. "actividad" debe coincidir EXACTAMENTE
   con una actividad autorizada para ese lugar.

6. No combines actividades.

7. No parafrasees actividades.

8. No inventes lugares.

9. No inventes actividades.

10. NO calcules costos.

11. NO agregues precios.

12. NO agregues horarios.

13. NO agregues transporte.

14. No repitas un lugar más de dos veces.

15. Prioriza los intereses del usuario.

16. Devuelve únicamente JSON válido.

FORMATO EXACTO:

{{
  "dias": [
    {{
      "dia": 1,
      "manana": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta"
      }},
      "tarde": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta"
      }},
      "noche": {{
        "lugar": "nombre exacto",
        "actividad": "actividad exacta"
      }}
    }}
  ]
}}
"""


# ============================================================
# ENDPOINTS
# ============================================================

@app.get("/")
def root():

    return {
        "status": "ok",
        "service": "Travel AI API",
        "version": "1.2"
    }


@app.post("/generate")
def generate_itinerary(
    data: TravelRequest
):

    # Validaciones básicas de entrada
    if data.adults < 1:

        raise HTTPException(
            status_code=400,
            detail="Debe existir al menos un adulto."
        )


    if data.children < 0:

        raise HTTPException(
            status_code=400,
            detail="El número de niños no puede ser negativo."
        )


    if data.budget <= 0:

        raise HTTPException(
            status_code=400,
            detail="El presupuesto debe ser mayor que cero."
        )


    prompt = build_prompt(data)


    messages = [
        {
            "role": "system",
            "content": (
                "Eres un planificador profesional "
                "de viajes. Debes seleccionar "
                "exclusivamente valores exactos "
                "de la base autorizada y devolver "
                "únicamente JSON válido."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1400,
            do_sample=False
        )


    raw_response = tokenizer.decode(
        outputs[0][
            inputs["input_ids"].shape[1]:
        ],
        skip_special_tokens=True
    )


    cleaned_response = clean_json_response(
        raw_response
    )


    # ========================================================
    # JSON
    # ========================================================

    try:

        itinerary = json.loads(
            cleaned_response
        )

    except json.JSONDecodeError as e:

        raise HTTPException(
            status_code=422,
            detail={
                "message": (
                    "El modelo no generó JSON válido."
                ),
                "error": str(e),
                "raw_response": raw_response
            }
        )


    # ========================================================
    # VALIDACIÓN
    # ========================================================

    validation_errors = validate_itinerary(
        itinerary,
        data.days
    )


    if validation_errors:

        raise HTTPException(
            status_code=422,
            detail={
                "message": (
                    "El itinerario generado no "
                    "superó la validación determinista."
                ),
                "validation_errors": validation_errors,
                "raw_response": raw_response
            }
        )


    # ========================================================
    # COSTOS
    # ========================================================

    itinerary = add_costs_to_itinerary(
        itinerary=itinerary,
        adults=data.adults,
        children=data.children,
        budget=data.budget,
        currency=data.currency
    )


    return {
        "status": "validated",
        "itinerary": itinerary
    }


# ============================================================
# FRONTEND
# ============================================================

app.mount(
    "/ui",
    StaticFiles(
        directory="./frontend",
        html=True
    ),
    name="frontend"
)
'''


app_path = "/content/travel_ai_project/app.py"

with open(
    app_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(backend_code_v3)


print(
    "✅ app.py actualizado con control de costos "
    "y presupuesto."
)

print("Versión API: 1.2")

print("\nNuevas funciones:")
print("- Viajeros: adultos + niños")
print("- Costos por persona")
print("- Costos por grupo")
print("- Costos por día")
print("- Costo total mínimo/máximo")
print("- Comparación con presupuesto")

✅ app.py actualizado con control de costos y presupuesto.
Versión API: 1.2

Nuevas funciones:
- Viajeros: adultos + niños
- Costos por persona
- Costos por grupo
- Costos por día
- Costo total mínimo/máximo
- Comparación con presupuesto


In [100]:
import subprocess
import time
import os

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

# Detener servidor anterior
try:
    if server_process.poll() is None:
        print("Deteniendo servidor FastAPI anterior...")

        server_process.terminate()

        try:
            server_process.wait(timeout=10)
            print("✅ Servidor anterior detenido.")

        except subprocess.TimeoutExpired:
            server_process.kill()
            print("⚠️ Servidor anterior forzado a cerrar.")

except NameError:
    print("No se encontró un servidor anterior en memoria.")


# Iniciar API 1.2
server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Iniciando FastAPI versión 1.2...")

time.sleep(5)

if server_process.poll() is None:
    print("✅ Proceso FastAPI iniciado.")
    print("PID:", server_process.pid)
    print("Puerto: 8000")

else:
    print("❌ El servidor terminó inesperadamente.")
    print(server_process.stdout.read())

Deteniendo servidor FastAPI anterior...
✅ Servidor anterior detenido.

⏳ Iniciando FastAPI versión 1.2...
✅ Proceso FastAPI iniciado.
PID: 16449
Puerto: 8000


In [101]:
import requests
import json
import time

base_url = "http://127.0.0.1:8000"

# 1. Verificar versión
print("Verificando API...\n")

for attempt in range(1, 11):

    try:
        root_response = requests.get(
            f"{base_url}/",
            timeout=5
        )

        if root_response.ok:
            root_data = root_response.json()

            print("Status:", root_response.status_code)
            print("Respuesta:", root_data)
            break

    except requests.exceptions.RequestException:
        print(f"Intento {attempt}: todavía cargando...")
        time.sleep(3)


# 2. Probar generación con viajeros y presupuesto
test_payload_v12 = {
    "destination": "Bogotá, Colombia",
    "days": 4,
    "budget": 700,
    "currency": "USD",
    "adults": 2,
    "children": 0,
    "interests": [
        "cultura",
        "gastronomía"
    ],
    "travel_style": "moderado",
    "restrictions": [
        "sin actividades extremas"
    ]
}

print("\nProbando generación con costos...\n")

response = requests.post(
    f"{base_url}/generate",
    json=test_payload_v12,
    timeout=180
)

print("Status code:", response.status_code)

response_data = response.json()

print(
    json.dumps(
        response_data,
        ensure_ascii=False,
        indent=2
    )
)

Verificando API...

Status: 200
Respuesta: {'status': 'ok', 'service': 'Travel AI API', 'version': '1.2'}

Probando generación con costos...

Status code: 422
{
  "detail": {
    "message": "El itinerario generado no superó la validación determinista.",
    "validation_errors": [
      "El lugar 'Monserrate' aparece 4 veces; máximo permitido: 2."
    ],
    "raw_response": "{\n  \"dias\": [\n    {\n      \"dia\": 1,\n      \"manana\": {\n        \"lugar\": \"Usaquén\",\n        \"actividad\": \"caminar\"\n      },\n      \"tarde\": {\n        \"lugar\": \"Mercado de La Perseverancia\",\n        \"actividad\": \"conocer comida colombiana\"\n      },\n      \"noche\": {\n        \"lugar\": \"Parque de la 93\",\n        \"actividad\": \"caminar\"\n      }\n    },\n    {\n      \"dia\": 2,\n      \"manana\": {\n        \"lugar\": \"La Candelaria\",\n        \"actividad\": \"recorrer arquitectura colonial\"\n      },\n      \"tarde\": {\n        \"lugar\": \"Chorro de Quevedo\",\n        \"

In [102]:
app_path = "/content/travel_ai_project/app.py"

with open(
    app_path,
    "r",
    encoding="utf-8"
) as f:
    app_content = f.read()


old_block = '''
    validation_errors = validate_itinerary(
        itinerary,
        data.days
    )


    if validation_errors:

        raise HTTPException(
            status_code=422,
            detail={
                "message": (
                    "El itinerario generado no "
                    "superó la validación determinista."
                ),
                "validation_errors": validation_errors,
                "raw_response": raw_response
            }
        )


    # ========================================================
    # COSTOS
    # ========================================================

    itinerary = add_costs_to_itinerary(
'''

new_block = '''
    validation_errors = validate_itinerary(
        itinerary,
        data.days
    )


    # ========================================================
    # REINTENTO AUTOMÁTICO
    # ========================================================

    if validation_errors:

        correction_prompt = f"""
La respuesta anterior no superó la validación.

ERRORES DETECTADOS:
{json.dumps(validation_errors, ensure_ascii=False, indent=2)}

RESPUESTA ANTERIOR:
{json.dumps(itinerary, ensure_ascii=False, indent=2)}

Corrige exclusivamente los errores detectados.

REGLAS:

1. Mantén exactamente {data.days} días.
2. Cada día debe contener manana, tarde y noche.
3. Usa solo lugares de la BASE AUTORIZADA.
4. Usa solo actividades exactas autorizadas para cada lugar.
5. No repitas ningún lugar más de dos veces.
6. No inventes lugares ni actividades.
7. No añadas precios, horarios ni transporte.
8. Devuelve únicamente JSON válido.

BASE AUTORIZADA:
{json.dumps(prompt_places, ensure_ascii=False, indent=2)}
"""

        correction_messages = [
            {
                "role": "system",
                "content": (
                    "Corrige itinerarios usando exclusivamente "
                    "los valores de la base autorizada. "
                    "Devuelve únicamente JSON válido."
                )
            },
            {
                "role": "user",
                "content": correction_prompt
            }
        ]

        correction_text = tokenizer.apply_chat_template(
            correction_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        correction_inputs = tokenizer(
            correction_text,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():

            correction_outputs = model.generate(
                **correction_inputs,
                max_new_tokens=1400,
                do_sample=False
            )

        corrected_raw_response = tokenizer.decode(
            correction_outputs[0][
                correction_inputs["input_ids"].shape[1]:
            ],
            skip_special_tokens=True
        )

        corrected_cleaned = clean_json_response(
            corrected_raw_response
        )

        try:

            corrected_itinerary = json.loads(
                corrected_cleaned
            )

        except json.JSONDecodeError as e:

            raise HTTPException(
                status_code=422,
                detail={
                    "message": (
                        "El reintento no generó JSON válido."
                    ),
                    "error": str(e),
                    "first_validation_errors": validation_errors,
                    "raw_response": corrected_raw_response
                }
            )


        corrected_errors = validate_itinerary(
            corrected_itinerary,
            data.days
        )


        if corrected_errors:

            raise HTTPException(
                status_code=422,
                detail={
                    "message": (
                        "El itinerario no superó la validación "
                        "ni después del reintento automático."
                    ),
                    "first_validation_errors": validation_errors,
                    "second_validation_errors": corrected_errors,
                    "raw_response": corrected_raw_response
                }
            )


        itinerary = corrected_itinerary


    # ========================================================
    # COSTOS
    # ========================================================

    itinerary = add_costs_to_itinerary(
'''


# También necesitamos que prompt_places esté disponible
# fuera de build_prompt()
marker = '''
def build_prompt(data: TravelRequest):

    # El modelo necesita conocer nombres y actividades,
    # pero NO debe calcular los precios.
    prompt_places = {}
'''

replacement = '''
def get_prompt_places():

    prompt_places = {}

    for place, info in authorized_places.items():

        prompt_places[place] = {
            "category": info["category"],
            "allowed_activities": list(
                info["allowed_activities"].keys()
            )
        }

    return prompt_places


def build_prompt(data: TravelRequest):

    # El modelo necesita conocer nombres y actividades,
    # pero NO debe calcular los precios.
    prompt_places = get_prompt_places()
'''


if marker in app_content:
    app_content = app_content.replace(
        marker,
        replacement
    )


# Eliminar el bucle duplicado que quedó dentro de build_prompt
duplicate_block = '''
    for place, info in authorized_places.items():

        prompt_places[place] = {
            "category": info["category"],
            "allowed_activities": list(
                info["allowed_activities"].keys()
            )
        }


    authorized_places_json = json.dumps(
'''

replacement_block = '''
    authorized_places_json = json.dumps(
'''

if duplicate_block in app_content:
    app_content = app_content.replace(
        duplicate_block,
        replacement_block
    )


# Hacer prompt_places accesible en generate_itinerary
generate_marker = '''
    prompt = build_prompt(data)


    messages = [
'''

generate_replacement = '''
    prompt_places = get_prompt_places()

    prompt = build_prompt(data)


    messages = [
'''

if generate_marker in app_content:
    app_content = app_content.replace(
        generate_marker,
        generate_replacement
    )


if old_block in app_content:
    app_content = app_content.replace(
        old_block,
        new_block
    )

    with open(
        app_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(app_content)

    print("✅ Reintento automático añadido al backend.")

else:
    print(
        "⚠️ No se encontró el bloque principal esperado."
    )


print("\nVerificación:")
print(
    "get_prompt_places:",
    "def get_prompt_places()" in app_content
)
print(
    "correction_prompt:",
    "correction_prompt =" in app_content
)
print(
    "corrected_errors:",
    "corrected_errors =" in app_content
)

✅ Reintento automático añadido al backend.

Verificación:
get_prompt_places: True
correction_prompt: True
corrected_errors: True


In [103]:
import subprocess
import time
import os

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

# Detener servidor anterior
try:
    if server_process.poll() is None:
        print("Deteniendo servidor FastAPI anterior...")

        server_process.terminate()

        try:
            server_process.wait(timeout=10)
            print("✅ Servidor anterior detenido.")

        except subprocess.TimeoutExpired:
            server_process.kill()
            print("⚠️ Servidor anterior forzado a cerrar.")

except NameError:
    print("No se encontró un servidor anterior en memoria.")


# Iniciar nueva versión
server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Iniciando FastAPI con reintento automático...")

time.sleep(5)

if server_process.poll() is None:
    print("✅ Proceso FastAPI iniciado.")
    print("PID:", server_process.pid)
    print("Puerto: 8000")

else:
    print("❌ El servidor terminó inesperadamente.")
    print(server_process.stdout.read())

Deteniendo servidor FastAPI anterior...
✅ Servidor anterior detenido.

⏳ Iniciando FastAPI con reintento automático...
✅ Proceso FastAPI iniciado.
PID: 17452
Puerto: 8000


In [104]:
import requests
import json
import time

base_url = "http://127.0.0.1:8000"

# Esperar a que la API esté lista
print("Verificando disponibilidad de la API...\n")

for attempt in range(1, 11):
    try:
        root_response = requests.get(
            f"{base_url}/",
            timeout=5
        )

        if root_response.ok:
            print(
                "✅ API disponible:",
                root_response.json()
            )
            break

    except requests.exceptions.RequestException:
        print(
            f"Intento {attempt}: todavía cargando..."
        )
        time.sleep(3)


test_payload_v12 = {
    "destination": "Bogotá, Colombia",
    "days": 4,
    "budget": 700,
    "currency": "USD",
    "adults": 2,
    "children": 0,
    "interests": [
        "cultura",
        "gastronomía"
    ],
    "travel_style": "moderado",
    "restrictions": [
        "sin actividades extremas"
    ]
}

print("\nGenerando itinerario...\n")

response = requests.post(
    f"{base_url}/generate",
    json=test_payload_v12,
    timeout=240
)

print("Status code:", response.status_code)

try:
    response_data = response.json()
except Exception:
    response_data = {
        "raw_text": response.text
    }

print("\nRESPUESTA:\n")

print(
    json.dumps(
        response_data,
        ensure_ascii=False,
        indent=2
    )
)


if response.status_code == 200:

    print("\n✅ Itinerario validado y costos calculados.")

    itinerary = response_data.get(
        "itinerary",
        {}
    )

    budget_summary = itinerary.get(
        "resumen_presupuesto",
        {}
    )

    print("\nRESUMEN DE PRESUPUESTO:\n")

    print(
        json.dumps(
            budget_summary,
            ensure_ascii=False,
            indent=2
        )
    )

elif response.status_code == 422:

    print(
        "\n⚠️ El itinerario no superó "
        "la validación incluso después "
        "del reintento automático."
    )

else:

    print(
        "\n❌ Se produjo un error distinto "
        "al esperado."
    )

Verificando disponibilidad de la API...

✅ API disponible: {'status': 'ok', 'service': 'Travel AI API', 'version': '1.2'}

Generando itinerario...

Status code: 422

RESPUESTA:

{
  "detail": {
    "message": "El itinerario no superó la validación ni después del reintento automático.",
    "first_validation_errors": [
      "El lugar 'Monserrate' aparece 4 veces; máximo permitido: 2."
    ],
    "second_validation_errors": [
      "El lugar 'Monserrate' aparece 4 veces; máximo permitido: 2."
    ],
    "raw_response": "```json\n{\n  \"dias\": [\n    {\n      \"dia\": 1,\n      \"manana\": {\n        \"lugar\": \"Usaquén\",\n        \"actividad\": \"caminar\"\n      },\n      \"tarde\": {\n        \"lugar\": \"Mercado de La Perseverancia\",\n        \"actividad\": \"conocer comida colombiana\"\n      },\n      \"noche\": {\n        \"lugar\": \"Parque de la 93\",\n        \"actividad\": \"caminar\"\n      }\n    },\n    {\n      \"dia\": 2,\n      \"manana\": {\n        \"lugar\": \"La 

In [105]:
app_path = "/content/travel_ai_project/app.py"

with open(
    app_path,
    "r",
    encoding="utf-8"
) as f:
    app_content = f.read()


repair_function = r'''

# ============================================================
# REPARACIÓN DETERMINISTA DE REPETICIONES
# ============================================================

def repair_repeated_places(
    itinerary: dict,
    max_repetitions: int = 2
):

    periods = [
        "manana",
        "tarde",
        "noche"
    ]

    place_counter = {}

    # Contar uso actual
    for day in itinerary["dias"]:

        for period in periods:

            place = day[period]["lugar"]

            place_counter[place] = (
                place_counter.get(place, 0) + 1
            )


    # Uso progresivo durante reparación
    seen = {}

    for day in itinerary["dias"]:

        for period in periods:

            block = day[period]

            place = block["lugar"]

            seen[place] = (
                seen.get(place, 0) + 1
            )

            # Mantener las primeras apariciones
            if seen[place] <= max_repetitions:
                continue


            # Buscar alternativas con menor uso
            candidates = []

            for candidate_place, info in authorized_places.items():

                candidate_count = place_counter.get(
                    candidate_place,
                    0
                )

                if candidate_count < max_repetitions:

                    candidates.append(
                        (
                            candidate_count,
                            candidate_place
                        )
                    )


            if not candidates:
                continue


            # Priorizar el lugar menos utilizado
            candidates.sort(
                key=lambda x: (
                    x[0],
                    x[1]
                )
            )

            new_place = candidates[0][1]


            # Seleccionar una actividad válida
            available_activities = list(
                authorized_places[
                    new_place
                ]["allowed_activities"].keys()
            )

            new_activity = available_activities[0]


            # Actualizar contadores
            place_counter[place] -= 1

            place_counter[new_place] = (
                place_counter.get(new_place, 0) + 1
            )


            # Reemplazar bloque
            block["lugar"] = new_place
            block["actividad"] = new_activity


    return itinerary

'''


# Insertar función antes de add_costs_to_itinerary
marker = '''
# ============================================================
# AÑADIR COSTOS AL ITINERARIO
# ============================================================
'''

if "def repair_repeated_places(" not in app_content:

    app_content = app_content.replace(
        marker,
        repair_function + "\n" + marker
    )


old_retry_block = '''
        if corrected_errors:

            raise HTTPException(
                status_code=422,
                detail={
                    "message": (
                        "El itinerario no superó la validación "
                        "ni después del reintento automático."
                    ),
                    "first_validation_errors": validation_errors,
                    "second_validation_errors": corrected_errors,
                    "raw_response": corrected_raw_response
                }
            )


        itinerary = corrected_itinerary
'''


new_retry_block = '''
        if corrected_errors:

            # Reparación determinista final
            repaired_itinerary = repair_repeated_places(
                corrected_itinerary
            )

            repaired_errors = validate_itinerary(
                repaired_itinerary,
                data.days
            )

            if repaired_errors:

                raise HTTPException(
                    status_code=422,
                    detail={
                        "message": (
                            "El itinerario no superó la validación "
                            "después del reintento ni de la "
                            "reparación determinista."
                        ),
                        "first_validation_errors":
                            validation_errors,
                        "second_validation_errors":
                            corrected_errors,
                        "repair_validation_errors":
                            repaired_errors,
                        "raw_response":
                            corrected_raw_response
                    }
                )

            itinerary = repaired_itinerary

        else:

            itinerary = corrected_itinerary
'''


if old_retry_block in app_content:

    app_content = app_content.replace(
        old_retry_block,
        new_retry_block
    )

    with open(
        app_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(app_content)

    print(
        "✅ Reparación determinista de "
        "repeticiones añadida."
    )

else:

    print(
        "⚠️ No se encontró el bloque "
        "de reintento esperado."
    )


print("\nVerificación:")

print(
    "repair_repeated_places:",
    "def repair_repeated_places(" in app_content
)

print(
    "repaired_itinerary:",
    "repaired_itinerary =" in app_content
)

print(
    "repaired_errors:",
    "repaired_errors =" in app_content
)

✅ Reparación determinista de repeticiones añadida.

Verificación:
repair_repeated_places: True
repaired_itinerary: True
repaired_errors: True


In [106]:
import subprocess
import time
import os

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

# Detener servidor anterior
try:
    if server_process.poll() is None:
        print("Deteniendo servidor FastAPI anterior...")

        server_process.terminate()

        try:
            server_process.wait(timeout=10)
            print("✅ Servidor anterior detenido.")

        except subprocess.TimeoutExpired:
            server_process.kill()
            print("⚠️ Servidor anterior forzado a cerrar.")

except NameError:
    print("No se encontró un servidor anterior en memoria.")


# Iniciar nueva versión
server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Iniciando FastAPI con reparación determinista...")

time.sleep(5)

if server_process.poll() is None:
    print("✅ Proceso FastAPI iniciado.")
    print("PID:", server_process.pid)
    print("Puerto: 8000")

else:
    print("❌ El servidor terminó inesperadamente.")
    print(server_process.stdout.read())


Deteniendo servidor FastAPI anterior...
✅ Servidor anterior detenido.

⏳ Iniciando FastAPI con reparación determinista...
✅ Proceso FastAPI iniciado.
PID: 19009
Puerto: 8000


In [107]:
import requests
import json
import time

base_url = "http://127.0.0.1:8000"

print("Verificando disponibilidad de la API...\n")

for attempt in range(1, 11):

    try:
        root_response = requests.get(
            f"{base_url}/",
            timeout=5
        )

        if root_response.ok:
            print(
                "✅ API disponible:",
                root_response.json()
            )
            break

    except requests.exceptions.RequestException:
        print(
            f"Intento {attempt}: todavía cargando..."
        )

        time.sleep(3)


test_payload_final = {
    "destination": "Bogotá, Colombia",
    "days": 4,
    "budget": 700,
    "currency": "USD",
    "adults": 2,
    "children": 0,
    "interests": [
        "cultura",
        "gastronomía"
    ],
    "travel_style": "moderado",
    "restrictions": [
        "sin actividades extremas"
    ]
}

print("\nGenerando itinerario final...\n")

response = requests.post(
    f"{base_url}/generate",
    json=test_payload_final,
    timeout=240
)

print("Status code:", response.status_code)

try:
    response_data = response.json()
except Exception:
    response_data = {
        "raw_text": response.text
    }

print("\nRESPUESTA COMPLETA:\n")

print(
    json.dumps(
        response_data,
        ensure_ascii=False,
        indent=2
    )
)


if response.status_code == 200:

    itinerary = response_data.get(
        "itinerary",
        {}
    )

    print(
        "\n✅ Itinerario validado, "
        "reparado si fue necesario "
        "y costos calculados."
    )

    print("\nRESUMEN DE PRESUPUESTO:\n")

    print(
        json.dumps(
            itinerary.get(
                "resumen_presupuesto",
                {}
            ),
            ensure_ascii=False,
            indent=2
        )
    )

    print("\nCOSTOS POR DÍA:\n")

    for day in itinerary.get("dias", []):

        print(
            f"Día {day['dia']}:",
            day.get("costo_dia")
        )

elif response.status_code == 422:

    print(
        "\n⚠️ El itinerario todavía no "
        "superó alguna validación."
    )

else:

    print(
        "\n❌ Se produjo un error distinto "
        "al esperado."
    )

Verificando disponibilidad de la API...

✅ API disponible: {'status': 'ok', 'service': 'Travel AI API', 'version': '1.2'}

Generando itinerario final...

Status code: 200

RESPUESTA COMPLETA:

{
  "status": "validated",
  "itinerary": {
    "dias": [
      {
        "dia": 1,
        "manana": {
          "lugar": "Usaquén",
          "actividad": "caminar",
          "costo_estimado": {
            "moneda": "USD",
            "por_persona": {
              "min": 0,
              "max": 0
            },
            "grupo": {
              "min": 0,
              "max": 0
            }
          }
        },
        "tarde": {
          "lugar": "Mercado de La Perseverancia",
          "actividad": "conocer comida colombiana",
          "costo_estimado": {
            "moneda": "USD",
            "por_persona": {
              "min": 8,
              "max": 15
            },
            "grupo": {
              "min": 16,
              "max": 30
            }
          }
        },
 

In [108]:
place_details = {
    "Usaquén": {
        "descripcion": (
            "Sector tradicional de Bogotá con restaurantes, cafés, "
            "arquitectura y espacios culturales."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Usaquen+Bogota+Colombia"
        )
    },

    "Zona G": {
        "descripcion": (
            "Sector reconocido por su oferta gastronómica "
            "y variedad de restaurantes."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Zona+G+Bogota+Colombia"
        )
    },

    "Mercado de La Perseverancia": {
        "descripcion": (
            "Mercado tradicional conocido por su oferta "
            "de comida colombiana y gastronomía local."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Mercado+de+La+Perseverancia+Bogota"
        )
    },

    "Parque de la 93": {
        "descripcion": (
            "Zona urbana con restaurantes, cafés y espacios "
            "para caminar y realizar actividades relajadas."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Parque+de+la+93+Bogota"
        )
    },

    "La Candelaria": {
        "descripcion": (
            "Centro histórico de Bogotá, caracterizado por "
            "arquitectura colonial, museos, plazas y oferta cultural."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=La+Candelaria+Bogota+Colombia"
        )
    },

    "Chorro de Quevedo": {
        "descripcion": (
            "Espacio tradicional de La Candelaria relacionado "
            "con historia, arquitectura y recorridos culturales."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Chorro+de+Quevedo+Bogota"
        )
    },

    "Museo Nacional de Colombia": {
        "descripcion": (
            "Museo dedicado a historia, patrimonio, arte "
            "y cultura colombiana."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Museo+Nacional+de+Colombia+Bogota"
        )
    },

    "Biblioteca Luis Ángel Arango": {
        "descripcion": (
            "Espacio cultural relacionado con lectura, "
            "patrimonio, arte y actividades culturales."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Biblioteca+Luis+Angel+Arango+Bogota"
        )
    },

    "Museo Botero": {
        "descripcion": (
            "Museo del centro histórico con obras de Fernando Botero "
            "y artistas internacionales."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Museo+Botero+Bogota"
        )
    },

    "Monserrate": {
        "descripcion": (
            "Punto panorámico de Bogotá con acceso mediante "
            "teleférico o funicular y vistas de la ciudad."
        ),
        "google_maps_url": (
            "https://www.google.com/maps/search/?api=1"
            "&query=Monserrate+Bogota+Colombia"
        )
    }
}


for place, details in place_details.items():

    authorized_places_with_costs[
        place
    ]["descripcion"] = details["descripcion"]

    authorized_places_with_costs[
        place
    ]["google_maps_url"] = details["google_maps_url"]


# Guardar nuevamente el JSON utilizado por FastAPI
authorized_places_path = (
    "/content/travel_ai_project/authorized_places.json"
)

with open(
    authorized_places_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        authorized_places_with_costs,
        f,
        ensure_ascii=False,
        indent=2
    )


print("✅ Información adicional añadida.")
print("Lugares actualizados:", len(place_details))

print("\nEjemplo - Museo Botero:")
print(
    json.dumps(
        authorized_places_with_costs["Museo Botero"],
        ensure_ascii=False,
        indent=2
    )
)

✅ Información adicional añadida.
Lugares actualizados: 10

Ejemplo - Museo Botero:
{
  "category": [
    "arte",
    "cultura"
  ],
  "allowed_activities": {
    "observar obras de Fernando Botero": {
      "min_cost_usd": 0,
      "max_cost_usd": 5
    },
    "observar obras de artistas internacionales": {
      "min_cost_usd": 0,
      "max_cost_usd": 5
    },
    "realizar visita cultural": {
      "min_cost_usd": 0,
      "max_cost_usd": 5
    }
  },
  "descripcion": "Museo del centro histórico con obras de Fernando Botero y artistas internacionales.",
  "google_maps_url": "https://www.google.com/maps/search/?api=1&query=Museo+Botero+Bogota"
}


In [109]:
app_path = "/content/travel_ai_project/app.py"

with open(
    app_path,
    "r",
    encoding="utf-8"
) as f:
    app_content = f.read()


old_block = '''
            # Costos añadidos a la tarjeta
            block["costo_estimado"] = {
                "moneda": "USD",

                "por_persona": {
                    "min": min_per_person,
                    "max": max_per_person
                },

                "grupo": {
                    "min": min_total,
                    "max": max_total
                }
            }
'''


new_block = '''
            # Información adicional del lugar
            block["descripcion"] = authorized_places[
                place
            ].get(
                "descripcion",
                ""
            )

            block["google_maps_url"] = authorized_places[
                place
            ].get(
                "google_maps_url",
                ""
            )


            # Costos añadidos a la tarjeta
            block["costo_estimado"] = {
                "moneda": "USD",

                "por_persona": {
                    "min": min_per_person,
                    "max": max_per_person
                },

                "grupo": {
                    "min": min_total,
                    "max": max_total
                }
            }
'''


if old_block in app_content:

    app_content = app_content.replace(
        old_block,
        new_block
    )

    with open(
        app_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(app_content)

    print(
        "✅ Descripción y Google Maps "
        "añadidos a las tarjetas del backend."
    )

else:

    print(
        "⚠️ No se encontró el bloque esperado "
        "en add_costs_to_itinerary()."
    )


print("\nVerificación:")

print(
    "descripcion:",
    'block["descripcion"]' in app_content
)

print(
    "google_maps_url:",
    'block["google_maps_url"]' in app_content
)

✅ Descripción y Google Maps añadidos a las tarjetas del backend.

Verificación:
descripcion: True
google_maps_url: True


In [110]:
script_path = "/content/travel_ai_project/frontend/script.js"

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    script_content = f.read()


old_function = '''
function createPeriodCard(title, data) {

    return `
        <div class="period-card">

            <div class="period-title">
                ${title}
            </div>

            <p class="place">
                ${data.lugar}
            </p>

            <p class="activity">
                ${data.actividad}
            </p>

        </div>
    `;
}
'''


new_function = '''
function createPeriodCard(title, data) {

    const cost = data.costo_estimado || {};
    const groupCost = cost.grupo || {};

    const minCost =
        groupCost.min !== undefined
        ? groupCost.min
        : "-";

    const maxCost =
        groupCost.max !== undefined
        ? groupCost.max
        : "-";

    return `
        <div class="period-card">

            <div class="period-title">
                ${title}
            </div>

            <p class="place">
                ${data.lugar}
            </p>

            <p class="activity">
                ${data.actividad}
            </p>

            <p class="activity">
                <strong>Costo estimado grupo:</strong>
                USD ${minCost} - ${maxCost}
            </p>

            <details class="place-details">

                <summary>
                    Ver detalles
                </summary>

                <p>
                    ${data.descripcion || "Información no disponible."}
                </p>

                ${
                    data.google_maps_url
                    ? `
                        <a
                            href="${data.google_maps_url}"
                            target="_blank"
                            rel="noopener noreferrer"
                            class="maps-link"
                        >
                            Abrir en Google Maps
                        </a>
                    `
                    : ""
                }

            </details>

        </div>
    `;
}
'''


if old_function in script_content:

    script_content = script_content.replace(
        old_function,
        new_function
    )

    with open(
        script_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(script_content)

    print("✅ Tarjetas actualizadas con costo y detalles.")

else:

    print(
        "⚠️ No se encontró la función "
        "createPeriodCard esperada."
    )


print("\nVerificación:")

print(
    "Costo visible:",
    "Costo estimado grupo" in script_content
)

print(
    "Detalles:",
    "<details" in script_content
)

print(
    "Google Maps:",
    "Abrir en Google Maps" in script_content
)

✅ Tarjetas actualizadas con costo y detalles.

Verificación:
Costo visible: True
Detalles: True
Google Maps: True


In [111]:
styles_path = "/content/travel_ai_project/frontend/styles.css"

with open(
    styles_path,
    "r",
    encoding="utf-8"
) as f:
    styles_content = f.read()


extra_styles = r'''

.place-details {
    margin-top: 14px;
    padding-top: 12px;
    border-top: 1px solid var(--border);
}

.place-details summary {
    cursor: pointer;
    font-weight: 700;
    color: var(--primary);
    user-select: none;
}

.place-details p {
    margin: 10px 0 12px;
    color: var(--muted);
    font-size: 0.92rem;
}

.maps-link {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    padding: 8px 11px;
    border-radius: 10px;
    background: #eff6ff;
    color: #1d4ed8;
    text-decoration: none;
    font-size: 0.88rem;
    font-weight: 700;
}

.maps-link:hover {
    background: #dbeafe;
}
'''


if ".place-details {" not in styles_content:

    styles_content += extra_styles

    with open(
        styles_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(styles_content)

    print("✅ Estilos de detalles y Google Maps añadidos.")

else:
    print("ℹ️ Los estilos ya estaban añadidos.")


print("\nVerificación:")
print(
    "place-details:",
    ".place-details {" in styles_content
)

print(
    "maps-link:",
    ".maps-link {" in styles_content
)

✅ Estilos de detalles y Google Maps añadidos.

Verificación:
place-details: True
maps-link: True


In [112]:
import subprocess
import time
import os

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)

# Detener servidor anterior
try:
    if server_process.poll() is None:
        print("Deteniendo servidor FastAPI anterior...")

        server_process.terminate()

        try:
            server_process.wait(timeout=10)
            print("✅ Servidor anterior detenido.")

        except subprocess.TimeoutExpired:
            server_process.kill()
            print("⚠️ Servidor anterior forzado a cerrar.")

except NameError:
    print("No se encontró un servidor anterior en memoria.")


# Iniciar nueva versión
server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Iniciando FastAPI con todas las mejoras...")

time.sleep(5)

if server_process.poll() is None:
    print("✅ Proceso FastAPI iniciado.")
    print("PID:", server_process.pid)
    print("Puerto: 8000")

else:
    print("❌ El servidor terminó inesperadamente.")
    print(server_process.stdout.read())

Deteniendo servidor FastAPI anterior...
✅ Servidor anterior detenido.

⏳ Iniciando FastAPI con todas las mejoras...
✅ Proceso FastAPI iniciado.
PID: 20419
Puerto: 8000


In [113]:
styles_path = "/content/travel_ai_project/frontend/styles.css"

with open(
    styles_path,
    "r",
    encoding="utf-8"
) as f:
    styles_content = f.read()


design_upgrade = r'''

/* ============================================================
   MEJORA VISUAL - ITINERARIO
   ============================================================ */

.results {
    background: transparent;
    border: 0;
    box-shadow: none;
    padding-left: 0;
    padding-right: 0;
}

.results-header {
    padding: 0 4px 18px;
    border-bottom: 1px solid var(--border);
}

.day-card {
    position: relative;
    margin-top: 34px;
    padding: 0;
    overflow: hidden;
    border: 1px solid #dfe3ea;
    border-radius: 20px;
    background: #ffffff;
    box-shadow: 0 8px 26px rgba(15, 23, 42, 0.05);
}

/* Encabezado visual de cada día */
.day-card h3 {
    margin: 0;
    padding: 18px 24px;
    background: #172033;
    color: #ffffff;
    font-size: 1.15rem;
    font-weight: 800;
    letter-spacing: -0.01em;
}

/* Línea de acento superior */
.day-card::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 6px;
    height: 58px;
    background: #2563eb;
    z-index: 2;
}

.period-grid {
    display: grid;
    grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 0;
    background: #ffffff;
}

.period-card {
    min-height: 250px;
    padding: 24px 22px;
    border: 0;
    border-radius: 0;
    background: #ffffff;
}

.period-card + .period-card {
    border-left: 1px solid #e4e7ec;
}

.period-title {
    display: inline-flex;
    align-items: center;
    margin-bottom: 16px;
    padding: 5px 9px;
    border-radius: 7px;
    background: #eff6ff;
    color: #1d4ed8;
    font-size: 0.74rem;
    font-weight: 800;
    letter-spacing: 0.08em;
}

.place {
    margin-bottom: 7px;
    font-size: 1.02rem;
    line-height: 1.35;
}

.activity {
    font-size: 0.94rem;
    line-height: 1.5;
}

.place-details {
    margin-top: 18px;
}

.place-details summary {
    display: inline-flex;
    align-items: center;
    gap: 5px;
    font-size: 0.9rem;
}

.maps-link {
    margin-top: 4px;
}

/* Separación más visible entre días */
.day-card + .day-card {
    margin-top: 42px;
}

/* Día alterno ligeramente distinto */
.day-card:nth-child(even) h3 {
    background: #25324a;
}

/* Responsive */
@media (max-width: 780px) {

    .period-grid {
        grid-template-columns: 1fr;
    }

    .period-card + .period-card {
        border-left: 0;
        border-top: 1px solid #e4e7ec;
    }

    .period-card {
        min-height: auto;
    }

    .day-card {
        margin-top: 28px;
    }
}
'''


if "MEJORA VISUAL - ITINERARIO" not in styles_content:

    styles_content += design_upgrade

    with open(
        styles_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(styles_content)

    print("✅ Diseño del itinerario mejorado.")

else:
    print("ℹ️ La mejora visual ya estaba aplicada.")


print("\nCambios aplicados:")
print("- Encabezado oscuro por día")
print("- Separación visual más fuerte")
print("- Menos apariencia de tarjetas IA")
print("- Franjas mañana/tarde/noche más limpias")
print("- Mejor jerarquía tipográfica")

✅ Diseño del itinerario mejorado.

Cambios aplicados:
- Encabezado oscuro por día
- Separación visual más fuerte
- Menos apariencia de tarjetas IA
- Franjas mañana/tarde/noche más limpias
- Mejor jerarquía tipográfica


In [114]:
# ============================================================
# AJUSTE DE COSTOS ESTIMADOS DEL VIAJE
#
# Los valores representan gasto aproximado POR PERSONA
# durante cada bloque del itinerario.
# No representan únicamente precio de entrada.
# ============================================================

updated_costs = {

    "Usaquén": {
        "caminar": (10, 30),
        "visitar restaurantes": (20, 50),
        "visitar cafés": (8, 20),
        "observar arquitectura": (5, 15),
        "realizar actividades culturales": (10, 30)
    },

    "Zona G": {
        "experiencia culinaria": (30, 70),
        "visitar restaurantes": (25, 60),
        "cenar": (30, 70)
    },

    "Mercado de La Perseverancia": {
        "conocer comida colombiana": (10, 25),
        "explorar gastronomía local": (10, 25)
    },

    "Parque de la 93": {
        # Para 2 viajeros:
        # USD 25-75 x 2 = USD 50-150 aprox.
        "caminar": (25, 75),
        "visitar restaurantes": (30, 70),
        "visitar cafés": (10, 30),
        "realizar actividades relajadas": (20, 60)
    },

    "La Candelaria": {
        "recorrer arquitectura colonial": (5, 20),
        "caminar": (5, 15),
        "visitar museos": (10, 30),
        "visitar plazas": (5, 15),
        "realizar recorridos culturales": (15, 35)
    },

    "Chorro de Quevedo": {
        "conocer historia": (5, 15),
        "observar arquitectura": (5, 15),
        "realizar recorridos culturales": (10, 30)
    },

    "Museo Nacional de Colombia": {
        "conocer historia": (5, 20),
        "conocer patrimonio": (5, 20),
        "observar arte": (5, 20),
        "realizar visita cultural": (5, 20)
    },

    "Biblioteca Luis Ángel Arango": {
        "actividades de lectura": (5, 15),
        "conocer patrimonio": (5, 15),
        "observar arte": (5, 15),
        "realizar actividades culturales": (5, 20)
    },

    "Museo Botero": {
        "observar obras de Fernando Botero": (5, 15),
        "observar obras de artistas internacionales": (5, 15),
        "realizar visita cultural": (5, 15)
    },

    "Monserrate": {
        "observar vistas de la ciudad": (15, 35),
        "subir en teleférico": (15, 35),
        "subir en funicular": (15, 35)
    }
}


# ============================================================
# ACTUALIZAR BASE
# ============================================================

for place, activities in updated_costs.items():

    for activity, cost_range in activities.items():

        min_cost, max_cost = cost_range

        authorized_places_with_costs[
            place
        ]["allowed_activities"][
            activity
        ]["min_cost_usd"] = min_cost

        authorized_places_with_costs[
            place
        ]["allowed_activities"][
            activity
        ]["max_cost_usd"] = max_cost


# ============================================================
# GUARDAR JSON ACTUALIZADO
# ============================================================

authorized_places_path = (
    "/content/travel_ai_project/authorized_places.json"
)

with open(
    authorized_places_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        authorized_places_with_costs,
        f,
        ensure_ascii=False,
        indent=2
    )


print("✅ Rangos de gasto actualizados correctamente.")

print("\nEJEMPLO - PARQUE DE LA 93 / CAMINAR")

park_cost = authorized_places_with_costs[
    "Parque de la 93"
]["allowed_activities"]["caminar"]

print(
    json.dumps(
        park_cost,
        ensure_ascii=False,
        indent=2
    )
)

print("\nPara 2 viajeros:")

print(
    "Costo mínimo grupo:",
    park_cost["min_cost_usd"] * 2,
    "USD"
)

print(
    "Costo máximo grupo:",
    park_cost["max_cost_usd"] * 2,
    "USD"
)

✅ Rangos de gasto actualizados correctamente.

EJEMPLO - PARQUE DE LA 93 / CAMINAR
{
  "min_cost_usd": 25,
  "max_cost_usd": 75
}

Para 2 viajeros:
Costo mínimo grupo: 50 USD
Costo máximo grupo: 150 USD


In [115]:
styles_path = "/content/travel_ai_project/frontend/styles.css"
script_path = "/content/travel_ai_project/frontend/script.js"

# ============================================================
# 1. CSS - hover y selección
# ============================================================

with open(
    styles_path,
    "r",
    encoding="utf-8"
) as f:
    styles_content = f.read()


interaction_styles = r'''

/* ============================================================
   INTERACCIÓN DE TARJETAS
   ============================================================ */

.period-card {
    cursor: pointer;
    transition:
        transform 0.22s ease,
        box-shadow 0.22s ease,
        border-color 0.22s ease,
        background 0.22s ease;
}

.period-card:hover {
    position: relative;
    z-index: 3;
    transform: translateY(-4px);
    background: #ffffff;
    box-shadow:
        0 12px 28px rgba(15, 23, 42, 0.10);
}

.period-card:hover .place {
    color: #1d4ed8;
}

.period-card.selected {
    position: relative;
    z-index: 4;
    transform: translateY(-3px);
    background: #f8fbff;
    box-shadow:
        0 0 0 2px rgba(37, 99, 235, 0.22),
        0 14px 30px rgba(15, 23, 42, 0.10);
}

.period-card.selected .place {
    color: #1d4ed8;
}

.period-card.selected .period-title {
    background: #dbeafe;
    color: #1d4ed8;
}

.period-card:focus-visible {
    outline: 3px solid rgba(37, 99, 235, 0.30);
    outline-offset: -3px;
}
'''


if "INTERACCIÓN DE TARJETAS" not in styles_content:

    styles_content += interaction_styles

    with open(
        styles_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(styles_content)

    print("✅ Estilos interactivos añadidos.")

else:
    print("ℹ️ Los estilos interactivos ya existían.")


# ============================================================
# 2. JS - click en tarjeta
# ============================================================

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    script_content = f.read()


interaction_js = r'''

function activatePeriodCards() {

    const cards = document.querySelectorAll(
        ".period-card"
    );

    cards.forEach(card => {

        card.setAttribute(
            "tabindex",
            "0"
        );

        card.addEventListener(
            "click",
            event => {

                // Permitir que los enlaces de Google Maps
                // funcionen normalmente
                if (
                    event.target.closest(
                        ".maps-link"
                    )
                ) {
                    return;
                }

                // Quitar selección anterior
                cards.forEach(
                    otherCard => {
                        if (otherCard !== card) {
                            otherCard.classList.remove(
                                "selected"
                            );
                        }
                    }
                );

                // Activar tarjeta actual
                card.classList.toggle(
                    "selected"
                );

                // Abrir detalles automáticamente
                const details =
                    card.querySelector(
                        ".place-details"
                    );

                if (
                    details &&
                    card.classList.contains(
                        "selected"
                    )
                ) {
                    details.open = true;
                }
            }
        );


        // Accesibilidad con teclado
        card.addEventListener(
            "keydown",
            event => {

                if (
                    event.key === "Enter" ||
                    event.key === " "
                ) {

                    event.preventDefault();
                    card.click();
                }
            }
        );
    });
}

'''


if "function activatePeriodCards()" not in script_content:

    # Insertar antes de renderItinerary()
    marker = "function renderItinerary(data) {"

    script_content = script_content.replace(
        marker,
        interaction_js + "\n" + marker
    )


# Activar eventos después de dibujar las tarjetas
old_render_end = '''
        itineraryContainer.appendChild(dayCard);
    });

    hideStatus();
'''

new_render_end = '''
        itineraryContainer.appendChild(dayCard);
    });

    activatePeriodCards();

    hideStatus();
'''


if old_render_end in script_content:

    script_content = script_content.replace(
        old_render_end,
        new_render_end
    )


with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(script_content)


print("✅ Interacción por clic añadida.")

print("\nComportamiento:")
print("- Hover: elevación y sombra")
print("- Click: tarjeta seleccionada")
print("- Click: abre detalles")
print("- Nueva selección: desmarca la anterior")
print("- Google Maps sigue funcionando")
print("- Compatible con teclado")

✅ Estilos interactivos añadidos.
✅ Interacción por clic añadida.

Comportamiento:
- Hover: elevación y sombra
- Click: tarjeta seleccionada
- Click: abre detalles
- Nueva selección: desmarca la anterior
- Google Maps sigue funcionando
- Compatible con teclado


In [116]:
import os
import subprocess
import time


# ============================================================
# 1. AJUSTAR UX DE LAS TARJETAS
# ============================================================

styles_path = "/content/travel_ai_project/frontend/styles.css"

with open(
    styles_path,
    "r",
    encoding="utf-8"
) as f:
    styles_content = f.read()


ux_override = r'''

/* ============================================================
   UX TARJETAS V2 - INTERACCIÓN LIMPIA
   ============================================================ */

.period-card {
    cursor: pointer;

    /* Eliminar sensación de movimiento */
    transform: none !important;
    box-shadow: none !important;

    transition:
        background-color 0.14s ease,
        color 0.14s ease;
}


/* Hover: cambio de fondo únicamente */
.period-card:hover {
    position: relative;
    z-index: 1;

    transform: none !important;
    box-shadow: none !important;

    background: #f4f7fc;
}


/* Lugar ligeramente destacado */
.period-card:hover .place {
    color: #172033;
}


/* Tarjeta seleccionada */
.period-card.selected {
    position: relative;
    z-index: 1;

    transform: none !important;
    box-shadow: none !important;

    background: #eaf2ff;
}


/* Lugar seleccionado */
.period-card.selected .place {
    color: #1d4ed8;
}


/* Etiqueta mañana/tarde/noche */
.period-card.selected .period-title {
    background: #dbeafe;
    color: #1d4ed8;
}


/* Sin borde de foco exagerado */
.period-card:focus {
    outline: none;
}


/* Accesibilidad teclado */
.period-card:focus-visible {
    outline: 2px solid #bfdbfe;
    outline-offset: -2px;
}
'''


if "UX TARJETAS V2" not in styles_content:

    styles_content += ux_override

    with open(
        styles_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(styles_content)

    print("✅ Interacción visual V2 aplicada.")

else:
    print("ℹ️ Interacción visual V2 ya estaba aplicada.")


# ============================================================
# 2. REINICIAR FASTAPI
#    Necesario para volver a cargar authorized_places.json
#    con los nuevos costos.
# ============================================================

project_dir = "/content/travel_ai_project"

os.chdir(project_dir)


try:

    if server_process.poll() is None:

        print("\nDeteniendo FastAPI anterior...")

        server_process.terminate()

        try:
            server_process.wait(timeout=10)
            print("✅ Servidor anterior detenido.")

        except subprocess.TimeoutExpired:
            server_process.kill()
            print("⚠️ Servidor anterior forzado a cerrar.")

except NameError:

    print(
        "\nNo se encontró un proceso FastAPI anterior."
    )


server_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


print(
    "\n⏳ Reiniciando FastAPI con "
    "los nuevos rangos de costos..."
)

time.sleep(5)


if server_process.poll() is None:

    print("✅ FastAPI reiniciado.")
    print("PID:", server_process.pid)

    print("\nNuevos valores esperados:")
    print(
        "Parque de la 93 / caminar / 2 viajeros:"
    )
    print("USD 50 - 150")

else:

    print("❌ FastAPI terminó inesperadamente.")
    print(server_process.stdout.read())

✅ Interacción visual V2 aplicada.

Deteniendo FastAPI anterior...
✅ Servidor anterior detenido.

⏳ Reiniciando FastAPI con los nuevos rangos de costos...
✅ FastAPI reiniciado.
PID: 25032

Nuevos valores esperados:
Parque de la 93 / caminar / 2 viajeros:
USD 50 - 150


In [117]:
script_path = "/content/travel_ai_project/frontend/script.js"

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    script_content = f.read()


# ============================================================
# 1. CREAR FUNCIÓN DE RESUMEN DE PRESUPUESTO
# ============================================================

budget_function = r'''

function createBudgetSummary(summary) {

    if (!summary) {
        return "";
    }

    const budget =
        summary.presupuesto_ingresado ?? "-";

    const minCost =
        summary.costo_estimado_min ?? "-";

    const maxCost =
        summary.costo_estimado_max ?? "-";

    const currency =
        summary.moneda_presupuesto || "USD";

    const status =
        summary.estado || "";

    let statusText = "";
    let statusClass = "";

    if (status === "dentro_del_presupuesto") {
        statusText = "Dentro del presupuesto";
        statusClass = "budget-ok";
    }

    else if (
        status === "riesgo_de_superar_presupuesto"
    ) {
        statusText = "Riesgo de superar el presupuesto";
        statusClass = "budget-warning";
    }

    else if (
        status === "fuera_del_presupuesto"
    ) {
        statusText = "Fuera del presupuesto";
        statusClass = "budget-danger";
    }

    else {
        statusText = "Revisión de presupuesto requerida";
        statusClass = "budget-neutral";
    }


    return `
        <section class="budget-summary">

            <div class="budget-item">
                <span class="budget-label">
                    Presupuesto
                </span>

                <strong>
                    ${currency} ${budget}
                </strong>
            </div>


            <div class="budget-item">
                <span class="budget-label">
                    Gasto estimado
                </span>

                <strong>
                    USD ${minCost} - ${maxCost}
                </strong>
            </div>


            <div class="budget-item">
                <span class="budget-label">
                    Viajeros
                </span>

                <strong>
                    ${summary.viajeros?.total ?? "-"}
                </strong>
            </div>


            <div class="budget-status ${statusClass}">
                ${statusText}
            </div>

        </section>
    `;
}

'''


if "function createBudgetSummary(" not in script_content:

    marker = "function renderItinerary(data) {"

    script_content = script_content.replace(
        marker,
        budget_function + "\n" + marker
    )


# ============================================================
# 2. INSERTAR RESUMEN ANTES DE LOS DÍAS
# ============================================================

old_render = '''
    itineraryContainer.innerHTML = "";

    data.dias.forEach(day => {
'''

new_render = '''
    itineraryContainer.innerHTML = "";

    itineraryContainer.insertAdjacentHTML(
        "beforeend",
        createBudgetSummary(
            data.resumen_presupuesto
        )
    );

    data.dias.forEach(day => {
'''


if old_render in script_content:

    script_content = script_content.replace(
        old_render,
        new_render
    )


with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(script_content)


print("✅ Resumen económico añadido al frontend.")

print("\nVerificación:")
print(
    "createBudgetSummary:",
    "function createBudgetSummary(" in script_content
)

print(
    "resumen_presupuesto:",
    "data.resumen_presupuesto" in script_content
)

✅ Resumen económico añadido al frontend.

Verificación:
createBudgetSummary: True
resumen_presupuesto: True


In [118]:
styles_path = "/content/travel_ai_project/frontend/styles.css"

with open(
    styles_path,
    "r",
    encoding="utf-8"
) as f:
    styles_content = f.read()


budget_styles = r'''

/* ============================================================
   RESUMEN DE PRESUPUESTO
   ============================================================ */

.budget-summary {
    display: grid;
    grid-template-columns:
        repeat(3, minmax(0, 1fr))
        auto;

    align-items: center;

    gap: 18px;

    margin: 24px 0 36px;

    padding: 20px 22px;

    border: 1px solid #dfe3ea;
    border-radius: 16px;

    background: #ffffff;
}

.budget-item {
    display: flex;
    flex-direction: column;
    gap: 4px;
}

.budget-label {
    color: #667085;
    font-size: 0.78rem;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 0.04em;
}

.budget-item strong {
    color: #172033;
    font-size: 1.05rem;
}

.budget-status {
    display: inline-flex;
    align-items: center;
    justify-content: center;

    min-height: 42px;

    padding: 9px 14px;

    border-radius: 10px;

    font-size: 0.88rem;
    font-weight: 800;

    white-space: nowrap;
}

.budget-ok {
    background: #ecfdf3;
    color: #027a48;
    border: 1px solid #abefc6;
}

.budget-warning {
    background: #fffaeb;
    color: #b54708;
    border: 1px solid #fedf89;
}

.budget-danger {
    background: #fef3f2;
    color: #b42318;
    border: 1px solid #fecdca;
}

.budget-neutral {
    background: #f2f4f7;
    color: #475467;
    border: 1px solid #d0d5dd;
}


@media (max-width: 900px) {

    .budget-summary {
        grid-template-columns:
            repeat(2, minmax(0, 1fr));
    }

    .budget-status {
        width: 100%;
    }
}


@media (max-width: 600px) {

    .budget-summary {
        grid-template-columns: 1fr;
    }
}
'''


if "RESUMEN DE PRESUPUESTO" not in styles_content:

    styles_content += budget_styles

    with open(
        styles_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(styles_content)

    print("✅ Estilos del resumen económico añadidos.")

else:
    print("ℹ️ Los estilos ya estaban aplicados.")


print("\nVerificación:")
print(
    "budget-summary:",
    ".budget-summary {" in styles_content
)

print(
    "budget-ok:",
    ".budget-ok {" in styles_content
)

print(
    "budget-danger:",
    ".budget-danger {" in styles_content
)

✅ Estilos del resumen económico añadidos.

Verificación:
budget-summary: True
budget-ok: True
budget-danger: True


In [119]:
script_path = "/content/travel_ai_project/frontend/script.js"

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    script_content = f.read()


old_day_header = '''
        dayCard.innerHTML = `
            <h3>Día ${day.dia}</h3>

            <div class="period-grid">
'''


new_day_header = '''
        const dayCost = day.costo_dia || {};

        const dayMin =
            dayCost.min !== undefined
            ? dayCost.min
            : "-";

        const dayMax =
            dayCost.max !== undefined
            ? dayCost.max
            : "-";


        dayCard.innerHTML = `
            <div class="day-header">

                <h3>
                    Día ${day.dia}
                </h3>

                <span class="day-cost">
                    USD ${dayMin} - ${dayMax}
                </span>

            </div>

            <div class="period-grid">
'''


if old_day_header in script_content:

    script_content = script_content.replace(
        old_day_header,
        new_day_header
    )

    with open(
        script_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(script_content)

    print("✅ Costo diario añadido al encabezado.")

else:

    print(
        "⚠️ No se encontró el encabezado "
        "de día esperado."
    )


print("\nVerificación:")

print(
    "day-cost:",
    'class="day-cost"' in script_content
)

print(
    "costo_dia:",
    "day.costo_dia" in script_content
)

✅ Costo diario añadido al encabezado.

Verificación:
day-cost: True
costo_dia: True


In [121]:
readme_content = r"""
# Travel AI Planner

## Caso Práctico – Generative AI

Travel AI Planner es un prototipo de planificación inteligente de viajes desarrollado
como parte del módulo de **Generative AI**.

El proyecto explora y compara diferentes estrategias de personalización de modelos
de lenguaje:

- Prompt Engineering
- Retrieval-Augmented Generation (RAG)
- Fine-tuning mediante LoRA / PEFT
- Arquitectura híbrida RAG + LoRA
- Salida estructurada en JSON
- Validación determinista
- Control de presupuesto
- Interfaz web

---

# 1. Objetivo

Desarrollar un sistema capaz de generar itinerarios de viaje personalizados a partir
de preferencias introducidas por el usuario.

El usuario puede indicar:

- destino;
- duración del viaje;
- presupuesto;
- moneda;
- número de adultos;
- número de niños;
- intereses;
- ritmo de viaje;
- restricciones o preferencias.

El sistema devuelve un itinerario organizado por:

- mañana;
- tarde;
- noche.

Además, proporciona:

- lugar recomendado;
- actividad;
- rango de gasto estimado;
- descripción del sitio;
- acceso a Google Maps;
- resumen global de presupuesto.

---

# 2. Modelo utilizado

Modelo base:

`Qwen/Qwen2.5-1.5B-Instruct`

El modelo fue cargado mediante Hugging Face Transformers.

---

# 3. Fine-tuning

Se realizó una prueba de fine-tuning utilizando:

- PEFT;
- LoRA;
- Hugging Face Trainer.

Configuración principal:

- LoRA rank: 8
- LoRA alpha: 16
- LoRA dropout: 0.05
- Learning rate: 2e-4
- Epochs: 5
- Batch size: 1
- Gradient accumulation: 2

Parámetros entrenables:

`2,179,072`

Parámetros totales:

`1,545,893,376`

Porcentaje entrenado:

`0.141 %`

El entrenamiento finalizó con:

`train_loss ≈ 1.62`

El dataset utilizado contenía únicamente tres ejemplos, por lo que el fine-tuning
se considera una **prueba de concepto académica**, no un entrenamiento suficiente
para producción.

---

# 4. Estrategias evaluadas

Durante el desarrollo se compararon los siguientes enfoques:

| Enfoque | JSON válido | Esquema correcto | Grounding | Alucinaciones |
|---|---:|---:|---:|---:|
| Modelo base | Sí | No | No | Sí |
| Prompting | Parcial | Parcial | No | Sí |
| RAG inicial | Parcial | Parcial | Parcial | Sí |
| RAG ampliado | Parcial | Parcial | Parcial | Sí |
| RAG estructurado | Sí | Sí | Sí | No |
| Fine-tuning LoRA | Sí | No | No | Sí |
| Híbrido RAG + LoRA | Sí | Sí | Sí | No |

La arquitectura híbrida presentó el mejor comportamiento dentro del experimento.

---

# 5. Arquitectura final

    Usuario
       |
       v
    Frontend HTML / CSS / JavaScript
       |
       v
    FastAPI
       |
       v
    Prompt Engineering
       |
       v
    Base de conocimiento estructurada
       |
       v
    Qwen2.5 + LoRA
       |
       v
    Validación determinista
       |
       v
    Reparación de inconsistencias
       |
       v
    Cálculo de costos en Python
       |
       v
    Control de presupuesto
       |
       v
    JSON validado
       |
       v
    Frontend

---

# 6. RAG y grounding

Inicialmente se utilizó una base de conocimiento pequeña y recuperación mediante:

- Sentence Transformers;
- embeddings multilingües;
- FAISS.

Modelo de embeddings:

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

Los primeros experimentos mostraron que RAG reducía las alucinaciones, pero no
garantizaba que el modelo utilizara exclusivamente la información recuperada.

Por este motivo se evolucionó hacia un **RAG estructurado**, donde cada lugar contiene
un conjunto explícito de actividades autorizadas.

Ejemplo conceptual:

    {
      "Museo Botero": {
        "category": [
          "arte",
          "cultura"
        ],
        "allowed_activities": {
          "observar obras de Fernando Botero": {
            "min_cost_usd": 5,
            "max_cost_usd": 15
          }
        }
      }
    }

---

# 7. Control de alucinaciones

La salida del modelo no se acepta directamente.

Después de la generación, Python valida:

1. JSON válido.
2. Número correcto de días.
3. Existencia de mañana, tarde y noche.
4. Lugares existentes en la base autorizada.
5. Actividades autorizadas para cada lugar.
6. Máximo de repeticiones por lugar.
7. Estructura requerida.

Si una respuesta falla, el sistema puede intentar una corrección y posteriormente
aplicar una reparación determinista.

Solo los itinerarios que pasan la validación son enviados al frontend.

---

# 8. Presupuesto y costos

Los costos NO son generados libremente por el LLM.

Los rangos de gasto se almacenan dentro de la base estructurada y Python calcula:

- costo estimado por persona;
- costo estimado para el grupo;
- costo diario;
- costo mínimo del viaje;
- costo máximo del viaje;
- comparación con el presupuesto.

Los valores utilizados en este prototipo son **estimaciones académicas de planificación**
y no deben interpretarse como precios oficiales o en tiempo real.

Ejemplo:

    Presupuesto: USD 1,700
    Gasto estimado: USD 270 - 740
    Viajeros: 2
    Estado: Dentro del presupuesto

---

# 9. Información adicional de los lugares

Cada recomendación puede contener:

- descripción;
- rango estimado de gasto;
- enlace a Google Maps.

Los enlaces de Google Maps utilizan búsquedas por nombre del sitio para evitar depender
de coordenadas o direcciones almacenadas que puedan quedar desactualizadas.

---

# 10. Interfaz

La aplicación utiliza:

- HTML5;
- CSS3;
- JavaScript.

El frontend permite:

- configurar el viaje;
- seleccionar intereses;
- definir presupuesto;
- indicar viajeros;
- generar el itinerario;
- visualizar gasto estimado;
- consultar información adicional;
- abrir los lugares en Google Maps.

Las tarjetas cuentan con interacción mediante hover y selección, facilitando la
exploración visual del itinerario.

Los días se presentan mediante bloques diferenciados con mañana, tarde y noche,
mejorando la jerarquía visual y la experiencia del usuario.

---

# 11. Backend

El backend fue construido utilizando:

`FastAPI`

Endpoints principales:

    GET /
    POST /generate

El endpoint `/generate` recibe las preferencias y devuelve únicamente itinerarios
que superan la validación.

La API integra:

- generación con Qwen;
- adaptador LoRA;
- base estructurada;
- validación determinista;
- reparación de repeticiones;
- cálculo de costos;
- evaluación de presupuesto.

---

# 12. Estructura del proyecto

    travel_ai_project/
    │
    ├── app.py
    ├── app_config.json
    ├── authorized_places.json
    ├── requirements.txt
    ├── README.md
    │
    ├── qwen_travel_lora/
    │   ├── adapter_config.json
    │   ├── adapter_model.safetensors
    │   ├── tokenizer.json
    │   ├── tokenizer_config.json
    │   ├── chat_template.jinja
    │   └── README.md
    │
    └── frontend/
        ├── index.html
        ├── styles.css
        └── script.js

---

# 13. Instalación

Instalar dependencias:

    pip install -r requirements.txt

Ejecutar FastAPI:

    uvicorn app:app --host 0.0.0.0 --port 8000

Abrir la interfaz:

    http://localhost:8000/ui/

---

# 14. Tecnologías utilizadas

- Python
- PyTorch
- Hugging Face Transformers
- Qwen2.5
- PEFT
- LoRA
- Sentence Transformers
- FAISS
- FastAPI
- Pydantic
- HTML
- CSS
- JavaScript
- Google Colab
- Google Maps

---

# 15. Resultados

El experimento mostró que el modelo base puede generar itinerarios coherentes,
pero presenta problemas de factualidad y control de formato.

Prompt Engineering mejoró considerablemente el seguimiento de instrucciones, pero
no eliminó las alucinaciones.

RAG permitió introducir conocimiento externo, aunque el contexto por sí solo no
garantizó que el modelo respetara completamente las restricciones.

El RAG estructurado y la validación determinista permitieron controlar de forma
mucho más efectiva los lugares y actividades generados.

El fine-tuning mediante LoRA demostró que es posible adaptar eficientemente un LLM,
entrenando únicamente una pequeña fracción de sus parámetros. No obstante, el dataset
de tres ejemplos fue insuficiente para conseguir una mejora funcional robusta de forma
independiente.

La combinación de:

- Prompt Engineering;
- RAG estructurado;
- LoRA;
- validación determinista;
- reparación programática;

produjo el resultado más consistente del experimento.

---

# 16. Conclusiones

RAG y fine-tuning resuelven problemas diferentes.

**RAG** es adecuado cuando la aplicación necesita incorporar conocimiento externo,
controlado o actualizable.

**Fine-tuning** resulta útil para adaptar patrones de comportamiento, formato,
estilo o especialización del modelo.

Para aplicaciones donde la exactitud y el control de las respuestas son importantes,
no es recomendable confiar únicamente en el LLM.

Una arquitectura que combine:

`LLM + RAG + structured output + validación determinista`

puede ofrecer un mayor nivel de control y reducir significativamente las alucinaciones.

El desarrollo también evidenció que la generación de lenguaje y la lógica de negocio
deben mantenerse separadas cuando existen restricciones objetivas. Por esta razón,
los costos y la validación de presupuesto fueron implementados mediante lógica
determinista en Python en lugar de solicitar al LLM que estimara libremente los valores.

---

# 17. Limitaciones

Este proyecto es una prueba de concepto académica.

Principales limitaciones:

- base de conocimiento limitada principalmente a Bogotá;
- costos estimados y no conectados a precios en tiempo real;
- dataset de fine-tuning de solo tres ejemplos;
- no existe todavía una fuente externa de información turística dinámica;
- la versión de prueba ejecuta el modelo en infraestructura temporal de Google Colab;
- los valores de moneda distintos a USD requieren conversión externa;
- no se incluyen costos completos de vuelos, alojamiento u otros componentes del viaje;
- la base de lugares y actividades es controlada y relativamente pequeña;
- los costos son rangos académicos orientativos.

---

# 18. Posibles mejoras

Como evolución del prototipo se podrían implementar:

- integración con APIs turísticas;
- precios reales de hoteles y actividades;
- información meteorológica;
- conversión automática de moneda;
- rutas geográficas optimizadas;
- recomendaciones de restaurantes;
- base RAG para múltiples ciudades;
- evaluación automática de relevancia;
- dataset de fine-tuning de mayor tamaño;
- autenticación de usuarios;
- almacenamiento de viajes;
- exportación del itinerario;
- generación de itinerarios en PDF;
- backend desplegado permanentemente en infraestructura cloud;
- integración con vuelos y alojamiento;
- uso de información turística actualizada;
- incorporación de coordenadas geográficas;
- optimización de trayectos entre lugares.

---

# 19. Flujo de funcionamiento

El flujo final de la aplicación es:

1. El usuario introduce sus preferencias.
2. El frontend envía los datos a FastAPI.
3. El backend crea el prompt estructurado.
4. Qwen + LoRA genera una propuesta.
5. Python valida el esquema.
6. Se comprueban lugares y actividades.
7. Se controlan las repeticiones.
8. Si es necesario, se aplica reparación.
9. Python añade los costos autorizados.
10. Se calcula el gasto mínimo y máximo del grupo.
11. Se compara el gasto con el presupuesto.
12. La API devuelve el JSON validado.
13. El frontend renderiza el itinerario.
14. El usuario puede consultar detalles y abrir Google Maps.

---

# 20. Aprendizajes del experimento

El desarrollo permitió comprobar en la práctica varias diferencias importantes
entre Prompting, RAG y Fine-tuning.

### Prompting

Es una técnica rápida y económica para modificar el comportamiento del modelo,
pero no garantiza factualidad.

### RAG

Permite incorporar información externa sin modificar los pesos del modelo.

Su calidad depende significativamente de:

- calidad de la base documental;
- diversidad del contexto;
- estrategia de recuperación;
- estructura de la información recuperada.

### Fine-tuning

Permite modificar patrones aprendidos por el modelo, pero necesita:

- suficientes ejemplos;
- datos de buena calidad;
- evaluación posterior;
- recursos de entrenamiento.

### Arquitectura híbrida

El mejor comportamiento se obtuvo al asignar responsabilidades diferentes a
cada componente:

- LLM: generación y selección;
- RAG: conocimiento;
- LoRA: adaptación;
- Python: reglas y validación;
- frontend: experiencia del usuario.

---

# 21. Contexto académico

Proyecto desarrollado como Caso Práctico de la Unidad 1 del módulo de
**Generative AI**, enfocado en personalización de modelos de lenguaje mediante:

- Prompting;
- Retrieval-Augmented Generation;
- Fine-tuning;
- arquitecturas híbridas.

El proyecto se desarrolló de forma experimental, comparando diferentes estrategias
y documentando tanto los resultados exitosos como las limitaciones encontradas.
"""


# ============================================================
# GUARDAR README
# ============================================================

readme_path = "/content/travel_ai_project/README.md"

with open(
    readme_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(readme_content)


# ============================================================
# VERIFICACIONES
# ============================================================

print("✅ README.md final creado correctamente.")
print("Archivo:", readme_path)
print("Tamaño:", len(readme_content), "caracteres")


with open(
    readme_path,
    "r",
    encoding="utf-8"
) as f:
    saved_readme = f.read()


print("\nVERIFICACIÓN:")
print(
    "Título encontrado:",
    "# Travel AI Planner" in saved_readme
)

print(
    "Sección Fine-tuning encontrada:",
    "# 3. Fine-tuning" in saved_readme
)

print(
    "Sección RAG encontrada:",
    "# 6. RAG y grounding" in saved_readme
)

print(
    "Sección presupuesto encontrada:",
    "# 8. Presupuesto y costos" in saved_readme
)

print(
    "Conclusiones encontradas:",
    "# 16. Conclusiones" in saved_readme
)

print(
    "Contexto académico encontrado:",
    "# 21. Contexto académico" in saved_readme
)

✅ README.md final creado correctamente.
Archivo: /content/travel_ai_project/README.md
Tamaño: 12727 caracteres

VERIFICACIÓN:
Título encontrado: True
Sección Fine-tuning encontrada: True
Sección RAG encontrada: True
Sección presupuesto encontrada: True
Conclusiones encontradas: True
Contexto académico encontrado: True


In [2]:
import os

print("📁 Contenido actual de /content:\n")

for item in os.listdir("/content"):
    path = os.path.join("/content", item)

    if os.path.isdir(path):
        print("📂", item)
    else:
        print("📄", item)

📁 Contenido actual de /content:

📂 .config
📂 sample_data
